In [ ]:
# =========================================================
# BraTS 2024 – 2D Slice Dataset + UNet Training (ALL-IN-ONE)
# Jupyter / Windows safe, lazy-loading
# =========================================================

import os
import json
import numpy as np
import nibabel as nib
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Optional
from tqdm import tqdm

# =========================
# Dataset
# =========================
class BraTS2024SliceDataset(Dataset):
    """
    input : FLAIR axial slice (1 channel)
    target: segmentation mask slice (0..4)
    """
    def __init__(
        self,
        index_json: str,
        split_json: str,
        image_size: int = 256,
        tumor_only: bool = True,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = False,
    ):
        self.image_size = image_size
        self.tumor_only = tumor_only
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        with open(index_json, "r") as f:
            self.subjects = json.load(f)

        with open(split_json, "r") as f:
            self.sids = json.load(f)

        # Lazy-loading: store slices info, don't load full NIfTI yet
        self.samples = []
        self._ram_cache = {}

        print("Selecting slices per subject...")
        for sid in tqdm(self.sids, desc="Subjects"):
            paths = self.subjects[sid]
            mask = nib.load(paths["mask"]).get_fdata().astype(np.int16)
            Z = mask.shape[2]

            zs = []
            for z in range(Z):
                if not self.tumor_only:
                    zs.append(z)
                else:
                    if (mask[:, :, z] > 0).mean() >= self.tumor_min_frac:
                        zs.append(z)

            if self.tumor_only and len(zs) == 0:
                zs = list(range(Z))

            if self.max_slices_per_subject is not None and len(zs) > self.max_slices_per_subject:
                idx = np.linspace(0, len(zs)-1, self.max_slices_per_subject).astype(int)
                zs = [zs[i] for i in idx]

            for z in zs:
                self.samples.append((sid, z))

        if len(self.samples) == 0:
            raise RuntimeError("No slices selected. Disable tumor_only or lower tumor_min_frac.")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self._ram_cache:
            return self._ram_cache[sid]

        paths = self.subjects[sid]
        flair = nib.load(paths["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(paths["mask"]).get_fdata().astype(np.int16)

        if self.cache_in_ram:
            self._ram_cache[sid] = (flair, mask)

        return flair, mask

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        flair, mask = self._load_subject(sid)

        img = self._zscore(flair[:, :, z])
        msk = mask[:, :, z]

        img = cv2.resize(img, (self.image_size, self.image_size), cv2.INTER_LINEAR)
        msk = cv2.resize(msk, (self.image_size, self.image_size), cv2.INTER_NEAREST)

        x = torch.from_numpy(img).unsqueeze(0).float()
        y = torch.from_numpy(msk).long()

        return x, y

# =========================
# UNet
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): 
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_c=1, num_classes=5, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base)
        self.enc2 = ConvBlock(base, base*2)
        self.enc3 = ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)

        self.bot = ConvBlock(base*4, base*8)

        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.dec3 = ConvBlock(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.dec2 = ConvBlock(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.dec1 = ConvBlock(base*2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bot(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))

        return self.out(d1)

# =========================
# Metrics
# =========================
@torch.no_grad()
def mean_dice(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        inter = (p * t).sum()
        denom = p.sum() + t.sum()
        dices.append((2*inter + eps) / (denom + eps))
    return torch.stack(dices).mean().item()

# =========================
# Training
# =========================
def main():
    INDEX = r"C:/GDNET/experiments/brats2024_index.json"
    TRAIN = r"C:/GDNET/splits/brats2024_train_labeled.json"
    VAL   = r"C:/GDNET/splits/brats2024_val_labeled.json"

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", DEVICE)

    train_ds = BraTS2024SliceDataset(INDEX, TRAIN, tumor_only=True)
    val_ds   = BraTS2024SliceDataset(INDEX, VAL, tumor_only=False)

    print("Train slices:", len(train_ds))
    print("Val slices:", len(val_ds))

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)

    model = UNetSmall().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    ce = nn.CrossEntropyLoss()

    best = -1

    for epoch in range(1, 6):
        model.train()
        tr_loss = 0

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch} training"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = ce(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tr_loss += loss.item() * x.size(0)

        tr_loss /= len(train_ds)

        model.eval()
        vd, vl, n = 0, 0, 0
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            vl += ce(logits, y).item() * x.size(0)
            vd += mean_dice(logits.argmax(1), y) * x.size(0)
            n += x.size(0)

        print(f"Epoch {epoch} | train_loss={tr_loss:.4f} | val_loss={vl/n:.4f} | val_dice={vd/n:.4f}")

if __name__ == "__main__":
    main()


Using device: cuda
Selecting slices per subject...


Subjects: 100%|████████████████████████████████████████████████████████████████████| 1143/1143 [02:31<00:00,  7.55it/s]


Selecting slices per subject...


Subjects: 100%|██████████████████████████████████████████████████████████████████████| 248/248 [00:31<00:00,  7.98it/s]


Train slices: 72052
Val slices: 45136


Epoch 1 training: 100%|██████████████████████████████████████████████████████████| 9007/9007 [7:25:30<00:00,  2.97s/it]


Epoch 1 | train_loss=0.0633 | val_loss=0.0180 | val_dice=0.6532


Epoch 2 training: 100%|██████████████████████████████████████████████████████████| 9007/9007 [6:08:31<00:00,  2.45s/it]


In [1]:
import os
import json
import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# =========================
# Config
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = r"C:/GDNET/logs/step5_6_unet_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 256
BATCH = 4
EPOCHS = 2
LR = 1e-3
NUM_CLASSES = 5

# Windows-safe: start with 0
NUM_WORKERS = 0

TUMOR_ONLY_TRAIN = True
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 40   # make it small to confirm it runs
CACHE_IN_RAM = False

# =========================
# Dataset
# =========================
class BraTS2024SliceDataset(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        tumor_only=True,
        tumor_min_frac=0.001,
        max_slices_per_subject=None,
        cache_in_ram=False,
    ):
        self.subjects = subjects_index
        self.sids = subject_ids
        self.image_size = image_size
        self.tumor_only = tumor_only
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.samples = []
        self._ram_cache = {}

        for sid in self.sids:
            mask_path = self.subjects[sid]["mask"]
            m = nib.load(mask_path).get_fdata().astype(np.int16)
            Z = m.shape[2]

            candidate_z = []
            if tumor_only:
                for z in range(Z):
                    if (m[:, :, z] > 0).mean() >= tumor_min_frac:
                        candidate_z.append(z)
                if not candidate_z:
                    candidate_z = list(range(Z))
            else:
                candidate_z = list(range(Z))

            if max_slices_per_subject is not None and len(candidate_z) > max_slices_per_subject:
                idx = np.linspace(0, len(candidate_z) - 1, max_slices_per_subject).astype(int)
                candidate_z = [candidate_z[i] for i in idx]

            for z in candidate_z:
                self.samples.append((sid, z))

        if len(self.samples) == 0:
            raise RuntimeError("No slices selected. Lower tumor_min_frac or set tumor_only=False.")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self._ram_cache:
            return self._ram_cache[sid]

        flair = nib.load(self.subjects[sid]["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)

        if self.cache_in_ram:
            self._ram_cache[sid] = (flair, mask)
        return flair, mask

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        flair_vol, mask_vol = self._load_subject(sid)

        img = self._zscore(flair_vol[:, :, z])
        msk = mask_vol[:, :, z]

        img = cv2.resize(img, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        msk = cv2.resize(msk, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        x = torch.from_numpy(img).unsqueeze(0).float()
        y = torch.from_numpy(msk).long()
        return x, y

# =========================
# Model
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_c=1, num_classes=5, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)

@torch.no_grad()
def mean_dice(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        dices.append((2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps))
    return torch.stack(dices).mean().item()

def main():
    print("=== START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(TRAIN_SPLIT, "r", encoding="utf-8") as f:
        train_ids = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    print(f"Train subjects={len(train_ids)} | Val subjects={len(val_ids)}", flush=True)

    train_ds = BraTS2024SliceDataset(
        subjects, train_ids, image_size=IMG_SIZE,
        tumor_only=TUMOR_ONLY_TRAIN, tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM
    )
    val_ds = BraTS2024SliceDataset(
        subjects, val_ids, image_size=IMG_SIZE,
        tumor_only=False, max_slices_per_subject=MAX_SLICES_PER_SUBJECT
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)}", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

    model = UNetSmall(in_c=1, num_classes=NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    ce = nn.CrossEntropyLoss()

    best = -1.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()
            opt.step()

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        tr_loss /= len(train_loader.dataset)

        model.eval()
        va_loss, va_dice, n = 0.0, 0.0, 0

        pbarv = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} VAL", leave=True)
        for x, y in pbarv:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = ce(logits, y)

            va_loss += loss.item() * x.size(0)
            pred = torch.argmax(logits, dim=1)
            va_dice += mean_dice(pred, y, num_classes=NUM_CLASSES) * x.size(0)
            n += x.size(0)

            pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= len(val_loader.dataset)
        va_dice /= max(n, 1)

        print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_meanDice={va_dice:.4f}", flush=True)

        if va_dice > best:
            best = va_dice
            ckpt = os.path.join(OUT_DIR, "best.pt")
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_dice": best}, ckpt)
            print("Saved:", ckpt, flush=True)

    print("=== DONE === Best val_meanDice:", best, flush=True)

if __name__ == "__main__":
    main()


=== START ===
Device: cuda
Torch: 2.5.1+cu121
Train subjects=1143 | Val subjects=248
Train slices=44129 | Val slices=9920


Epoch 1/2 VAL: 100%|█████████████████████████████████████████████████| 2480/2480 [57:40<00:00,  1.40s/it, vloss=0.0000]

Epoch 01 | train_loss=0.0582 | val_loss=0.0165 | val_meanDice=0.7124


Saved: C:/GDNET/logs/step5_6_unet_baseline\best.pt


Epoch 2/2 VAL: 100%|█████████████████████████████████████████████████| 2480/2480 [57:50<00:00,  1.40s/it, vloss=0.0041]

Epoch 02 | train_loss=0.0354 | val_loss=0.0149 | val_meanDice=0.7240
Saved: C:/GDNET/logs/step5_6_unet_baseline\best.pt
=== DONE === Best val_meanDice: 0.7239938411688913


In [3]:
# C:/GDNET/experiments/step7_train_unet_4ch_amp_tqdm.py
# Step 7: BraTS2024 2D axial slices, 4-channel input (T1/T1CE/T2/FLAIR) -> multi-class mask (0..4)
# Includes: tqdm progress, Windows-safe defaults, optional RAM caching, AMP (mixed precision), speed-friendly DataLoader.

import os
import json
import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# =========================
# Config
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = r"C:/GDNET/logs/step7_unet_4ch_amp"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 256
BATCH = 4                 # increase if GPU allows
EPOCHS = 3                # start small; increase later
LR = 1e-3
NUM_CLASSES = 5           # BraTS mask values 0..4

# Windows-safe: start with 0; if stable, set to 2 or 4 for speed
NUM_WORKERS = 0

# Data sampling
TUMOR_ONLY_TRAIN = True
TUMOR_MIN_FRAC = 0.001     # 0.1% pixels positive on mask slice
MAX_SLICES_PER_SUBJECT = 40  # set None for full; keep 40 for iteration speed

# Speed features
CACHE_IN_RAM = False       # True if you have ample RAM (recommended if >=64-128GB)
PIN_MEMORY = True          # helps GPU transfer
USE_AMP = True             # mixed precision on CUDA

# =========================
# Dataset
# =========================
class BraTS2024SliceDataset4Ch(Dataset):
    """
    Input: 4 channels (T1, T1CE, T2, FLAIR) axial slice
    Target: segmentation mask axial slice (class ids 0..4)
    """
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        tumor_only: bool = True,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject = None,
        cache_in_ram: bool = False,
    ):
        self.subjects = subjects_index
        self.sids = subject_ids
        self.image_size = image_size
        self.tumor_only = tumor_only
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.samples = []       # (sid, z)
        self._ram_cache = {}    # sid -> (t1, t1ce, t2, flair, mask)

        # Build slice index using mask only (fast, deterministic)
        for sid in self.sids:
            mask_path = self.subjects[sid]["mask"]
            m = nib.load(mask_path).get_fdata().astype(np.int16)
            Z = m.shape[2]

            candidate_z = []
            if tumor_only:
                for z in range(Z):
                    if (m[:, :, z] > 0).mean() >= tumor_min_frac:
                        candidate_z.append(z)
                if not candidate_z:
                    candidate_z = list(range(Z))
            else:
                candidate_z = list(range(Z))

            if max_slices_per_subject is not None and len(candidate_z) > max_slices_per_subject:
                idx = np.linspace(0, len(candidate_z) - 1, max_slices_per_subject).astype(int)
                candidate_z = [candidate_z[i] for i in idx]

            for z in candidate_z:
                self.samples.append((sid, z))

        if len(self.samples) == 0:
            raise RuntimeError("No slices selected. Lower tumor_min_frac or set tumor_only=False.")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _zscore(x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid: str):
        if self.cache_in_ram and sid in self._ram_cache:
            return self._ram_cache[sid]

        paths = self.subjects[sid]
        t1    = nib.load(paths["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(paths["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(paths["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(paths["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(paths["mask"]).get_fdata().astype(np.int16)

        data = (t1, t1ce, t2, flair, mask)
        if self.cache_in_ram:
            self._ram_cache[sid] = data
        return data

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = self._zscore(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)

        x1 = prep(t1)
        x2 = prep(t1ce)
        x3 = prep(t2)
        x4 = prep(flair)

        msk = mask_vol[:, :, z]
        msk = cv2.resize(msk, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        x = torch.from_numpy(np.stack([x1, x2, x3, x4], axis=0)).float()  # (4,H,W)
        y = torch.from_numpy(msk).long()                                   # (H,W)
        return x, y

# =========================
# Model
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)

# =========================
# Metric
# =========================
@torch.no_grad()
def mean_dice(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        dices.append((2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps))
    return torch.stack(dices).mean().item()

# =========================
# Main
# =========================
def main():
    print("=== STEP 7 START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(TRAIN_SPLIT, "r", encoding="utf-8") as f:
        train_ids = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    print(f"Train subjects={len(train_ids)} | Val subjects={len(val_ids)}", flush=True)

    train_ds = BraTS2024SliceDataset4Ch(
        subjects, train_ids,
        image_size=IMG_SIZE,
        tumor_only=TUMOR_ONLY_TRAIN,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM
    )
    val_ds = BraTS2024SliceDataset4Ch(
        subjects, val_ids,
        image_size=IMG_SIZE,
        tumor_only=False,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=False
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)}", flush=True)

    # DataLoaders
    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY if DEVICE == "cuda" else False
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY if DEVICE == "cuda" else False
    )

    # Model + opt + loss
    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    ce = nn.CrossEntropyLoss()

    scaler = GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

    best = -1.0

    for epoch in range(1, EPOCHS + 1):
        # -------- Train --------
        model.train()
        tr_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            if DEVICE == "cuda":
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
            else:
                x, y = x.to(DEVICE), y.to(DEVICE)

            opt.zero_grad(set_to_none=True)

            with autocast(enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = ce(logits, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        tr_loss /= len(train_loader.dataset)

        # -------- Val --------
        model.eval()
        va_loss, va_dice, n = 0.0, 0.0, 0

        pbarv = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} VAL", leave=True)
        with torch.no_grad():
            for x, y in pbarv:
                if DEVICE == "cuda":
                    x = x.to(DEVICE, non_blocking=True)
                    y = y.to(DEVICE, non_blocking=True)
                else:
                    x, y = x.to(DEVICE), y.to(DEVICE)

                with autocast(enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = ce(logits, y)

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice += mean_dice(pred, y, num_classes=NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= len(val_loader.dataset)
        va_dice /= max(n, 1)

        print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_meanDice={va_dice:.4f}", flush=True)

        if va_dice > best:
            best = va_dice
            ckpt = os.path.join(OUT_DIR, "best.pt")
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_dice": best}, ckpt)
            print("Saved:", ckpt, flush=True)

    print("=== STEP 7 DONE === Best val_meanDice:", best, flush=True)

if __name__ == "__main__":
    main()


=== STEP 7 START ===
Device: cuda
Torch: 2.5.1+cu121
Train subjects=1143 | Val subjects=248
Train slices=44129 | Val slices=9920


C:\Users\admin\AppData\Local\Temp\ipykernel_14968\2019702689.py:260: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))
C:\Users\admin\AppData\Local\Temp\ipykernel_14968\2019702689.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and DEVICE == "cuda")):
Epoch 1/3 TRAIN: 100%|████████████████████████████████████████████| 11033/11033 [8:27:22<00:00,  2.76s/it, loss=0.0132]
C:\Users\admin\AppData\Local\Temp\ipykernel_14968\2019702689.py:305: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and DEVICE == "cuda")):
Epoch 1/3 VAL: 100%|███████████████████████████████████████████████| 2480/2480 [1:43:13<00:00,  2.50s/it, vloss=0.0014]

Epoch 01 | train_loss=0.0389 | val_loss=0.0112 | val_meanDice=0.7438
Saved: C:/GDNET/logs/step7_unet_4ch_amp\best.pt



Epoch 2/3 VAL: 100%|███████████████████████████████████████| 2480/2480 [1:47:55<00:00,  2.61s/it, vloss=0.0002]

Epoch 02 | train_loss=0.0208 | val_loss=0.0110 | val_meanDice=0.7080



Epoch 3/3 VAL: 100%|███████████████████████████████████████| 2480/2480 [1:46:32<00:00,  2.58s/it, vloss=0.0005]

Epoch 03 | train_loss=0.0178 | val_loss=0.0079 | val_meanDice=0.7792
Saved: C:/GDNET/logs/step7_unet_4ch_amp\best.pt
=== STEP 7 DONE === Best val_meanDice: 0.7792460923894279


# Need for Running

In [4]:
# C:/GDNET/experiments/step7_1_train_unet_4ch_amp_tqdm_stable.py
# Step 7.1: Improvements before GDNet
# - 4-channel input (T1/T1CE/T2/FLAIR) -> multi-class mask (0..4)
# - tqdm progress bars
# - torch.amp API (no FutureWarnings)
# - fixed seeds (repeatability)
# - CE + Soft Dice loss (stability on class imbalance)
# - cosine LR schedule
# - optional in-RAM caching
# - Windows-safe defaults (num_workers=0 by default)

import os
import json
import random
import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

# =========================
# Repro / Speed knobs
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Speed: allow fast kernels (recommended for training)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# =========================
# Paths
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = r"C:/GDNET/logs/step7_1_unet_4ch_amp_stable"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Hyperparams
# =========================
IMG_SIZE = 256
BATCH = 4
EPOCHS = 5
LR = 5e-4               # slightly lower than 1e-3 to reduce instability
NUM_CLASSES = 5

# Windows-safe: start with 0; if stable in script mode, set 2 or 4
NUM_WORKERS = 0
PIN_MEMORY = True
USE_AMP = True

# Sampling
TUMOR_ONLY_TRAIN = True
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 40     # keep for iteration; set None for full
CACHE_IN_RAM = False            # True if enough RAM

# Loss weights (common, stable)
W_CE = 0.5
W_DICE = 0.5

# =========================
# Dataset
# =========================
class BraTS2024SliceDataset4Ch(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        tumor_only=True,
        tumor_min_frac=0.001,
        max_slices_per_subject=None,
        cache_in_ram=False,
    ):
        self.subjects = subjects_index
        self.sids = subject_ids
        self.image_size = image_size
        self.tumor_only = tumor_only
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.samples = []      # (sid, z)
        self._ram_cache = {}   # sid -> (t1, t1ce, t2, flair, mask)

        # Build slice index using mask only (fast, deterministic)
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            candidate_z = []
            if self.tumor_only:
                for z in range(Z):
                    if (m[:, :, z] > 0).mean() >= self.tumor_min_frac:
                        candidate_z.append(z)
                if not candidate_z:
                    candidate_z = list(range(Z))
            else:
                candidate_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(candidate_z) > self.max_slices_per_subject:
                idx = np.linspace(0, len(candidate_z) - 1, self.max_slices_per_subject).astype(int)
                candidate_z = [candidate_z[i] for i in idx]

            for z in candidate_z:
                self.samples.append((sid, z))

        if len(self.samples) == 0:
            raise RuntimeError("No slices selected. Lower tumor_min_frac or set tumor_only=False.")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self._ram_cache:
            return self._ram_cache[sid]

        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)

        data = (t1, t1ce, t2, flair, mask)
        if self.cache_in_ram:
            self._ram_cache[sid] = data
        return data

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = self._zscore(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)

        x1 = prep(t1)
        x2 = prep(t1ce)
        x3 = prep(t2)
        x4 = prep(flair)

        msk = mask_vol[:, :, z]
        msk = cv2.resize(msk, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        x = torch.from_numpy(np.stack([x1, x2, x3, x4], axis=0)).float()  # (4,H,W)
        y = torch.from_numpy(msk).long()                                   # (H,W)
        return x, y

# =========================
# Model
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)

# =========================
# Losses / Metrics
# =========================
def soft_dice_loss(logits, target, num_classes=5, eps=1e-6):
    """
    Multi-class soft Dice loss (excludes background channel 0).
    logits: (N,C,H,W)
    target: (N,H,W) with int labels [0..C-1]
    """
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs = probs[:, 1:]       # exclude background
    target_1h = target_1h[:, 1:]

    dims = (0, 2, 3)
    inter = (probs * target_1h).sum(dims)
    denom = probs.sum(dims) + target_1h.sum(dims)
    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()

@torch.no_grad()
def mean_dice(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        dices.append((2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps))
    return torch.stack(dices).mean().item()

# =========================
# Main
# =========================
def main():
    print("=== STEP 7.1 START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("Seed:", SEED, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(TRAIN_SPLIT, "r", encoding="utf-8") as f:
        train_ids = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    print(f"Train subjects={len(train_ids)} | Val subjects={len(val_ids)}", flush=True)

    train_ds = BraTS2024SliceDataset4Ch(
        subjects, train_ids,
        image_size=IMG_SIZE,
        tumor_only=TUMOR_ONLY_TRAIN,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM
    )
    val_ds = BraTS2024SliceDataset4Ch(
        subjects, val_ids,
        image_size=IMG_SIZE,
        tumor_only=False,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=False
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)}", flush=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda")
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda")
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    ce = nn.CrossEntropyLoss()

    # Cosine schedule over EPOCHS
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))

    best = -1.0

    for epoch in range(1, EPOCHS + 1):
        # -------- Train --------
        model.train()
        tr_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            if DEVICE == "cuda":
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
            else:
                x, y = x.to(DEVICE), y.to(DEVICE)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss_ce = ce(logits, y)
                loss_d  = soft_dice_loss(logits, y, num_classes=NUM_CLASSES)
                loss = W_CE * loss_ce + W_DICE * loss_d

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")

        tr_loss /= len(train_loader.dataset)

        # -------- Val --------
        model.eval()
        va_loss, va_dice, n = 0.0, 0.0, 0

        pbarv = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} VAL", leave=True)
        with torch.no_grad():
            for x, y in pbarv:
                if DEVICE == "cuda":
                    x = x.to(DEVICE, non_blocking=True)
                    y = y.to(DEVICE, non_blocking=True)
                else:
                    x, y = x.to(DEVICE), y.to(DEVICE)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss_ce = ce(logits, y)
                    loss_d  = soft_dice_loss(logits, y, num_classes=NUM_CLASSES)
                    loss = W_CE * loss_ce + W_DICE * loss_d

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice += mean_dice(pred, y, num_classes=NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= len(val_loader.dataset)
        va_dice /= max(n, 1)

        sched.step()

        print(
            f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_meanDice={va_dice:.4f} | lr={opt.param_groups[0]['lr']:.2e}",
            flush=True
        )

        if va_dice > best:
            best = va_dice
            ckpt = os.path.join(OUT_DIR, "best.pt")
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_dice": best}, ckpt)
            print("Saved:", ckpt, flush=True)

    print("=== STEP 7.1 DONE === Best val_meanDice:", best, flush=True)

if __name__ == "__main__":
    main()


=== STEP 7.1 START ===
Device: cuda
Torch: 2.5.1+cu121
Seed: 42
Train subjects=1143 | Val subjects=248
Train slices=44129 | Val slices=9920


Epoch 1/5 VAL: 100%|███████████████████████████████████████| 2480/2480 [1:48:48<00:00,  2.63s/it, vloss=0.5005]

Epoch 01 | train_loss=0.2831 | val_loss=0.4420 | val_meanDice=0.6688 | lr=4.52e-04
Saved: C:/GDNET/logs/step7_1_unet_4ch_amp_stable\best.pt



Epoch 2/5 VAL: 100%|███████████████████████████████████████| 2480/2480 [1:46:46<00:00,  2.58s/it, vloss=0.5068]

Epoch 02 | train_loss=0.2288 | val_loss=0.4369 | val_meanDice=0.6669 | lr=3.27e-04


...

Epoch 03 | train_loss=0.2133 | val_loss=0.4341 | val_meanDice=0.7185 | lr=1.73e-04
Saved: C:/GDNET/logs/step7_1_unet_4ch_amp_stable\best.pt



Epoch 4/5 VAL: 100%|███████████████████████████████████████| 2480/2480 [1:47:45<00:00,  2.61s/it, vloss=0.5002]

Epoch 04 | train_loss=0.2016 | val_loss=0.4310 | val_meanDice=0.7426 | lr=4.77e-05
Saved: C:/GDNET/logs/step7_1_unet_4ch_amp_stable\best.pt



Epoch 5/5 VAL: 100%|███████████████████████████████████████| 2480/2480 [1:49:10<00:00,  2.64s/it, vloss=0.6654]

Epoch 05 | train_loss=0.1938 | val_loss=0.4746 | val_meanDice=0.6073 | lr=0.00e+00
=== STEP 7.1 DONE === Best val_meanDice: 0.7425721217221102


In [ ]:
# C:/GDNET/experiments/step7_1b_unet_4ch_amp_dicefix_4060_16gb.py
# Step 7.1b (pre-GDNet): robust multi-class training config tuned for RTX 4060 + 16GB RAM
# - 4-channel input (T1/T1CE/T2/FLAIR) axial slices
# - CE + stable soft Dice (ignores empty classes per batch)
# - torch.amp (no deprecation warnings)
# - AMP enabled (mixed precision)
# - Grad clipping
# - Cosine LR schedule
# - tqdm
# - LRU caching (RAM-safe) for faster IO on 16GB RAM
# - DataLoader tuned: pin_memory, persistent_workers, prefetch_factor (if num_workers>0)

import os
import json
import random
from collections import OrderedDict

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

# =========================
# Repro / Speed knobs
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# =========================
# Paths
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = r"C:/GDNET/logs/step7_1b_unet_4ch_amp_dicefix_4060_16gb"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Hyperparams (RTX 4060 + 16GB RAM profile)
# =========================
IMG_SIZE = 256

# RTX 4060: start with 2. If you have headroom, try 3 or 4.
BATCH = 2

# Iterate with modest epochs; increase later.
EPOCHS = 5

# Stable for small batch + AMP
LR = 3e-4

NUM_CLASSES = 5

# Windows: if you get DataLoader hangs, set back to 0.
NUM_WORKERS = 0

# GPU transfer optimizations
PIN_MEMORY = True
USE_AMP = True

# Sampling
TUMOR_ONLY_TRAIN = True
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 32  # iteration speed; set None for full training

# 16GB RAM: do NOT full-cache; use LRU instead
CACHE_IN_RAM = False
LRU_CACHE_MAX_SUBJECTS = 6    # safe on 16GB RAM; adjust 4-8

# Loss weights (keep CE dominant)
W_CE = 0.7
W_DICE = 0.3

# Gradient clipping
MAX_GRAD_NORM = 1.0

# =========================
# Dataset
# =========================
class BraTS2024SliceDataset4Ch(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        tumor_only=True,
        tumor_min_frac=0.001,
        max_slices_per_subject=None,
        cache_in_ram=False,
        lru_cache_max_subjects=6,
    ):
        self.subjects = subjects_index
        self.sids = subject_ids
        self.image_size = image_size
        self.tumor_only = tumor_only
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        # RAM-safe LRU cache: stores only a few subjects
        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples = []  # (sid, z)

        # Build slice index using mask only (fast, deterministic)
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            candidate_z = []
            if self.tumor_only:
                for z in range(Z):
                    if (m[:, :, z] > 0).mean() >= self.tumor_min_frac:
                        candidate_z.append(z)
                if not candidate_z:
                    candidate_z = list(range(Z))
            else:
                candidate_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(candidate_z) > self.max_slices_per_subject:
                idx = np.linspace(0, len(candidate_z) - 1, self.max_slices_per_subject).astype(int)
                candidate_z = [candidate_z[i] for i in idx]

            for z in candidate_z:
                self.samples.append((sid, z))

        if len(self.samples) == 0:
            raise RuntimeError("No slices selected. Lower tumor_min_frac or set tumor_only=False.")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        # Full cache mode (not recommended for 16GB RAM; kept for compatibility)
        if self.cache_in_ram and sid in self._ram_cache:
            return self._ram_cache[sid]

        # LRU cache path
        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)

        data = (t1, t1ce, t2, flair, mask)

        # Store in LRU
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)

        # Evict oldest
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)

        return data

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = self._zscore(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)

        x1 = prep(t1)
        x2 = prep(t1ce)
        x3 = prep(t2)
        x4 = prep(flair)

        msk = mask_vol[:, :, z]
        msk = cv2.resize(msk, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        x = torch.from_numpy(np.stack([x1, x2, x3, x4], axis=0)).float()  # (4,H,W)
        y = torch.from_numpy(msk).long()                                   # (H,W)
        return x, y

# =========================
# Model
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)

# =========================
# Losses / Metrics
# =========================
def soft_dice_loss_ignore_empty(logits, target, num_classes=5, eps=1e-6):
    """
    Soft Dice loss that ignores classes absent in the GT for the batch.
    Excludes background (0).
    logits: (N,C,H,W), target: (N,H,W)
    """
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs_fg = probs[:, 1:]       # (N,C-1,H,W)
    targ_fg  = target_1h[:, 1:]

    dims = (0, 2, 3)
    inter = (probs_fg * targ_fg).sum(dims)
    denom = probs_fg.sum(dims) + targ_fg.sum(dims)

    dice = (2 * inter + eps) / (denom + eps)  # (C-1,)

    present = (targ_fg.sum(dims) > 0).float()
    if present.sum() < 1:
        return torch.tensor(0.0, device=logits.device)

    dice_present = (dice * present).sum() / (present.sum() + eps)
    return 1.0 - dice_present

@torch.no_grad()
def mean_dice(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        dices.append((2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps))
    return torch.stack(dices).mean().item()

# =========================
# Main
# =========================
def main():
    print("=== STEP 7.1b (4060+16GB) START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("Seed:", SEED, flush=True)
    print(f"BATCH={BATCH} IMG_SIZE={IMG_SIZE} NUM_WORKERS={NUM_WORKERS} LRU={LRU_CACHE_MAX_SUBJECTS}", flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(TRAIN_SPLIT, "r", encoding="utf-8") as f:
        train_ids = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    print(f"Train subjects={len(train_ids)} | Val subjects={len(val_ids)}", flush=True)

    train_ds = BraTS2024SliceDataset4Ch(
        subjects, train_ids,
        image_size=IMG_SIZE,
        tumor_only=TUMOR_ONLY_TRAIN,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS
    )
    val_ds = BraTS2024SliceDataset4Ch(
        subjects, val_ids,
        image_size=IMG_SIZE,
        tumor_only=False,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=False,
        lru_cache_max_subjects=2  # small cache ok for val
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)}", flush=True)

    # DataLoaders
    dl_common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    # prefetch_factor may error in some setups; keep guarded
    try:
        if NUM_WORKERS > 0:
            dl_common["prefetch_factor"] = 2
    except Exception:
        pass

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, **dl_common)
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, **dl_common)

    # Model + opt + loss
    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    ce = nn.CrossEntropyLoss()

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))

    best = -1.0

    for epoch in range(1, EPOCHS + 1):
        # -------- Train --------
        model.train()
        tr_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            if DEVICE == "cuda":
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
            else:
                x, y = x.to(DEVICE), y.to(DEVICE)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss_ce = ce(logits, y)
                loss_d  = soft_dice_loss_ignore_empty(logits, y, num_classes=NUM_CLASSES)
                loss = W_CE * loss_ce + W_DICE * loss_d

            scaler.scale(loss).backward()

            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(opt)
            scaler.update()

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")

        tr_loss /= len(train_loader.dataset)

        # -------- Val --------
        model.eval()
        va_loss, va_dice, n = 0.0, 0.0, 0

        pbarv = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} VAL", leave=True)
        with torch.no_grad():
            for x, y in pbarv:
                if DEVICE == "cuda":
                    x = x.to(DEVICE, non_blocking=True)
                    y = y.to(DEVICE, non_blocking=True)
                else:
                    x, y = x.to(DEVICE), y.to(DEVICE)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss_ce = ce(logits, y)
                    loss_d  = soft_dice_loss_ignore_empty(logits, y, num_classes=NUM_CLASSES)
                    loss = W_CE * loss_ce + W_DICE * loss_d

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice += mean_dice(pred, y, num_classes=NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= len(val_loader.dataset)
        va_dice /= max(n, 1)

        sched.step()

        print(
            f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_meanDice={va_dice:.4f} | lr={opt.param_groups[0]['lr']:.2e}",
            flush=True
        )

        if va_dice > best:
            best = va_dice
            ckpt = os.path.join(OUT_DIR, "best.pt")
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_dice": best}, ckpt)
            print("Saved:", ckpt, flush=True)

    print("=== STEP 7.1b DONE === Best val_meanDice:", best, flush=True)

if __name__ == "__main__":
    main()


=== STEP 7.1b (4060+16GB) START ===
Device: cuda
Torch: 2.5.1+cu121
Seed: 42
BATCH=2 IMG_SIZE=256 NUM_WORKERS=0 LRU=6
Train subjects=1143 | Val subjects=248
Train slices=35848 | Val slices=7936


Epoch 1/5 TRAIN:   0%|▏                                 | 76/17924 [01:44<6:18:49,  1.27s/it, loss=0.9000, lr=3.00e-04]

In [1]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, Dict, List

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# =========================
# GLOBAL CONFIG
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

BASE_OUT = r"C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0"
os.makedirs(BASE_OUT, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

# REQUIRED
NUM_WORKERS = 0
PIN_MEMORY = True

# Training length
# Option A (recommended): train up to TOTAL_EPOCHS (absolute epoch number)
TOTAL_EPOCHS = 10
# Option B: train EXTRA_EPOCHS beyond checkpoint epoch (set >0 to use)
EXTRA_EPOCHS = 0

# LR / schedule
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2

# Batch
BATCH = 2

# Dataset selection
MAX_SLICES_PER_SUBJECT = 32
TUMOR_MIN_FRAC = 0.001

# Sweep values
P_TUMOR_LIST = [0.5, 0.6, 0.7, 0.8]

# Cache
LRU_CACHE_MAX_SUBJECTS = 6
CACHE_IN_RAM = True

# Loss weights (DiceFix)
W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

# EMA
EMA_DECAY = 0.999

# Resume controls
RESUME_MODE = "best"     # "best" or "last"
SAVE_OPTIM_STATE = True  # save optimizer/scaler for true resume (new checkpoints)

# Perf knobs
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


# =========================
# Repro
# =========================
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# =========================
# LR schedule
# =========================
def lr_at_epoch(epoch0: int, total_epochs: int, base_lr: float, warmup_epochs: int):
    """
    epoch0: 0-indexed epoch number
    """
    if warmup_epochs > 0 and epoch0 < warmup_epochs:
        return base_lr * float(epoch0 + 1) / float(warmup_epochs)
    t = (epoch0 - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * t))


# =========================
# EMA (device-safe)
# =========================
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = float(decay)
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                # keep EMA tensor on same device as param
                self.shadow[n] = p.detach().clone().to(p.device)
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                # safety: if loaded from CPU ckpt, move to correct device
                if self.shadow[n].device != p.device:
                    self.shadow[n] = self.shadow[n].to(p.device)
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                # ensure matching device
                if self.shadow[n].device != p.device:
                    self.shadow[n] = self.shadow[n].to(p.device)
                p.copy_(self.shadow[n])

    @torch.no_grad()
    def restore(self, model: nn.Module):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


# =========================
# DiceFix loss + metric
# =========================
def soft_dice_loss_all_classes(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    tgt = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs_fg = probs[:, 1:]
    tgt_fg   = tgt[:, 1:]

    dims = (0, 2, 3)
    inter = (probs_fg * tgt_fg).sum(dims)
    p_sum = probs_fg.sum(dims)
    t_sum = tgt_fg.sum(dims)

    dice = (2.0 * inter + eps) / (p_sum + t_sum + eps)
    empty = (t_sum < eps) & (p_sum < eps)
    dice = torch.where(empty, torch.ones_like(dice), dice)
    return 1.0 - dice.mean()

@torch.no_grad()
def mean_dice(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps)
        dices.append(d)
    return torch.stack(dices).mean().item()


# =========================
# Dataset (4ch + mixed sampling)
# =========================
class BraTS2024SliceDataset4ChMixed(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        is_train: bool = True,
        p_tumor: float = 0.7,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = 32,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.is_train = bool(is_train)
        self.p_tumor = float(p_tumor)
        self.tumor_min_frac = float(tumor_min_frac)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.all_samples = []
        self.tumor_samples = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            all_z = list(range(Z))
            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z)-1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            tum_z = [z for z in all_z if (m[:, :, z] > 0).mean() >= self.tumor_min_frac]

            for z in all_z:
                self.all_samples.append((sid, z))
            for z in tum_z:
                self.tumor_samples.append((sid, z))

        if len(self.all_samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.all_samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        if self.is_train and (len(self.tumor_samples) > 0) and (random.random() < self.p_tumor):
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.all_samples[idx]

        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = self._zscore(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


# =========================
# Model (UNetSmall)
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, gn_groups=8, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot  = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3  = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1  = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)


# =========================
# Resume helpers (device-safe EMA)
# =========================
def pick_resume_path(out_dir: str) -> Optional[str]:
    best_p = os.path.join(out_dir, "best.pt")
    last_p = os.path.join(out_dir, "last.pt")
    mode = RESUME_MODE.lower().strip()

    if mode == "best":
        if os.path.exists(best_p):
            return best_p
        if os.path.exists(last_p):
            return last_p
    else:
        if os.path.exists(last_p):
            return last_p
        if os.path.exists(best_p):
            return best_p
    return None


def load_checkpoint(path: str, model: nn.Module, ema: EMA, opt=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)

    # move EMA to same device as model params
    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        dev = next(model.parameters()).device
        ema.shadow = {k: v.clone().to(dev) for k, v in ckpt["ema"].items()}

    # optional: optimizer/scaler for true resume
    if opt is not None and "opt" in ckpt and ckpt["opt"] is not None:
        try:
            opt.load_state_dict(ckpt["opt"])
        except Exception as e:
            print(f"[WARN] Could not load optimizer state: {e}", flush=True)

    if scaler is not None and "scaler" in ckpt and ckpt["scaler"] is not None:
        try:
            scaler.load_state_dict(ckpt["scaler"])
        except Exception as e:
            print(f"[WARN] Could not load scaler state: {e}", flush=True)

    start_epoch = int(ckpt.get("epoch", 0))  # last completed epoch (1-indexed)
    best = float(ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))
    return start_epoch, best


# =========================
# Train/eval one setting
# =========================
def run_one(p_tumor: float) -> Dict:
    run_name = f"ptumor_{p_tumor:.2f}"
    out_dir = os.path.join(BASE_OUT, run_name)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n=== RUN {run_name} START ===", flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(TRAIN_SPLIT, "r", encoding="utf-8") as f:
        train_ids = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    train_ds = BraTS2024SliceDataset4ChMixed(
        subjects, train_ids, image_size=IMG_SIZE,
        is_train=True, p_tumor=p_tumor,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )
    val_ds = BraTS2024SliceDataset4ChMixed(
        subjects, val_ids, image_size=IMG_SIZE,
        is_train=False, p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=2,
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)} | Tumor pool={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=0, pin_memory=(PIN_MEMORY and DEVICE == "cuda")
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=0, pin_memory=(PIN_MEMORY and DEVICE == "cuda")
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY)

    # ---- RESUME ----
    resume_path = pick_resume_path(out_dir)
    last_done_epoch = 0   # 1-indexed
    best = -1.0
    best_epoch = -1

    if resume_path is not None:
        last_done_epoch, best = load_checkpoint(resume_path, model, ema, opt=opt, scaler=scaler)
        best_epoch = last_done_epoch if best > -1 else -1
        print(f"[RESUME] Loaded {resume_path} (last_done_epoch={last_done_epoch}, best={best:.6f})", flush=True)

    # Determine end epoch
    if EXTRA_EPOCHS > 0:
        end_epoch = last_done_epoch + EXTRA_EPOCHS
    else:
        end_epoch = TOTAL_EPOCHS

    if last_done_epoch >= end_epoch:
        print(f"[SKIP] {run_name}: last_done_epoch ({last_done_epoch}) >= end_epoch ({end_epoch}). Nothing to do.", flush=True)
        return {
            "p_tumor": p_tumor,
            "best_val_meanDice_ema": float(best),
            "best_epoch": int(best_epoch),
            "out_dir": out_dir,
            "resumed_from": resume_path or "",
        }

    # Continue from next epoch
    for epoch_num in range(last_done_epoch + 1, end_epoch + 1):
        epoch0 = epoch_num - 1  # 0-indexed for schedule
        lr = lr_at_epoch(epoch0, end_epoch, BASE_LR, WARMUP_EPOCHS)
        for pg in opt.param_groups:
            pg["lr"] = lr

        # --- train ---
        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_loader, desc=f"{run_name} Ep {epoch_num}/{end_epoch} TRAIN", leave=True)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()

            ema.update(model)

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        tr_loss /= max(1, len(train_loader.dataset))

        # --- val on EMA ---
        model.eval()
        ema.apply_to(model)

        va_loss, va_dice_sum, n = 0.0, 0.0, 0
        with torch.no_grad():
            pbarv = tqdm(val_loader, desc=f"{run_name} Ep {epoch_num}/{end_epoch} VAL(EMA)", leave=True)
            for x, y in pbarv:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice_sum += mean_dice(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)
                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= max(1, len(val_loader.dataset))
        va_dice = va_dice_sum / max(1, n)

        ema.restore(model)

        print(
            f"{run_name} Epoch {epoch_num:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | "
            f"val_meanDice(EMA)={va_dice:.4f} | lr={lr:.2e}",
            flush=True
        )

        # ---- SAVE LAST (resume-friendly) ----
        last_path = os.path.join(out_dir, "last.pt")
        payload = {
            "model": model.state_dict(),
            "ema": ema.shadow,
            "epoch": epoch_num,            # last completed epoch
            "val_dice": float(va_dice),
            "best_val_dice": float(best),
        }
        if SAVE_OPTIM_STATE:
            payload["opt"] = opt.state_dict()
            payload["scaler"] = scaler.state_dict()

        torch.save(payload, last_path)

        # ---- SAVE BEST ----
        if va_dice > best:
            best = float(va_dice)
            best_epoch = int(epoch_num)
            best_path = os.path.join(out_dir, "best.pt")
            payload_best = dict(payload)
            payload_best["best_val_dice"] = float(best)
            torch.save(payload_best, best_path)
            print("Saved best:", best_path, flush=True)

    result = {
        "p_tumor": p_tumor,
        "best_val_meanDice_ema": float(best),
        "best_epoch": int(best_epoch),
        "out_dir": out_dir,
        "resumed_from": resume_path or "",
    }
    print(f"=== RUN {run_name} DONE === best={best:.6f} at epoch={best_epoch}", flush=True)
    return result


def write_summary_csv(rows: List[Dict], path: str):
    import csv
    keys = ["p_tumor", "best_val_meanDice_ema", "best_epoch", "out_dir", "resumed_from"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in keys})


def main():
    print("=== STEP 7.1d (RESUME) SWEEP START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print(f"Seed={SEED} BATCH={BATCH} IMG_SIZE={IMG_SIZE} WORKERS={NUM_WORKERS}", flush=True)
    print(f"TOTAL_EPOCHS={TOTAL_EPOCHS} EXTRA_EPOCHS={EXTRA_EPOCHS} BASE_LR={BASE_LR} WARMUP_EPOCHS={WARMUP_EPOCHS} EMA_DECAY={EMA_DECAY}", flush=True)
    print("Resume mode:", RESUME_MODE, flush=True)
    print("Sweep P_TUMOR:", P_TUMOR_LIST, flush=True)

    results = []
    for p in P_TUMOR_LIST:
        seed_all(SEED)  # reset between runs
        results.append(run_one(p))

    results_sorted = sorted(results, key=lambda d: d["best_val_meanDice_ema"], reverse=True)

    summary_path = os.path.join(BASE_OUT, "summary_resume.csv")
    write_summary_csv(results_sorted, summary_path)

    print("\n=== SWEEP SUMMARY (best-first) ===", flush=True)
    for r in results_sorted:
        print(
            f"P_TUMOR={r['p_tumor']:.2f} bestDice={r['best_val_meanDice_ema']:.6f} "
            f"bestEpoch={r['best_epoch']} dir={r['out_dir']} resumed_from={r['resumed_from']}",
            flush=True
        )

    print("\nSaved summary CSV:", summary_path, flush=True)
    print("=== STEP 7.1d (RESUME) SWEEP DONE ===", flush=True)


if __name__ == "__main__":
    main()


=== STEP 7.1d (RESUME) SWEEP START ===
Device: cuda
Torch: 2.5.1+cu121
Seed=42 BATCH=2 IMG_SIZE=256 WORKERS=0
TOTAL_EPOCHS=10 EXTRA_EPOCHS=0 BASE_LR=0.0002 WARMUP_EPOCHS=2 EMA_DECAY=0.999
Resume mode: best
Sweep P_TUMOR: [0.5, 0.6, 0.7, 0.8]

=== RUN ptumor_0.50 START ===
Train slices=36576 | Val slices=7936 | Tumor pool=12357
[RESUME] Loaded C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.50\best.pt (last_done_epoch=10, best=0.895030)
[SKIP] ptumor_0.50: last_done_epoch (10) >= end_epoch (10). Nothing to do.

=== RUN ptumor_0.60 START ===


C:\Users\admin\AppData\Local\Temp\ipykernel_6788\4014965897.py:356: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location="cpu")


Train slices=36576 | Val slices=7936 | Tumor pool=12357
[RESUME] Loaded C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.60\best.pt (last_done_epoch=10, best=0.886073)
[SKIP] ptumor_0.60: last_done_epoch (10) >= end_epoch (10). Nothing to do.

=== RUN ptumor_0.70 START ===
Train slices=36576 | Val slices=7936 | Tumor pool=12357
[RESUME] Loaded C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.70\best.pt (last_done_epoch=9, best=0.890596)


ptumor_0.70 Ep 10/10 VAL(EMA): 100%|█████████████████████████| 3968/3968 [03:28<00:00, 19.04it/s, vloss=0.2456]

ptumor_0.70 Epoch 10 | train_loss=0.1469 | val_loss=0.2220 | val_meanDice(EMA)=0.8887 | lr=7.61e-06


...

ptumor_0.80 Epoch 09 | train_loss=0.1824 | val_loss=0.3209 | val_meanDice(EMA)=0.8775 | lr=2.93e-05


Saved best: C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.80\best.pt


ptumor_0.80 Ep 10/10 VAL(EMA): 100%|█████████████████████████| 3968/3968 [03:31<00:00, 18.76it/s, vloss=0.3634]

ptumor_0.80 Epoch 10 | train_loss=0.1780 | val_loss=0.3204 | val_meanDice(EMA)=0.8813 | lr=7.61e-06


Saved best: C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.80\best.pt
=== RUN ptumor_0.80 DONE === best=0.881281 at epoch=10

=== SWEEP SUMMARY (best-first) ===
P_TUMOR=0.50 bestDice=0.895030 bestEpoch=10 dir=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.50 resumed_from=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.50\best.pt
P_TUMOR=0.70 bestDice=0.890596 bestEpoch=9 dir=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.70 resumed_from=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.70\best.pt
P_TUMOR=0.60 bestDice=0.886073 bestEpoch=10 dir=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.60 resumed_from=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.60\best.pt
P_TUMOR=0.80 bestDice=0.881281 bestEpoch=10 dir=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.80 resumed_from=C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0\ptumor_0.80\best.pt

Saved summary CSV: C:/GDNET/logs/step7_1d_ptumor_sweep_ema_worke

In [2]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# =========================
# STEP 7.2 — EVAL + VIS (VAL + TEST)
# Uses BEST baseline from Step 7.1d (EMA checkpoint)
# =========================

# ---- Paths (edit if needed)
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

# Best run from your sweep (P_TUMOR=0.50)
CKPT_BEST = r"C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step7_2_eval_baseline_ptumor_0.50"
os.makedirs(OUT_DIR, exist_ok=True)
VIS_DIR_VAL = os.path.join(OUT_DIR, "vis_val")
VIS_DIR_TEST = os.path.join(OUT_DIR, "vis_test")
os.makedirs(VIS_DIR_VAL, exist_ok=True)
os.makedirs(VIS_DIR_TEST, exist_ok=True)

# ---- Runtime config
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

BATCH = 2
NUM_WORKERS = 0          # per your requirement
PIN_MEMORY = True

# Visualization
N_VIS_SUBJECTS = 10       # number of subjects to sample for visualization
N_VIS_SLICES_PER_SUBJECT = 5
OVERLAY_ALPHA = 0.45


# =========================
# Repro
# =========================
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# =========================
# Small utilities
# =========================
def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def to_uint8_01(x: np.ndarray) -> np.ndarray:
    # x expected float, will be normalized to 0..255
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx <= mn + 1e-12:
        return np.zeros_like(x, dtype=np.uint8)
    y = (x - mn) / (mx - mn)
    return (y * 255.0).clip(0, 255).astype(np.uint8)

def stack_montage_4ch(x4: np.ndarray) -> np.ndarray:
    """
    x4: (4,H,W) float
    return: (H, 4W, 3) uint8
    """
    tiles = []
    for c in range(4):
        tiles.append(cv2.cvtColor(to_uint8_01(x4[c]), cv2.COLOR_GRAY2BGR))
    return np.concatenate(tiles, axis=1)

def colorize_mask(mask: np.ndarray) -> np.ndarray:
    """
    mask: (H,W) int
    returns BGR uint8
    """
    h, w = mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)

    # BGR colors (distinct)
    # 0 background stays black
    colors = {
        1: (0, 255, 0),     # green
        2: (255, 0, 0),     # blue
        3: (0, 255, 255),   # yellow
        4: (0, 0, 255),     # red
    }
    for k, col in colors.items():
        out[mask == k] = col
    return out

def overlay_mask_on_gray(gray_u8: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """
    gray_u8: (H,W) uint8
    mask: (H,W) int
    """
    base = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2BGR)
    col = colorize_mask(mask)
    # only overlay where mask>0
    m = (mask > 0).astype(np.float32)[..., None]
    blended = base.astype(np.float32) * (1.0 - alpha * m) + col.astype(np.float32) * (alpha * m)
    return blended.clip(0, 255).astype(np.uint8)


# =========================
# Dataset (subject-based for eval)
# =========================
class BraTS2024SliceDataset4ChEval(Dataset):
    """
    Deterministic slice listing for evaluation:
    - For each subject, take ALL slices (or max_slices_per_subject evenly spaced)
    - Returns (x4, y, sid, z)
    """
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z)-1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found in eval dataset.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = zscore2d(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z)


# =========================
# Model (UNetSmall)
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, gn_groups=8, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot  = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3  = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1  = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)


# =========================
# EMA weight loader (use EMA shadow for eval)
# =========================
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError(f"Checkpoint has no EMA dict: {ckpt_path}")

    ema_sd = ckpt["ema"]
    # ema dict is name->tensor; build state_dict-like mapping for params only
    model_sd = model.state_dict()
    # Some entries in model_sd are buffers (e.g., GN running stats not used here), but GroupNorm has no running stats.
    # We'll replace param tensors where available.
    for k in model_sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            model_sd[k] = ema_sd[k]
    model.load_state_dict(model_sd, strict=True)

    meta = {
        "epoch": int(ckpt.get("epoch", -1)),
        "val_dice": float(ckpt.get("val_dice", ckpt.get("best_val_dice", -1.0))),
        "best_val_dice": float(ckpt.get("best_val_dice", -1.0)),
    }
    return meta


# =========================
# Metrics (per-class + BraTS composites)
# =========================
@torch.no_grad()
def dice_binary(pred_bin: torch.Tensor, tgt_bin: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (pred_bin & tgt_bin).sum().item()
    ps = pred_bin.sum().item()
    ts = tgt_bin.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))

@torch.no_grad()
def compute_metrics_batch(pred: torch.Tensor, tgt: torch.Tensor, num_classes: int = 5) -> Dict[str, float]:
    """
    pred/tgt: (B,H,W) int64
    returns sums (not averaged) for accumulation; caller will average by count
    """
    out = {}

    # per-class dice for labels 1..(C-1)
    for c in range(1, num_classes):
        out[f"dice_c{c}"] = dice_binary((pred == c), (tgt == c))

    # BraTS composites (label-set based)
    # WT = {1,2,4} (+3 if present in your data)
    # TC = {1,4} (+3 if you use it as tumor core; usually label 3 not in BraTS 2021+)
    # ET = {4}
    pred_wt = (pred == 1) | (pred == 2) | (pred == 4) | (pred == 3)
    tgt_wt  = (tgt  == 1) | (tgt  == 2) | (tgt  == 4) | (tgt  == 3)
    out["dice_WT"] = dice_binary(pred_wt, tgt_wt)

    pred_tc = (pred == 1) | (pred == 4) | (pred == 3)
    tgt_tc  = (tgt  == 1) | (tgt  == 4) | (tgt  == 3)
    out["dice_TC"] = dice_binary(pred_tc, tgt_tc)

    out["dice_ET"] = dice_binary((pred == 4), (tgt == 4))

    # mean dice (foreground classes 1..4)
    out["dice_mean_fg"] = float(np.mean([out[f"dice_c{c}"] for c in range(1, num_classes)]))
    return out


def _accum_init() -> Dict[str, float]:
    keys = [f"dice_c{c}" for c in range(1, NUM_CLASSES)] + ["dice_WT", "dice_TC", "dice_ET", "dice_mean_fg"]
    return {k: 0.0 for k in keys}

def _accum_add(acc: Dict[str, float], m: Dict[str, float]):
    for k, v in m.items():
        acc[k] += float(v)


# =========================
# Evaluation loop
# =========================
@torch.no_grad()
def evaluate_split(subjects: dict, split_ids: list, split_name: str, vis_dir: str) -> Dict[str, float]:
    ds = BraTS2024SliceDataset4ChEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,     # evaluate all slices
        cache_in_ram=True,
        lru_cache_max_subjects=3
    )

    loader = DataLoader(
        ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda")
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] Loaded EMA ckpt: epoch={meta['epoch']} best_val_dice={meta['best_val_dice']:.6f}", flush=True)

    # ---- metrics accumulation
    acc = _accum_init()
    count = 0

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

        pred = torch.argmax(logits, dim=1)

        # batch metrics (average within batch)
        mb = compute_metrics_batch(pred, y, num_classes=NUM_CLASSES)
        _accum_add(acc, mb)
        count += 1

        pbar.set_postfix(mean_fg=f"{mb['dice_mean_fg']:.4f}", WT=f"{mb['dice_WT']:.4f}", ET=f"{mb['dice_ET']:.4f}")

    # average over batches
    out = {k: (v / max(1, count)) for k, v in acc.items()}
    out["batches"] = int(count)
    out["split"] = split_name
    return out


# =========================
# Visualization export (random subjects + slices)
# =========================
def export_visuals(subjects: dict, split_ids: list, split_name: str, vis_dir: str):
    seed_all(SEED + 123)
    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] Visuals: using EMA epoch={meta['epoch']} best_val_dice={meta['best_val_dice']:.6f}", flush=True)

    chosen = split_ids[:]
    random.shuffle(chosen)
    chosen = chosen[:min(N_VIS_SUBJECTS, len(chosen))]

    for sid in tqdm(chosen, desc=f"VIS {split_name}", leave=True):
        p = subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)

        Z = mask.shape[2]
        # pick mix of tumor/non-tumor slices
        tumor_z = [z for z in range(Z) if (mask[:, :, z] > 0).mean() >= 0.001]
        non_z = [z for z in range(Z) if (mask[:, :, z] > 0).mean() < 0.001]

        picks = []
        if len(tumor_z) > 0:
            picks += random.sample(tumor_z, k=min(len(tumor_z), max(1, N_VIS_SLICES_PER_SUBJECT // 2)))
        if len(non_z) > 0 and len(picks) < N_VIS_SLICES_PER_SUBJECT:
            picks += random.sample(non_z, k=min(len(non_z), N_VIS_SLICES_PER_SUBJECT - len(picks)))

        picks = sorted(set(int(z) for z in picks))
        for z in picks:
            def prep(vol):
                s = zscore2d(vol[:, :, z])
                return cv2.resize(s, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

            x4 = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
            y2 = cv2.resize(mask[:, :, z], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST).astype(np.int64)

            xt = torch.from_numpy(x4).unsqueeze(0).to(DEVICE)
            if DEVICE == "cuda":
                xt = xt.contiguous(memory_format=torch.channels_last)

            with torch.no_grad(), autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(xt)
                pred = torch.argmax(logits, dim=1).squeeze(0).detach().cpu().numpy().astype(np.int64)

            # montage + overlays (use FLAIR as base for overlay)
            base_gray = to_uint8_01(x4[3])  # flair
            overlay_gt = overlay_mask_on_gray(base_gray, y2, alpha=OVERLAY_ALPHA)
            overlay_pr = overlay_mask_on_gray(base_gray, pred, alpha=OVERLAY_ALPHA)
            montage = stack_montage_4ch(x4)

            # compose final canvas
            # row1: 4ch montage
            # row2: GT overlay | Pred overlay
            h, w, _ = montage.shape
            row2 = np.concatenate([overlay_gt, overlay_pr], axis=1)
            # make same width as montage
            if row2.shape[1] != montage.shape[1]:
                # pad or resize row2 to match montage width
                row2 = cv2.resize(row2, (montage.shape[1], overlay_gt.shape[0]), interpolation=cv2.INTER_NEAREST)

            canvas = np.concatenate([montage, row2], axis=0)

            fn = f"{sid}_z{z:03d}.png"
            cv2.imwrite(os.path.join(vis_dir, fn), canvas)


def save_json(obj: Dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def main():
    print("=== STEP 7.2 START (VAL+TEST EVAL + VIS) ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)
    print(f"BATCH={BATCH} IMG_SIZE={IMG_SIZE} WORKERS={NUM_WORKERS} AMP={USE_AMP}", flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)

    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    # EVAL
    val_metrics = evaluate_split(subjects, val_ids, "VAL", VIS_DIR_VAL)
    test_metrics = evaluate_split(subjects, test_ids, "TEST", VIS_DIR_TEST)

    print("\n=== METRICS (EMA) ===", flush=True)
    for name, m in [("VAL", val_metrics), ("TEST", test_metrics)]:
        print(f"\n[{name}] batches={m['batches']}", flush=True)
        print(f"  dice_mean_fg: {m['dice_mean_fg']:.6f}", flush=True)
        print(f"  dice_WT:      {m['dice_WT']:.6f}", flush=True)
        print(f"  dice_TC:      {m['dice_TC']:.6f}", flush=True)
        print(f"  dice_ET:      {m['dice_ET']:.6f}", flush=True)
        for c in range(1, NUM_CLASSES):
            print(f"  dice_c{c}:     {m[f'dice_c{c}']:.6f}", flush=True)

    # SAVE METRICS
    save_json(val_metrics, os.path.join(OUT_DIR, "metrics_val.json"))
    save_json(test_metrics, os.path.join(OUT_DIR, "metrics_test.json"))

    # VISUALS
    print("\n=== EXPORT VISUALS ===", flush=True)
    print("VAL ->", VIS_DIR_VAL, flush=True)
    export_visuals(subjects, val_ids, "VAL", VIS_DIR_VAL)
    print("TEST ->", VIS_DIR_TEST, flush=True)
    export_visuals(subjects, test_ids, "TEST", VIS_DIR_TEST)

    print("\nSaved metrics to:", OUT_DIR, flush=True)
    print("=== STEP 7.2 DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 7.2 START (VAL+TEST EVAL + VIS) ===
Device: cuda
Torch: 2.5.1+cu121
CKPT: C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt
BATCH=2 IMG_SIZE=256 WORKERS=0 AMP=True
[VAL] Loaded EMA ckpt: epoch=10 best_val_dice=0.895030


C:\Users\admin\AppData\Local\Temp\ipykernel_6788\4263166148.py:268: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL: 

[TEST] Loaded EMA ckpt: epoch=10 best_val_dice=0.895030


EVAL TEST: 100%|███████████████████| 20930/20930 [07:39<00:00, 45.52it/s, ET=1.0000, WT=1.0000, mean_fg=1.0000]


=== METRICS (EMA) ===



[VAL] batches=22568
  dice_mean_fg: 0.903338
  dice_WT:      0.845767
  dice_TC:      0.887821
  dice_ET:      0.874301
  dice_c1:     0.959341
  dice_c2:     0.849873
  dice_c3:     0.929836
  dice_c4:     0.874301

[TEST] batches=20930
  dice_mean_fg: 0.900852
  dice_WT:      0.834268
  dice_TC:      0.889330
  dice_ET:      0.874230
  dice_c1:     0.963280
  dice_c2:     0.835674
  dice_c3:     0.930223
  dice_c4:     0.874230

=== EXPORT VISUALS ===
VAL -> C:/GDNET/logs/step7_2_eval_baseline_ptumor_0.50\vis_val
[VAL] Visuals: using EMA epoch=10 best_val_dice=0.895030


VIS VAL: 100%|█████████████████████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.21it/s]

TEST -> C:/GDNET/logs/step7_2_eval_baseline_ptumor_0.50\vis_test
[TEST] Visuals: using EMA epoch=10 best_val_dice=0.895030



VIS TEST: 100%|████████████████████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.16it/s]


Saved metrics to: C:/GDNET/logs/step7_2_eval_baseline_ptumor_0.50
=== STEP 7.2 DONE ===


In [3]:
import json, nibabel as nib, numpy as np

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"

subjects = json.load(open(INDEX, "r"))
val_ids = json.load(open(VAL_SPLIT, "r"))

labs = set()
for sid in val_ids[:50]:
    m = nib.load(subjects[sid]["mask"]).get_fdata().astype(np.int16)
    labs.update(np.unique(m).tolist())
print("Unique labels in sample:", sorted(labs))

Unique labels in sample: [0, 1, 2, 3, 4]


In [4]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# =========================
# STEP 7.2b — AUDIT-GRADE EVAL (VAL + TEST)
# - Slice-averaged Dice
# - Positive-only Dice (GT-present)
# - Correct composites for labels [0,1,2,3,4]
# =========================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

# Best run from your sweep (P_TUMOR=0.50)
CKPT_BEST = r"C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step7_2b_eval_audit_ptumor_0.50"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

BATCH = 2
NUM_WORKERS = 0          # required
PIN_MEMORY = True


# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# -------------------------
# Preprocess
# -------------------------
def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# -------------------------
# Dataset for eval (deterministic)
# -------------------------
class BraTS2024SliceDataset4ChEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z)-1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = zscore2d(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z)


# -------------------------
# Model (UNetSmall)
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, gn_groups=8, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot  = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3  = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1  = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)


# -------------------------
# Load EMA weights into model
# -------------------------
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA weights dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)
    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best_val_dice": float(ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0))),
    }


# -------------------------
# Metrics (slice-wise)
# -------------------------
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))

def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    # Labels are [0,1,2,3,4] in your dataset
    WT = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
    TC = (lbl == 1) | (lbl == 3) | (lbl == 4)
    ET = (lbl == 4)
    return {"WT": WT, "TC": TC, "ET": ET}

def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    s = {}
    for k in keys:
        s[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return s

def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset4ChEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3
    )
    loader = DataLoader(
        ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best_val_dice={meta['best_val_dice']:.6f}", flush=True)

    stats = init_stats()

    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        all_mean = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        pos_mean = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(all_mean), float(pos_mean), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {"dice_all": a, "dice_pos_only": p, "n_pos_slices": npos}

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def main():
    print("=== STEP 7.2b START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    for rep in [val_report, test_report]:
        name = rep["split"]
        m = rep["metrics"]
        print(f"\n[{name}]", flush=True)
        for k in ["mean_fg", "WT", "TC", "ET"]:
            print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})", flush=True)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 7.2b DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 7.2b START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt
[VAL] EMA loaded: epoch=10 best_val_dice=0.895030


C:\Users\admin\AppData\Local\Temp\ipykernel_6788\2812274039.py:203: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL (

[TEST] EMA loaded: epoch=10 best_val_dice=0.895030


EVAL TEST (slice-avg): 100%|█████████████████████████████| 20930/20930 [09:40<00:00, 36.08it/s, mean_fg=0.9133]


=== SUMMARY (dice_all / dice_pos_only) ===



[VAL]
  mean_fg: 0.916366 / 0.817695 (npos=16476)
  WT     : 0.868622 / 0.805846 (npos=16476)
  TC     : 0.904938 / 0.757526 (npos=9839)
  ET     : 0.893853 / 0.698255 (npos=6847)

[TEST]
  mean_fg: 0.913293 / 0.805831 (npos=14364)
  WT     : 0.858864 / 0.790243 (npos=14364)
  TC     : 0.904767 / 0.739571 (npos=9490)
  ET     : 0.891673 / 0.663476 (npos=6661)

Saved: C:/GDNET/logs/step7_2b_eval_audit_ptumor_0.50
=== STEP 7.2b DONE ===


In [2]:
# ============================================================
# STEP 7.3 — Tumor-centered cropping (spatial sampling) + EMA + Mixed Sampling
# - Keeps your Step 7.1c/7.1d pipeline style
# - workers = 0 (required)
# - AMP + channels_last + cudnn benchmark
# - Resume from best.pt automatically (optional)
# - Saves best.pt (EMA weights) and last.pt each epoch
# ============================================================

import os, json, math, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# =========================
# CONFIG
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

# Choose one p_tumor (best from your sweep was 0.50)
P_TUMOR = 0.50

OUT_DIR = rf"C:/GDNET/logs/step7_3_crop_ema_mixed_workers0_ptumor_{P_TUMOR:.2f}"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

# REQUIRED
NUM_WORKERS = 0
PIN_MEMORY = True

# Train
EPOCHS = 12                 # slightly longer
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2

BATCH = 2

# Slice selection
MAX_SLICES_PER_SUBJECT = 32
TUMOR_MIN_FRAC = 0.001

# Cache
LRU_CACHE_MAX_SUBJECTS = 6
CACHE_IN_RAM = True

# Loss
W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

# EMA
EMA_DECAY = 0.999

# Spatial sampling (NEW in Step 7.3)
# Crop a patch (CROP_SIZE) then resize back to IMG_SIZE
CROP_SIZE = 192
P_CROP_TUMOR = 0.75         # probability to do tumor-centered crop on a training sample
JITTER = 16                 # pixels jitter around tumor center (in original resized 256 space)

# Resume behavior
RESUME_IF_EXISTS = True     # if best.pt exists in OUT_DIR, resume


# Perf knobs
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


# =========================
# Repro
# =========================
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# =========================
# LR schedule
# =========================
def lr_at_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int):
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    t = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * t))


# =========================
# EMA
# =========================
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device: str = "cpu"):
        self.decay = float(decay)
        self.device = device
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.detach().to(self.device), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model: nn.Module):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


# =========================
# Losses / metrics
# =========================
def soft_dice_loss_all_classes(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    tgt = F.one_hot(target, num_classes=num_classes).permute(0,3,1,2).float()

    probs_fg = probs[:, 1:]
    tgt_fg   = tgt[:, 1:]

    dims = (0,2,3)
    inter = (probs_fg * tgt_fg).sum(dims)
    p_sum = probs_fg.sum(dims)
    t_sum = tgt_fg.sum(dims)

    dice = (2.0 * inter + eps) / (p_sum + t_sum + eps)
    empty = (t_sum < eps) & (p_sum < eps)
    dice = torch.where(empty, torch.ones_like(dice), dice)
    return 1.0 - dice.mean()

@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps)
        dices.append(d)
    return torch.stack(dices).mean().item()


# =========================
# Dataset — 4ch + mixed sampling + tumor-centered crop (NEW)
# =========================
class BraTS2024SliceDataset4ChMixedCrop(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        is_train: bool = True,
        p_tumor: float = 0.5,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = 32,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 6,
        crop_size: int = 192,
        p_crop_tumor: float = 0.75,
        jitter: int = 16,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.is_train = bool(is_train)
        self.p_tumor = float(p_tumor)
        self.tumor_min_frac = float(tumor_min_frac)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self.crop_size = int(crop_size)
        self.p_crop_tumor = float(p_crop_tumor)
        self.jitter = int(jitter)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.all_samples: List[Tuple[str, int]] = []
        self.tumor_samples: List[Tuple[str, int]] = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            all_z = list(range(Z))
            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z)-1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            tum_z = [z for z in all_z if (m[:, :, z] > 0).mean() >= self.tumor_min_frac]

            for z in all_z:
                self.all_samples.append((sid, int(z)))
            for z in tum_z:
                self.tumor_samples.append((sid, int(z)))

        if len(self.all_samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.all_samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _resize_slice(self, vol2d: np.ndarray) -> np.ndarray:
        s = self._zscore(vol2d)
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def _resize_mask(self, m2d: np.ndarray) -> np.ndarray:
        return cv2.resize(m2d, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

    def _crop_centered(self, x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        x: (C,H,W) float32, y: (H,W) int64, both in image_size space
        - If tumor exists, crop around tumor bbox center (+ jitter)
        - Else random crop
        - Crop_size -> resize back to image_size
        """
        H = W = self.image_size
        cs = self.crop_size
        if cs >= H:
            return x, y

        if (y > 0).any():
            ys, xs = np.where(y > 0)
            cy = int(np.mean(ys))
            cx = int(np.mean(xs))
            # jitter
            cy = int(np.clip(cy + random.randint(-self.jitter, self.jitter), 0, H-1))
            cx = int(np.clip(cx + random.randint(-self.jitter, self.jitter), 0, W-1))
        else:
            cy = random.randint(0, H-1)
            cx = random.randint(0, W-1)

        y1 = int(np.clip(cy - cs//2, 0, H - cs))
        x1 = int(np.clip(cx - cs//2, 0, W - cs))
        y2 = y1 + cs
        x2 = x1 + cs

        x_crop = x[:, y1:y2, x1:x2]  # (C,cs,cs)
        y_crop = y[y1:y2, x1:x2]     # (cs,cs)

        # resize back to IMG_SIZE
        x_out = np.stack(
            [cv2.resize(x_crop[c], (H, W), interpolation=cv2.INTER_AREA) for c in range(x_crop.shape[0])],
            axis=0
        ).astype(np.float32)
        y_out = cv2.resize(y_crop, (H, W), interpolation=cv2.INTER_NEAREST).astype(np.int64)
        return x_out, y_out

    def __getitem__(self, idx):
        # Mixed sampling at slice selection
        if self.is_train and (len(self.tumor_samples) > 0) and (random.random() < self.p_tumor):
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.all_samples[idx]

        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        x = np.stack([
            self._resize_slice(t1[:, :, z]),
            self._resize_slice(t1ce[:, :, z]),
            self._resize_slice(t2[:, :, z]),
            self._resize_slice(flair[:, :, z]),
        ], axis=0).astype(np.float32)

        y = self._resize_mask(mask_vol[:, :, z])

        # NEW: spatial crop on train
        if self.is_train and (random.random() < self.p_crop_tumor):
            x, y = self._crop_centered(x, y)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


# =========================
# Model (UNetSmall)
# =========================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, gn_groups=8, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot  = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3  = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1  = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)


# =========================
# Checkpoint IO (resume)
# =========================
def save_ckpt(path: str, model: nn.Module, ema: EMA, opt, scaler, epoch: int, best: float):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().clone().cpu() for k, v in ema.shadow.items()},
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best": float(best),
    }, path)

def load_ckpt(path: str, model: nn.Module, ema: EMA, opt=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)

    # load EMA shadow to the same device as EMA shadow storage
    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)

    if opt is not None and "opt" in ckpt:
        opt.load_state_dict(ckpt["opt"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0))
    best = float(ckpt.get("best", -1.0))
    return start_epoch, best


# =========================
# Main training
# =========================
def main():
    print("=== STEP 7.3 START (CROP + EMA + MIXED, workers=0) ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print(f"Seed={SEED} BATCH={BATCH} IMG_SIZE={IMG_SIZE} WORKERS={NUM_WORKERS}", flush=True)
    print(f"P_TUMOR={P_TUMOR} P_CROP_TUMOR={P_CROP_TUMOR} CROP_SIZE={CROP_SIZE} JITTER={JITTER}", flush=True)

    subjects = json.load(open(INDEX, "r", encoding="utf-8"))
    train_ids = json.load(open(TRAIN_SPLIT, "r", encoding="utf-8"))
    val_ids   = json.load(open(VAL_SPLIT, "r", encoding="utf-8"))

    train_ds = BraTS2024SliceDataset4ChMixedCrop(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
        crop_size=CROP_SIZE,
        p_crop_tumor=P_CROP_TUMOR,
        jitter=JITTER,
    )
    val_ds = BraTS2024SliceDataset4ChMixedCrop(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
        crop_size=IMG_SIZE,
        p_crop_tumor=0.0,
        jitter=0,
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)} | Tumor pool(train)={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))

    # Store EMA on GPU to avoid device-mismatch and keep it fast
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0
    start_epoch = 0

    best_path = os.path.join(OUT_DIR, "best.pt")
    last_path = os.path.join(OUT_DIR, "last.pt")

    if RESUME_IF_EXISTS and os.path.exists(best_path):
        start_epoch, best = load_ckpt(best_path, model, ema, opt=opt, scaler=scaler)
        print(f"Resumed from best.pt: start_epoch={start_epoch} best={best:.6f}", flush=True)

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_at_epoch(epoch, EPOCHS, BASE_LR, WARMUP_EPOCHS)
        for pg in opt.param_groups:
            pg["lr"] = lr

        # --- train ---
        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()

            ema.update(model)

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        tr_loss /= max(1, len(train_loader.dataset))

        # --- val on EMA ---
        model.eval()
        ema.apply_to(model)

        va_loss, va_dice_sum, n = 0.0, 0.0, 0
        with torch.no_grad():
            pbarv = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} VAL(EMA)", leave=True)
            for x, y in pbarv:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice_sum += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)
                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= max(1, len(val_loader.dataset))
        va_dice = va_dice_sum / max(1, n)

        ema.restore(model)

        print(f"Epoch {epoch+1:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_meanDiceFG(EMA)={va_dice:.4f} | lr={lr:.2e}", flush=True)

        # always save last
        save_ckpt(last_path, model, ema, opt, scaler, epoch+1, best)

        if va_dice > best:
            best = va_dice
            save_ckpt(best_path, model, ema, opt, scaler, epoch+1, best)
            print("Saved best:", best_path, flush=True)

    print("=== STEP 7.3 DONE === Best val_meanDiceFG(EMA):", best, flush=True)
    print("Out:", OUT_DIR, flush=True)


if __name__ == "__main__":
    main()

=== STEP 7.3 START (CROP + EMA + MIXED, workers=0) ===
Device: cuda
Torch: 2.5.1+cu121
Seed=42 BATCH=2 IMG_SIZE=256 WORKERS=0
P_TUMOR=0.5 P_CROP_TUMOR=0.75 CROP_SIZE=192 JITTER=16
Train slices=36576 | Val slices=7936 | Tumor pool(train)=12357
Resumed from best.pt: start_epoch=10 best=0.887242


C:\Users\admin\AppData\Local\Temp\ipykernel_400\2624877816.py:407: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location="cpu")
Ep 11/12 VAL(EMA

Epoch 11 | train_loss=0.2191 | val_loss=0.3217 | val_meanDiceFG(EMA)=0.8861 | lr=1.91e-05



Ep 12/12 VAL(EMA): 100%|████████████████████████████████████████████████████| 3968/3968 [03:30<00:00, 18.84it/s, vloss=0.3750]

Epoch 12 | train_loss=0.2186 | val_loss=0.3216 | val_meanDiceFG(EMA)=0.8902 | lr=4.89e-06


Saved best: C:/GDNET/logs/step7_3_crop_ema_mixed_workers0_ptumor_0.50\best.pt
=== STEP 7.3 DONE === Best val_meanDiceFG(EMA): 0.8901945455300231
Out: C:/GDNET/logs/step7_3_crop_ema_mixed_workers0_ptumor_0.50


In [1]:
# ============================================================
# STEP 7.3b — Crop-phase training with LR reset
# - Load model+EMA from INIT_CKPT
# - Reset optimizer/scaler/schedule (key!)
# - workers=0
# ============================================================

import os, json, math, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# =========================
# CONFIG
# =========================
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

# Init from Step 7.1d best (recommended)
INIT_CKPT = r"C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt"

P_TUMOR = 0.50
OUT_DIR = rf"C:/GDNET/logs/step7_3b_crop_phase_lrreset_ptumor_{P_TUMOR:.2f}"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

NUM_WORKERS = 0      # required
PIN_MEMORY = True

# Crop-phase training length (short but effective)
EPOCHS = 8
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2
BATCH = 2

MAX_SLICES_PER_SUBJECT = 32
TUMOR_MIN_FRAC = 0.001

LRU_CACHE_MAX_SUBJECTS = 6
CACHE_IN_RAM = True

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999

# Spatial sampling
CROP_SIZE = 192
P_CROP_TUMOR = 0.85   # slightly stronger than 0.75
JITTER = 24           # slightly more exploration

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def lr_at_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int):
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    t = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * t))


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device: str = "cpu"):
        self.decay = float(decay)
        self.device = device
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.detach().to(self.device), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model: nn.Module):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


def soft_dice_loss_all_classes(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    tgt = F.one_hot(target, num_classes=num_classes).permute(0,3,1,2).float()

    probs_fg = probs[:, 1:]
    tgt_fg   = tgt[:, 1:]

    dims = (0,2,3)
    inter = (probs_fg * tgt_fg).sum(dims)
    p_sum = probs_fg.sum(dims)
    t_sum = tgt_fg.sum(dims)

    dice = (2.0 * inter + eps) / (p_sum + t_sum + eps)
    empty = (t_sum < eps) & (p_sum < eps)
    dice = torch.where(empty, torch.ones_like(dice), dice)
    return 1.0 - dice.mean()

@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2*(p*t).sum()+eps)/(p.sum()+t.sum()+eps)
        dices.append(d)
    return torch.stack(dices).mean().item()


class BraTS2024SliceDataset4ChMixedCrop(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        is_train: bool = True,
        p_tumor: float = 0.5,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = 32,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 6,
        crop_size: int = 192,
        p_crop_tumor: float = 0.85,
        jitter: int = 24,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.is_train = bool(is_train)
        self.p_tumor = float(p_tumor)
        self.tumor_min_frac = float(tumor_min_frac)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self.crop_size = int(crop_size)
        self.p_crop_tumor = float(p_crop_tumor)
        self.jitter = int(jitter)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.all_samples: List[Tuple[str, int]] = []
        self.tumor_samples: List[Tuple[str, int]] = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            all_z = list(range(Z))
            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z)-1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            tum_z = [z for z in all_z if (m[:, :, z] > 0).mean() >= self.tumor_min_frac]

            for z in all_z:
                self.all_samples.append((sid, int(z)))
            for z in tum_z:
                self.tumor_samples.append((sid, int(z)))

        if len(self.all_samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.all_samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _resize_slice(self, vol2d: np.ndarray) -> np.ndarray:
        s = self._zscore(vol2d)
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def _resize_mask(self, m2d: np.ndarray) -> np.ndarray:
        return cv2.resize(m2d, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

    def _crop_centered(self, x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        H = W = self.image_size
        cs = self.crop_size
        if cs >= H:
            return x, y

        if (y > 0).any():
            ys, xs = np.where(y > 0)
            cy = int(np.mean(ys))
            cx = int(np.mean(xs))
            cy = int(np.clip(cy + random.randint(-self.jitter, self.jitter), 0, H-1))
            cx = int(np.clip(cx + random.randint(-self.jitter, self.jitter), 0, W-1))
        else:
            cy = random.randint(0, H-1)
            cx = random.randint(0, W-1)

        y1 = int(np.clip(cy - cs//2, 0, H - cs))
        x1 = int(np.clip(cx - cs//2, 0, W - cs))
        y2 = y1 + cs
        x2 = x1 + cs

        x_crop = x[:, y1:y2, x1:x2]
        y_crop = y[y1:y2, x1:x2]

        x_out = np.stack([cv2.resize(x_crop[c], (H, W), interpolation=cv2.INTER_AREA)
                          for c in range(x_crop.shape[0])], axis=0).astype(np.float32)
        y_out = cv2.resize(y_crop, (H, W), interpolation=cv2.INTER_NEAREST).astype(np.int64)
        return x_out, y_out

    def __getitem__(self, idx):
        if self.is_train and (len(self.tumor_samples) > 0) and (random.random() < self.p_tumor):
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.all_samples[idx]

        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        x = np.stack([
            self._resize_slice(t1[:, :, z]),
            self._resize_slice(t1ce[:, :, z]),
            self._resize_slice(t2[:, :, z]),
            self._resize_slice(flair[:, :, z]),
        ], axis=0).astype(np.float32)

        y = self._resize_mask(mask_vol[:, :, z])

        if self.is_train and (random.random() < self.p_crop_tumor):
            x, y = self._crop_centered(x, y)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, gn_groups=8, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot  = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3  = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1  = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)


def load_model_ema_only(init_ckpt: str, model: nn.Module, ema: EMA):
    ckpt = torch.load(init_ckpt, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)
    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)
    ep = int(ckpt.get("epoch", -1))
    dv = float(ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))
    return ep, dv


def save_ckpt(path: str, model: nn.Module, ema: EMA, opt, scaler, epoch: int, best: float):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().clone().cpu() for k, v in ema.shadow.items()},
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best": float(best),
    }, path)


def main():
    print("=== STEP 7.3b START (CROP PHASE + LR RESET, workers=0) ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print(f"INIT_CKPT: {INIT_CKPT}", flush=True)
    print(f"P_TUMOR={P_TUMOR} P_CROP_TUMOR={P_CROP_TUMOR} CROP_SIZE={CROP_SIZE} JITTER={JITTER}", flush=True)
    print(f"EPOCHS={EPOCHS} BASE_LR={BASE_LR} WARMUP={WARMUP_EPOCHS} BATCH={BATCH}", flush=True)

    subjects = json.load(open(INDEX, "r", encoding="utf-8"))
    train_ids = json.load(open(TRAIN_SPLIT, "r", encoding="utf-8"))
    val_ids   = json.load(open(VAL_SPLIT, "r", encoding="utf-8"))

    train_ds = BraTS2024SliceDataset4ChMixedCrop(
        subjects, train_ids, image_size=IMG_SIZE,
        is_train=True, p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
        crop_size=CROP_SIZE, p_crop_tumor=P_CROP_TUMOR, jitter=JITTER,
    )
    val_ds = BraTS2024SliceDataset4ChMixedCrop(
        subjects, val_ids, image_size=IMG_SIZE,
        is_train=False, p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
        crop_size=IMG_SIZE, p_crop_tumor=0.0, jitter=0,
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)} | Tumor pool(train)={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))
    init_ep, init_d = load_model_ema_only(INIT_CKPT, model, ema)
    print(f"Loaded INIT model+EMA: epoch={init_ep} val_dice={init_d:.6f}", flush=True)

    # IMPORTANT: optimizer/scaler are RESET
    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ce = nn.CrossEntropyLoss()

    best = -1.0
    best_path = os.path.join(OUT_DIR, "best.pt")
    last_path = os.path.join(OUT_DIR, "last.pt")

    for epoch in range(EPOCHS):
        lr = lr_at_epoch(epoch, EPOCHS, BASE_LR, WARMUP_EPOCHS)
        for pg in opt.param_groups:
            pg["lr"] = lr

        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()

            ema.update(model)

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        tr_loss /= max(1, len(train_loader.dataset))

        model.eval()
        ema.apply_to(model)

        va_loss, va_dice_sum, n = 0.0, 0.0, 0
        with torch.no_grad():
            pbarv = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} VAL(EMA)", leave=True)
            for x, y in pbarv:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice_sum += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)
                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= max(1, len(val_loader.dataset))
        va_dice = va_dice_sum / max(1, n)

        ema.restore(model)

        print(f"Epoch {epoch+1:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_meanDiceFG(EMA)={va_dice:.4f} | lr={lr:.2e}", flush=True)

        save_ckpt(last_path, model, ema, opt, scaler, epoch+1, best)

        if va_dice > best:
            best = va_dice
            save_ckpt(best_path, model, ema, opt, scaler, epoch+1, best)
            print("Saved best:", best_path, flush=True)

    print("=== STEP 7.3b DONE === Best val_meanDiceFG(EMA):", best, flush=True)
    print("Out:", OUT_DIR, flush=True)


if __name__ == "__main__":
    main()

=== STEP 7.3b START (CROP PHASE + LR RESET, workers=0) ===
Device: cuda
Torch: 2.5.1+cu121
INIT_CKPT: C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt
P_TUMOR=0.5 P_CROP_TUMOR=0.85 CROP_SIZE=192 JITTER=24
EPOCHS=8 BASE_LR=0.0002 WARMUP=2 BATCH=2
Train slices=36576 | Val slices=7936 | Tumor pool(train)=12357
Loaded INIT model+EMA: epoch=10 val_dice=0.895030


C:\Users\admin\AppData\Local\Temp\ipykernel_21388\3624130149.py:351: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(init_ckpt, map_location="cpu")
Ep 1/8 VA

Epoch 01 | train_loss=0.1859 | val_loss=0.2222 | val_meanDiceFG(EMA)=0.8949 | lr=1.00e-04


Saved best: C:/GDNET/logs/step7_3b_crop_phase_lrreset_ptumor_0.50\best.pt


Ep 2/8 VAL(EMA): 100%|███████████████████████████████████████████████| 3968/3968 [03:25<00:00, 19.28it/s, vloss=0.2500]

...

Epoch 07 | train_loss=0.1646 | val_loss=0.2211 | val_meanDiceFG(EMA)=0.9005 | lr=5.00e-05


Saved best: C:/GDNET/logs/step7_3b_crop_phase_lrreset_ptumor_0.50\best.pt


Ep 8/8 VAL(EMA): 100%|███████████████████████████████████████████████| 3968/3968 [03:39<00:00, 18.06it/s, vloss=0.2500]

Epoch 08 | train_loss=0.1590 | val_loss=0.2207 | val_meanDiceFG(EMA)=0.8987 | lr=1.34e-05


=== STEP 7.3b DONE === Best val_meanDiceFG(EMA): 0.9005448628111833
Out: C:/GDNET/logs/step7_3b_crop_phase_lrreset_ptumor_0.50


In [2]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 7.2b — AUDIT EVAL + SUMMARY for NEW BEST CHECKPOINT
# - Slice-averaged Dice
# - Positive-only Dice (GT-present slices only)
# - Correct composites for labels [0,1,2,3,4]
# - Prints clean summary for VAL and TEST
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

# NEW baseline checkpoint from Step 7.3b
CKPT_BEST = r"C:/GDNET/logs/step7_3b_crop_phase_lrreset_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step7_2b_eval_audit_step7_3b_ptumor_0.50"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Runtime
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

# Your dataset contains label 3, so include it in composites
INCLUDE_LABEL3_IN_COMPOSITES = True


# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# -------------------------
# Preprocess
# -------------------------
def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# -------------------------
# Dataset for deterministic eval
# -------------------------
class BraTS2024SliceDataset4ChEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z)-1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = zscore2d(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z)


# -------------------------
# Model (same UNetSmall)
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, gn_groups=8, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(gn_groups, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x

class UNetSmall(nn.Module):
    def __init__(self, in_c=4, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot  = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3  = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1  = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], 1))
        return self.out(d1)


# -------------------------
# Load EMA weights into model
# -------------------------
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


# -------------------------
# Metrics
# -------------------------
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))

def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        WT = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        TC = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        WT = (lbl == 1) | (lbl == 2) | (lbl == 4)
        TC = (lbl == 1) | (lbl == 4)
    ET = (lbl == 4)
    return {"WT": WT, "TC": TC, "ET": ET}

def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats

def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset4ChEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNetSmall(in_c=4, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            # Per-class dice for labels 1..4
            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            # BraTS composites
            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(dice_all), float(dice_pos), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {
            "dice_all": a,
            "dice_pos_only": p,
            "n_pos_slices": npos,
        }

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    name = report["split"]
    m = report["metrics"]
    s = report["sanity"]

    print(f"\n[{name}]")
    print(f"  slices_total            : {s['slices_total']}")
    print(f"  slices_any_tumor_gt     : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred   : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt        : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred      : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")

    for c in [1, 2, 3, 4]:
        k = f"c{c}"
        print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})")


def main():
    print("=== STEP 7.2b SUMMARY START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 7.2b SUMMARY DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 7.2b SUMMARY START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step7_3b_crop_phase_lrreset_ptumor_0.50/best.pt
[VAL] EMA loaded: epoch=7 best=0.900545


C:\Users\admin\AppData\Local\Temp\ipykernel_21388\2679721905.py:213: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL 

[TEST] EMA loaded: epoch=7 best=0.900545


EVAL TEST (slice-avg): 100%|█████████████████████████████████████| 20930/20930 [09:55<00:00, 35.17it/s, mean_fg=0.9184]


=== SUMMARY (dice_all / dice_pos_only) ===



[VAL]
  slices_total            : 45136
  slices_any_tumor_gt     : 16476
  slices_any_tumor_pred   : 17840
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.919997 / 0.815381 (npos=16476)
  WT      : 0.879374 / 0.795425 (npos=16476)
  TC      : 0.910984 / 0.753141 (npos=9839)
  ET      : 0.900623 / 0.682125 (npos=6847)
  c1     : 0.961298 / 0.472187 (npos=2404)
  c2     : 0.877025 / 0.764620 (npos=16134)
  c3     : 0.941042 / 0.698630 (npos=5930)
  c4     : 0.900623 / 0.682125 (npos=6847)

[TEST]
  slices_total            : 41860
  slices_any_tumor_gt     : 14364
  slices_any_tumor_pred   : 15951
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.918449 / 0.806602 (npos=14364)
  WT      : 0.871266 / 0.783569 (npos=14364)
  TC      : 0.911899 / 0.733833 (npos=9490)
  ET      : 0.899897 / 0.653758 (npos=6661)
  c1     : 0.967125 / 0.474118 (npos=2054)
  c2     : 0.865472 / 0.742732 (n

In [1]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 8 — GDNet-style integration (FIXED GroupNorm)
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

OUT_DIR = r"C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed"
os.makedirs(OUT_DIR, exist_ok=True)

INIT_BASELINE_CKPT = r"C:/GDNET/logs/step7_1d_ptumor_sweep_ema_workers0/ptumor_0.50/best.pt"
USE_BASELINE_INIT = False

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 4
NUM_CLASSES = 5

NUM_WORKERS = 0
PIN_MEMORY = True

EPOCHS = 12
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2
BATCH = 2

P_TUMOR = 0.50
MAX_SLICES_PER_SUBJECT = 32
TUMOR_MIN_FRAC = 0.001

LRU_CACHE_MAX_SUBJECTS = 6
CACHE_IN_RAM = True

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999
RESUME_IF_EXISTS = True

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


# =========================
# Repro
# =========================
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# =========================
# Utility
# =========================
def valid_gn_groups(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


# =========================
# LR schedule
# =========================
def lr_at_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int):
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    t = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * t))


# =========================
# EMA
# =========================
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device: str = "cpu"):
        self.decay = float(decay)
        self.device = device
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.detach().to(self.device), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model: nn.Module):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


# =========================
# Losses / metrics
# =========================
def soft_dice_loss_all_classes(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    tgt = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs_fg = probs[:, 1:]
    tgt_fg = tgt[:, 1:]

    dims = (0, 2, 3)
    inter = (probs_fg * tgt_fg).sum(dims)
    p_sum = probs_fg.sum(dims)
    t_sum = tgt_fg.sum(dims)

    dice = (2.0 * inter + eps) / (p_sum + t_sum + eps)
    empty = (t_sum < eps) & (p_sum < eps)
    dice = torch.where(empty, torch.ones_like(dice), dice)
    return 1.0 - dice.mean()

@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)
        dices.append(d)
    return torch.stack(dices).mean().item()


# =========================
# Dataset
# =========================
class BraTS2024SliceDataset4ChMixed(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        is_train: bool = True,
        p_tumor: float = 0.5,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = 32,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.is_train = bool(is_train)
        self.p_tumor = float(p_tumor)
        self.tumor_min_frac = float(tumor_min_frac)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.all_samples = []
        self.tumor_samples = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            all_z = list(range(Z))
            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            tum_z = [z for z in all_z if (m[:, :, z] > 0).mean() >= self.tumor_min_frac]

            for z in all_z:
                self.all_samples.append((sid, z))
            for z in tum_z:
                self.tumor_samples.append((sid, z))

        if len(self.all_samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.all_samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        if self.is_train and (len(self.tumor_samples) > 0) and (random.random() < self.p_tumor):
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.all_samples[idx]

        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = self._zscore(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


# =========================
# GDNet-style model
# =========================
class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, growth, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(growth), num_channels=growth),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        out = self.block(x)
        return torch.cat([x, out], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, in_ch, growth=16, num_layers=3, out_ch=64):
        super().__init__()
        ch = in_ch
        layers = []
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth))
            ch += growth
        self.layers = nn.Sequential(*layers)
        self.compress = nn.Sequential(
            nn.Conv2d(ch, out_ch, 1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.layers(x)
        x = self.compress(x)
        return x


class GatedSkipFusion(nn.Module):
    def __init__(self, skip_ch, gate_ch, out_ch):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_ch, out_ch, 1, bias=False)
        self.gate_proj = nn.Conv2d(gate_ch, out_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        self.out = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, skip, gate):
        s = self.skip_proj(skip)
        g = self.gate_proj(gate)
        a = self.psi(s + g)
        fused = s * a
        return self.out(fused)


class GDNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, growth=16, layers=3):
        super().__init__()
        self.dense = DenseBlock(in_ch, growth=growth, num_layers=layers, out_ch=out_ch)
        self.refine = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, x):
        x = self.dense(x)
        x = self.refine(x)
        return x


class GDNet2D(nn.Module):
    def __init__(self, in_ch=4, num_classes=5, base=32):
        super().__init__()

        self.stem = ConvGNAct(in_ch, base)

        self.enc1 = GDNetBlock(base, base, growth=12, layers=3)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = GDNetBlock(base, base * 2, growth=16, layers=3)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = GDNetBlock(base * 2, base * 4, growth=20, layers=3)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = GDNetBlock(base * 4, base * 8, growth=24, layers=4)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.gate3 = GatedSkipFusion(skip_ch=base * 4, gate_ch=base * 4, out_ch=base * 4)
        self.dec3 = GDNetBlock(base * 8, base * 4, growth=20, layers=3)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.gate2 = GatedSkipFusion(skip_ch=base * 2, gate_ch=base * 2, out_ch=base * 2)
        self.dec2 = GDNetBlock(base * 4, base * 2, growth=16, layers=3)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.gate1 = GatedSkipFusion(skip_ch=base, gate_ch=base, out_ch=base)
        self.dec1 = GDNetBlock(base * 2, base, growth=12, layers=3)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        x0 = self.stem(x)

        e1 = self.enc1(x0)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        s3 = self.gate3(e3, d3)
        d3 = self.dec3(torch.cat([d3, s3], dim=1))

        d2 = self.up2(d3)
        s2 = self.gate2(e2, d2)
        d2 = self.dec2(torch.cat([d2, s2], dim=1))

        d1 = self.up1(d2)
        s1 = self.gate1(e1, d1)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        return self.out(d1)


# =========================
# Checkpoint IO
# =========================
def save_ckpt(path: str, model: nn.Module, ema: EMA, opt, scaler, epoch: int, best: float):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().clone().cpu() for k, v in ema.shadow.items()},
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best": float(best),
    }, path)

def load_ckpt(path: str, model: nn.Module, ema: EMA, opt=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)

    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)

    if opt is not None and "opt" in ckpt:
        opt.load_state_dict(ckpt["opt"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0))
    best = float(ckpt.get("best", -1.0))
    return start_epoch, best


def try_partial_init_from_baseline(model: nn.Module, baseline_ckpt: str):
    if not os.path.exists(baseline_ckpt):
        print("[WARN] Baseline checkpoint not found.", flush=True)
        return

    ckpt = torch.load(baseline_ckpt, map_location="cpu")
    src = ckpt.get("model", None)
    if src is None:
        print("[WARN] Baseline checkpoint has no model key.", flush=True)
        return

    dst = model.state_dict()
    copied = 0
    for k in dst.keys():
        if k in src and src[k].shape == dst[k].shape:
            dst[k] = src[k]
            copied += 1
    model.load_state_dict(dst, strict=False)
    print(f"[INIT] Copied {copied} tensors from baseline.", flush=True)


# =========================
# Main
# =========================
def main():
    print("=== STEP 8 START (GDNet integration, fixed) ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print(f"BATCH={BATCH} IMG_SIZE={IMG_SIZE} WORKERS={NUM_WORKERS}", flush=True)
    print(f"P_TUMOR={P_TUMOR} EPOCHS={EPOCHS} LR={BASE_LR}", flush=True)

    subjects = json.load(open(INDEX, "r", encoding="utf-8"))
    train_ids = json.load(open(TRAIN_SPLIT, "r", encoding="utf-8"))
    val_ids   = json.load(open(VAL_SPLIT, "r", encoding="utf-8"))

    train_ds = BraTS2024SliceDataset4ChMixed(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )
    val_ds = BraTS2024SliceDataset4ChMixed(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)} | Tumor pool(train)={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = GDNet2D(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    if USE_BASELINE_INIT:
        try_partial_init_from_baseline(model, INIT_BASELINE_CKPT)

    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0
    start_epoch = 0

    best_path = os.path.join(OUT_DIR, "best.pt")
    last_path = os.path.join(OUT_DIR, "last.pt")

    if RESUME_IF_EXISTS and os.path.exists(best_path):
        start_epoch, best = load_ckpt(best_path, model, ema, opt=opt, scaler=scaler)
        print(f"[RESUME] Loaded best.pt: start_epoch={start_epoch} best={best:.6f}", flush=True)

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_at_epoch(epoch, EPOCHS, BASE_LR, WARMUP_EPOCHS)
        for pg in opt.param_groups:
            pg["lr"] = lr

        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()

            ema.update(model)

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        tr_loss /= max(1, len(train_loader.dataset))

        model.eval()
        ema.apply_to(model)

        va_loss, va_dice_sum, n = 0.0, 0.0, 0
        with torch.no_grad():
            pbarv = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} VAL(EMA)", leave=True)
            for x, y in pbarv:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice_sum += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)
                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= max(1, len(val_loader.dataset))
        va_dice = va_dice_sum / max(1, n)

        ema.restore(model)

        print(
            f"Epoch {epoch+1:02d} | train_loss={tr_loss:.4f} | "
            f"val_loss={va_loss:.4f} | val_meanDiceFG(EMA)={va_dice:.4f} | lr={lr:.2e}",
            flush=True
        )

        save_ckpt(last_path, model, ema, opt, scaler, epoch + 1, best)

        if va_dice > best:
            best = va_dice
            save_ckpt(best_path, model, ema, opt, scaler, epoch + 1, best)
            print("Saved best:", best_path, flush=True)

    print("=== STEP 8 DONE === Best val_meanDiceFG(EMA):", best, flush=True)
    print("Out:", OUT_DIR, flush=True)


if __name__ == "__main__":
    main()

=== STEP 8 START (GDNet integration, fixed) ===
Device: cuda
Torch: 2.5.1+cu121
BATCH=2 IMG_SIZE=256 WORKERS=0
P_TUMOR=0.5 EPOCHS=12 LR=0.0002
Train slices=36576 | Val slices=7936 | Tumor pool(train)=12357


C:\Users\admin\AppData\Local\Temp\ipykernel_13884\3143653283.py:419: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location="cpu")


[RESUME] Loaded best.pt: start_epoch=7 best=0.902861


Ep 8/12 VAL(EMA): 100%|██████████████████████████████████████████████| 3968/3968 [04:28<00:00, 14.80it/s, vloss=0.0000]

Epoch 08 | train_loss=0.1055 | val_loss=0.0584 | val_meanDiceFG(EMA)=0.9094 | lr=1.00e-04


...

Epoch 11 | train_loss=0.0843 | val_loss=0.0556 | val_meanDiceFG(EMA)=0.9132 | lr=1.91e-05


Saved best: C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed\best.pt


Ep 12/12 VAL(EMA): 100%|█████████████████████████████████████████████| 3968/3968 [04:25<00:00, 14.93it/s, vloss=0.0000]

Epoch 12 | train_loss=0.0805 | val_loss=0.0563 | val_meanDiceFG(EMA)=0.9135 | lr=4.89e-06


Saved best: C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed\best.pt
=== STEP 8 DONE === Best val_meanDiceFG(EMA): 0.9135027625372694
Out: C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed


In [2]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 7.2b AUDIT EVAL FOR STEP 8 GDNET CHECKPOINT
# - Slice-averaged Dice
# - Positive-only Dice
# - Correct composites for labels [0,1,2,3,4]
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed/best.pt"

OUT_DIR = r"C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed_audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Runtime
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

# Your dataset contains label 3
INCLUDE_LABEL3_IN_COMPOSITES = True


# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# -------------------------
# Preprocess
# -------------------------
def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# -------------------------
# Dataset for deterministic eval
# -------------------------
class BraTS2024SliceDataset4ChEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = zscore2d(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z)


# -------------------------
# Model — MUST MATCH STEP 8 GDNET
# -------------------------
def valid_gn_groups(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, growth, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(growth), num_channels=growth),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        out = self.block(x)
        return torch.cat([x, out], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, in_ch, growth=16, num_layers=3, out_ch=64):
        super().__init__()
        ch = in_ch
        layers = []
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth))
            ch += growth
        self.layers = nn.Sequential(*layers)
        self.compress = nn.Sequential(
            nn.Conv2d(ch, out_ch, 1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.layers(x)
        x = self.compress(x)
        return x


class GatedSkipFusion(nn.Module):
    def __init__(self, skip_ch, gate_ch, out_ch):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_ch, out_ch, 1, bias=False)
        self.gate_proj = nn.Conv2d(gate_ch, out_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        self.out = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, skip, gate):
        s = self.skip_proj(skip)
        g = self.gate_proj(gate)
        a = self.psi(s + g)
        fused = s * a
        return self.out(fused)


class GDNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, growth=16, layers=3):
        super().__init__()
        self.dense = DenseBlock(in_ch, growth=growth, num_layers=layers, out_ch=out_ch)
        self.refine = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, x):
        x = self.dense(x)
        x = self.refine(x)
        return x


class GDNet2D(nn.Module):
    def __init__(self, in_ch=4, num_classes=5, base=32):
        super().__init__()

        self.stem = ConvGNAct(in_ch, base)

        self.enc1 = GDNetBlock(base, base, growth=12, layers=3)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = GDNetBlock(base, base * 2, growth=16, layers=3)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = GDNetBlock(base * 2, base * 4, growth=20, layers=3)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = GDNetBlock(base * 4, base * 8, growth=24, layers=4)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.gate3 = GatedSkipFusion(skip_ch=base * 4, gate_ch=base * 4, out_ch=base * 4)
        self.dec3 = GDNetBlock(base * 8, base * 4, growth=20, layers=3)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.gate2 = GatedSkipFusion(skip_ch=base * 2, gate_ch=base * 2, out_ch=base * 2)
        self.dec2 = GDNetBlock(base * 4, base * 2, growth=16, layers=3)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.gate1 = GatedSkipFusion(skip_ch=base, gate_ch=base, out_ch=base)
        self.dec1 = GDNetBlock(base * 2, base, growth=12, layers=3)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        x0 = self.stem(x)

        e1 = self.enc1(x0)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        s3 = self.gate3(e3, d3)
        d3 = self.dec3(torch.cat([d3, s3], dim=1))

        d2 = self.up2(d3)
        s2 = self.gate2(e2, d2)
        d2 = self.dec2(torch.cat([d2, s2], dim=1))

        d1 = self.up1(d2)
        s1 = self.gate1(e1, d1)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        return self.out(d1)


# -------------------------
# Load EMA weights into model
# -------------------------
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


# -------------------------
# Metrics
# -------------------------
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))

def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        WT = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        TC = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        WT = (lbl == 1) | (lbl == 2) | (lbl == 4)
        TC = (lbl == 1) | (lbl == 4)
    ET = (lbl == 4)
    return {"WT": WT, "TC": TC, "ET": ET}

def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats

def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset4ChEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = GDNet2D(in_ch=4, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(dice_all), float(dice_pos), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {
            "dice_all": a,
            "dice_pos_only": p,
            "n_pos_slices": npos,
        }

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    name = report["split"]
    m = report["metrics"]
    s = report["sanity"]

    print(f"\n[{name}]")
    print(f"  slices_total            : {s['slices_total']}")
    print(f"  slices_any_tumor_gt     : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred   : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt        : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred      : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")

    for c in [1, 2, 3, 4]:
        k = f"c{c}"
        print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})")


def main():
    print("=== STEP 8 AUDIT EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 8 AUDIT EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 8 AUDIT EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed/best.pt
[VAL] EMA loaded: epoch=12 best=0.913503


C:\Users\admin\AppData\Local\Temp\ipykernel_13884\1352659959.py:302: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL 

[TEST] EMA loaded: epoch=12 best=0.913503


EVAL TEST (slice-avg): 100%|█████████████████████████████████████| 20930/20930 [14:36<00:00, 23.87it/s, mean_fg=0.9287]


=== SUMMARY (dice_all / dice_pos_only) ===



[VAL]
  slices_total            : 45136
  slices_any_tumor_gt     : 16476
  slices_any_tumor_pred   : 16633
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.930542 / 0.829642 (npos=16476)
  WT      : 0.899794 / 0.794313 (npos=16476)
  TC      : 0.925813 / 0.744131 (npos=9839)
  ET      : 0.921971 / 0.665268 (npos=6847)
  c1     : 0.964042 / 0.505408 (npos=2404)
  c2     : 0.891445 / 0.767960 (npos=16134)
  c3     : 0.944708 / 0.705457 (npos=5930)
  c4     : 0.921971 / 0.665268 (npos=6847)

[TEST]
  slices_total            : 41860
  slices_any_tumor_gt     : 14364
  slices_any_tumor_pred   : 14646
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.928651 / 0.816351 (npos=14364)
  WT      : 0.896057 / 0.782649 (npos=14364)
  TC      : 0.923509 / 0.723506 (npos=9490)
  ET      : 0.917875 / 0.635527 (npos=6661)
  c1     : 0.968128 / 0.517936 (npos=2054)
  c2     : 0.885720 / 0.746820 (n

In [3]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 8.1 — GDNet + region-aware loss
# Goal:
#   improve positive-only WT / TC / ET
# Strategy:
#   keep Step 8 architecture, but optimize the actual BraTS regions
#   in addition to classwise CE + Dice
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

OUT_DIR = r"C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50"
os.makedirs(OUT_DIR, exist_ok=True)

# Start from your Step 8 checkpoint (recommended)
INIT_CKPT = r"C:/GDNET/logs/step8_gdnet_ptumor_0.50_fixed/best.pt"
USE_INIT_CKPT = True


# -------------------------
# Runtime config
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 4
NUM_CLASSES = 5

NUM_WORKERS = 0
PIN_MEMORY = True

# Training
EPOCHS = 10
BASE_LR = 1.5e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2
BATCH = 2

# Same winning sampling policy
P_TUMOR = 0.50
MAX_SLICES_PER_SUBJECT = 32
TUMOR_MIN_FRAC = 0.001

# Cache
LRU_CACHE_MAX_SUBJECTS = 6
CACHE_IN_RAM = True

# EMA
EMA_DECAY = 0.999
RESUME_IF_EXISTS = True

# Loss weights
# Old losses
W_CE = 0.35
W_DICE_CLASS = 0.25

# New region-aware losses
W_DICE_WT = 0.20
W_DICE_TC = 0.10
W_DICE_ET = 0.10

# label-3 is present in your dataset
INCLUDE_LABEL3 = True

# Optional stabilizer
MAX_GRAD_NORM = 1.0

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


# ============================================================
# Repro
# ============================================================
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# ============================================================
# Schedules / helpers
# ============================================================
def lr_at_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int):
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    t = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * t))


def valid_gn_groups(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


# ============================================================
# EMA
# ============================================================
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device: str = "cpu"):
        self.decay = float(decay)
        self.device = device
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.detach().to(self.device), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model: nn.Module):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


# ============================================================
# Dataset
# ============================================================
class BraTS2024SliceDataset4ChMixed(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        is_train: bool = True,
        p_tumor: float = 0.5,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = 32,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.is_train = bool(is_train)
        self.p_tumor = float(p_tumor)
        self.tumor_min_frac = float(tumor_min_frac)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.all_samples = []
        self.tumor_samples = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            all_z = list(range(Z))
            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            tum_z = [z for z in all_z if (m[:, :, z] > 0).mean() >= self.tumor_min_frac]

            for z in all_z:
                self.all_samples.append((sid, z))
            for z in tum_z:
                self.tumor_samples.append((sid, z))

        if len(self.all_samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.all_samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1    = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce  = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2    = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask  = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        if self.is_train and (len(self.tumor_samples) > 0) and (random.random() < self.p_tumor):
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.all_samples[idx]

        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = self._zscore(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


# ============================================================
# GDNet model (same as Step 8 fixed)
# ============================================================
class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, growth, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(growth), num_channels=growth),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        out = self.block(x)
        return torch.cat([x, out], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, in_ch, growth=16, num_layers=3, out_ch=64):
        super().__init__()
        ch = in_ch
        layers = []
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth))
            ch += growth
        self.layers = nn.Sequential(*layers)
        self.compress = nn.Sequential(
            nn.Conv2d(ch, out_ch, 1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.layers(x)
        x = self.compress(x)
        return x


class GatedSkipFusion(nn.Module):
    def __init__(self, skip_ch, gate_ch, out_ch):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_ch, out_ch, 1, bias=False)
        self.gate_proj = nn.Conv2d(gate_ch, out_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        self.out = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, skip, gate):
        s = self.skip_proj(skip)
        g = self.gate_proj(gate)
        a = self.psi(s + g)
        fused = s * a
        return self.out(fused)


class GDNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, growth=16, layers=3):
        super().__init__()
        self.dense = DenseBlock(in_ch, growth=growth, num_layers=layers, out_ch=out_ch)
        self.refine = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, x):
        x = self.dense(x)
        x = self.refine(x)
        return x


class GDNet2D(nn.Module):
    def __init__(self, in_ch=4, num_classes=5, base=32):
        super().__init__()

        self.stem = ConvGNAct(in_ch, base)

        self.enc1 = GDNetBlock(base, base, growth=12, layers=3)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = GDNetBlock(base, base * 2, growth=16, layers=3)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = GDNetBlock(base * 2, base * 4, growth=20, layers=3)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = GDNetBlock(base * 4, base * 8, growth=24, layers=4)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.gate3 = GatedSkipFusion(skip_ch=base * 4, gate_ch=base * 4, out_ch=base * 4)
        self.dec3 = GDNetBlock(base * 8, base * 4, growth=20, layers=3)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.gate2 = GatedSkipFusion(skip_ch=base * 2, gate_ch=base * 2, out_ch=base * 2)
        self.dec2 = GDNetBlock(base * 4, base * 2, growth=16, layers=3)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.gate1 = GatedSkipFusion(skip_ch=base, gate_ch=base, out_ch=base)
        self.dec1 = GDNetBlock(base * 2, base, growth=12, layers=3)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        x0 = self.stem(x)

        e1 = self.enc1(x0)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        s3 = self.gate3(e3, d3)
        d3 = self.dec3(torch.cat([d3, s3], dim=1))

        d2 = self.up2(d3)
        s2 = self.gate2(e2, d2)
        d2 = self.dec2(torch.cat([d2, s2], dim=1))

        d1 = self.up1(d2)
        s1 = self.gate1(e1, d1)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        return self.out(d1)


# ============================================================
# Region-aware loss
# ============================================================
def logits_to_region_probs(logits: torch.Tensor) -> Dict[str, torch.Tensor]:
    """
    logits: [B,C,H,W]
    returns probs for WT / TC / ET as [B,1,H,W]
    labels are [0,1,2,3,4]
    WT = {1,2,3,4}
    TC = {1,3,4}
    ET = {4}
    """
    probs = torch.softmax(logits, dim=1)

    if INCLUDE_LABEL3:
        wt = probs[:, 1:2] + probs[:, 2:3] + probs[:, 3:4] + probs[:, 4:5]
        tc = probs[:, 1:2] + probs[:, 3:4] + probs[:, 4:5]
    else:
        wt = probs[:, 1:2] + probs[:, 2:3] + probs[:, 4:5]
        tc = probs[:, 1:2] + probs[:, 4:5]

    et = probs[:, 4:5]
    return {"WT": wt, "TC": tc, "ET": et}


def target_to_region_masks(target: torch.Tensor) -> Dict[str, torch.Tensor]:
    """
    target: [B,H,W]
    returns region masks [B,1,H,W] float
    """
    if INCLUDE_LABEL3:
        wt = ((target == 1) | (target == 2) | (target == 3) | (target == 4)).float().unsqueeze(1)
        tc = ((target == 1) | (target == 3) | (target == 4)).float().unsqueeze(1)
    else:
        wt = ((target == 1) | (target == 2) | (target == 4)).float().unsqueeze(1)
        tc = ((target == 1) | (target == 4)).float().unsqueeze(1)

    et = (target == 4).float().unsqueeze(1)
    return {"WT": wt, "TC": tc, "ET": et}


def soft_dice_region(prob: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    prob/mask: [B,1,H,W]
    returns scalar loss = 1 - dice
    """
    dims = (0, 2, 3)
    inter = (prob * mask).sum(dims)
    p_sum = prob.sum(dims)
    m_sum = mask.sum(dims)

    dice = (2.0 * inter + eps) / (p_sum + m_sum + eps)

    # if region absent in GT and prob also near-zero -> no penalty
    empty = (m_sum < eps) & (p_sum < eps)
    dice = torch.where(empty, torch.ones_like(dice), dice)
    return 1.0 - dice.mean()


def class_and_region_loss(logits: torch.Tensor, target: torch.Tensor, ce_fn) -> Tuple[torch.Tensor, Dict[str, float]]:
    loss_ce = ce_fn(logits, target)
    loss_d_cls = soft_dice_loss_all_classes(logits, target, num_classes=NUM_CLASSES)

    rprob = logits_to_region_probs(logits)
    rmask = target_to_region_masks(target)

    loss_wt = soft_dice_region(rprob["WT"], rmask["WT"])
    loss_tc = soft_dice_region(rprob["TC"], rmask["TC"])
    loss_et = soft_dice_region(rprob["ET"], rmask["ET"])

    loss = (
        W_CE * loss_ce +
        W_DICE_CLASS * loss_d_cls +
        W_DICE_WT * loss_wt +
        W_DICE_TC * loss_tc +
        W_DICE_ET * loss_et
    )

    aux = {
        "ce": float(loss_ce.detach().item()),
        "dice_cls": float(loss_d_cls.detach().item()),
        "wt": float(loss_wt.detach().item()),
        "tc": float(loss_tc.detach().item()),
        "et": float(loss_et.detach().item()),
    }
    return loss, aux


# ============================================================
# Checkpoint IO
# ============================================================
def save_ckpt(path: str, model: nn.Module, ema: EMA, opt, scaler, epoch: int, best: float):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().clone().cpu() for k, v in ema.shadow.items()},
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best": float(best),
    }, path)


def load_ckpt(path: str, model: nn.Module, ema: EMA, opt=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)

    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)

    if opt is not None and "opt" in ckpt:
        opt.load_state_dict(ckpt["opt"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0))
    best = float(ckpt.get("best", -1.0))
    return start_epoch, best


def load_model_ema_only(init_ckpt: str, model: nn.Module, ema: EMA):
    ckpt = torch.load(init_ckpt, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)
    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)
    ep = int(ckpt.get("epoch", -1))
    best = float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0))))
    return ep, best


# ============================================================
# Main
# ============================================================
def main():
    print("=== STEP 8.1 START (GDNet + region-aware loss) ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print(f"BATCH={BATCH} IMG_SIZE={IMG_SIZE} WORKERS={NUM_WORKERS}", flush=True)
    print(f"P_TUMOR={P_TUMOR} EPOCHS={EPOCHS} LR={BASE_LR}", flush=True)
    print(
        f"Loss weights: CE={W_CE} Dcls={W_DICE_CLASS} WT={W_DICE_WT} TC={W_DICE_TC} ET={W_DICE_ET}",
        flush=True
    )

    subjects = json.load(open(INDEX, "r", encoding="utf-8"))
    train_ids = json.load(open(TRAIN_SPLIT, "r", encoding="utf-8"))
    val_ids   = json.load(open(VAL_SPLIT, "r", encoding="utf-8"))

    train_ds = BraTS2024SliceDataset4ChMixed(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )
    val_ds = BraTS2024SliceDataset4ChMixed(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)} | Tumor pool(train)={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH, shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = GDNet2D(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0
    start_epoch = 0

    best_path = os.path.join(OUT_DIR, "best.pt")
    last_path = os.path.join(OUT_DIR, "last.pt")

    # Resume first, else init from Step 8
    if RESUME_IF_EXISTS and os.path.exists(best_path):
        start_epoch, best = load_ckpt(best_path, model, ema, opt=opt, scaler=scaler)
        print(f"[RESUME] Loaded existing best.pt: start_epoch={start_epoch} best={best:.6f}", flush=True)
    elif USE_INIT_CKPT and os.path.exists(INIT_CKPT):
        ep0, best0 = load_model_ema_only(INIT_CKPT, model, ema)
        print(f"[INIT] Loaded model+EMA from INIT_CKPT: epoch={ep0} best={best0:.6f}", flush=True)

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_at_epoch(epoch, EPOCHS, BASE_LR, WARMUP_EPOCHS)
        for pg in opt.param_groups:
            pg["lr"] = lr

        # -----------------
        # Train
        # -----------------
        model.train()
        tr_loss = 0.0
        tr_aux = {"ce": 0.0, "dice_cls": 0.0, "wt": 0.0, "tc": 0.0, "et": 0.0}

        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss, aux = class_and_region_loss(logits, y, ce)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()

            ema.update(model)

            bs = x.size(0)
            tr_loss += loss.item() * bs
            for k in tr_aux:
                tr_aux[k] += aux[k] * bs

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                wt=f"{aux['wt']:.3f}",
                et=f"{aux['et']:.3f}",
                lr=f"{lr:.2e}"
            )

        tr_loss /= max(1, len(train_loader.dataset))
        for k in tr_aux:
            tr_aux[k] /= max(1, len(train_loader.dataset))

        # -----------------
        # Val on EMA
        # -----------------
        model.eval()
        ema.apply_to(model)

        va_loss, va_dice_sum, n = 0.0, 0.0, 0
        va_aux = {"ce": 0.0, "dice_cls": 0.0, "wt": 0.0, "tc": 0.0, "et": 0.0}

        with torch.no_grad():
            pbarv = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} VAL(EMA)", leave=True)
            for x, y in pbarv:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss, aux = class_and_region_loss(logits, y, ce)

                bs = x.size(0)
                va_loss += loss.item() * bs
                for k in va_aux:
                    va_aux[k] += aux[k] * bs

                pred = torch.argmax(logits, dim=1)
                va_dice_sum += mean_dice_fg(pred, y, NUM_CLASSES) * bs
                n += bs

                pbarv.set_postfix(vloss=f"{loss.item():.4f}", wt=f"{aux['wt']:.3f}", et=f"{aux['et']:.3f}")

        va_loss /= max(1, len(val_loader.dataset))
        for k in va_aux:
            va_aux[k] /= max(1, len(val_loader.dataset))
        va_dice = va_dice_sum / max(1, n)

        ema.restore(model)

        print(
            f"Epoch {epoch+1:02d} | "
            f"train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | "
            f"val_meanDiceFG(EMA)={va_dice:.4f} | lr={lr:.2e}",
            flush=True
        )
        print(
            f"           train_aux: ce={tr_aux['ce']:.4f} dcls={tr_aux['dice_cls']:.4f} "
            f"wt={tr_aux['wt']:.4f} tc={tr_aux['tc']:.4f} et={tr_aux['et']:.4f}",
            flush=True
        )
        print(
            f"             val_aux: ce={va_aux['ce']:.4f} dcls={va_aux['dice_cls']:.4f} "
            f"wt={va_aux['wt']:.4f} tc={va_aux['tc']:.4f} et={va_aux['et']:.4f}",
            flush=True
        )

        save_ckpt(last_path, model, ema, opt, scaler, epoch + 1, best)

        if va_dice > best:
            best = va_dice
            save_ckpt(best_path, model, ema, opt, scaler, epoch + 1, best)
            print("Saved best:", best_path, flush=True)

    print("=== STEP 8.1 DONE === Best val_meanDiceFG(EMA):", best, flush=True)
    print("Out:", OUT_DIR, flush=True)


if __name__ == "__main__":
    main()

=== STEP 8.1 START (GDNet + region-aware loss) ===
Device: cuda
Torch: 2.5.1+cu121
BATCH=2 IMG_SIZE=256 WORKERS=0
P_TUMOR=0.5 EPOCHS=10 LR=0.00015
Loss weights: CE=0.35 Dcls=0.25 WT=0.2 TC=0.1 ET=0.1
Train slices=36576 | Val slices=7936 | Tumor pool(train)=12357
[INIT] Loaded model+EMA from INIT_CKPT: epoch=12 best=0.913503


C:\Users\admin\AppData\Local\Temp\ipykernel_13884\1532289362.py:522: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(init_ckpt, map_location="cpu")
Ep 1/10 V

Epoch 01 | train_loss=0.1011 | val_loss=0.0792 | val_meanDiceFG(EMA)=0.9125 | lr=7.50e-05


           train_aux: ce=0.0193 dcls=0.1569 wt=0.1175 tc=0.1381 et=0.1779
             val_aux: ce=0.0145 dcls=0.0984 wt=0.1465 tc=0.0967 et=0.1060
Saved best: C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50\best.pt


Ep 2/10 VAL(EMA): 100%|██████████████████████████| 3968/3968 [04:30<00:00, 14.65it/s, et=0.000, vloss=0.0000, wt=0.000]

...

Epoch 09 | train_loss=0.0772 | val_loss=0.0725 | val_meanDiceFG(EMA)=0.9175 | lr=2.20e-05
           train_aux: ce=0.0152 dcls=0.1216 wt=0.0911 tc=0.1093 et=0.1225
             val_aux: ce=0.0148 dcls=0.0904 wt=0.1280 tc=0.0906 et=0.1004


Saved best: C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50\best.pt


Ep 10/10 VAL(EMA): 100%|█████████████████| 3968/3968 [04:25<00:00, 14.94it/s, et=0.000, vloss=0.0000, wt=0.000]

Epoch 10 | train_loss=0.0734 | val_loss=0.0716 | val_meanDiceFG(EMA)=0.9177 | lr=5.71e-06
           train_aux: ce=0.0148 dcls=0.1164 wt=0.0869 tc=0.1038 et=0.1136
             val_aux: ce=0.0150 dcls=0.0899 wt=0.1240 tc=0.0907 et=0.1005


Saved best: C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50\best.pt
=== STEP 8.1 DONE === Best val_meanDiceFG(EMA): 0.9177306182832727
Out: C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50


In [4]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 8.1 AUDIT EVAL
# - Evaluates GDNet + region-aware loss checkpoint
# - Slice-averaged Dice
# - Positive-only Dice
# - Correct composites for labels [0,1,2,3,4]
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50_audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Runtime
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

# Your dataset contains label 3
INCLUDE_LABEL3_IN_COMPOSITES = True


# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# -------------------------
# Helpers
# -------------------------
def valid_gn_groups(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# -------------------------
# Dataset for deterministic eval
# -------------------------
class BraTS2024SliceDataset4ChEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)

        def prep(vol):
            s = zscore2d(vol[:, :, z])
            return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

        x = np.stack([prep(t1), prep(t1ce), prep(t2), prep(flair)], axis=0).astype(np.float32)
        m = mask_vol[:, :, z]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z)


# -------------------------
# GDNet model — MUST MATCH STEP 8.1
# -------------------------
class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, growth, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(growth), num_channels=growth),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        out = self.block(x)
        return torch.cat([x, out], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, in_ch, growth=16, num_layers=3, out_ch=64):
        super().__init__()
        ch = in_ch
        layers = []
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth))
            ch += growth
        self.layers = nn.Sequential(*layers)
        self.compress = nn.Sequential(
            nn.Conv2d(ch, out_ch, 1, bias=False),
            nn.GroupNorm(num_groups=valid_gn_groups(out_ch), num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.layers(x)
        x = self.compress(x)
        return x


class GatedSkipFusion(nn.Module):
    def __init__(self, skip_ch, gate_ch, out_ch):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_ch, out_ch, 1, bias=False)
        self.gate_proj = nn.Conv2d(gate_ch, out_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        self.out = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, skip, gate):
        s = self.skip_proj(skip)
        g = self.gate_proj(gate)
        a = self.psi(s + g)
        fused = s * a
        return self.out(fused)


class GDNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, growth=16, layers=3):
        super().__init__()
        self.dense = DenseBlock(in_ch, growth=growth, num_layers=layers, out_ch=out_ch)
        self.refine = ConvGNAct(out_ch, out_ch, k=3, p=1)

    def forward(self, x):
        x = self.dense(x)
        x = self.refine(x)
        return x


class GDNet2D(nn.Module):
    def __init__(self, in_ch=4, num_classes=5, base=32):
        super().__init__()

        self.stem = ConvGNAct(in_ch, base)

        self.enc1 = GDNetBlock(base, base, growth=12, layers=3)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = GDNetBlock(base, base * 2, growth=16, layers=3)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = GDNetBlock(base * 2, base * 4, growth=20, layers=3)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = GDNetBlock(base * 4, base * 8, growth=24, layers=4)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.gate3 = GatedSkipFusion(skip_ch=base * 4, gate_ch=base * 4, out_ch=base * 4)
        self.dec3 = GDNetBlock(base * 8, base * 4, growth=20, layers=3)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.gate2 = GatedSkipFusion(skip_ch=base * 2, gate_ch=base * 2, out_ch=base * 2)
        self.dec2 = GDNetBlock(base * 4, base * 2, growth=16, layers=3)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.gate1 = GatedSkipFusion(skip_ch=base, gate_ch=base, out_ch=base)
        self.dec1 = GDNetBlock(base * 2, base, growth=12, layers=3)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        x0 = self.stem(x)

        e1 = self.enc1(x0)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        s3 = self.gate3(e3, d3)
        d3 = self.dec3(torch.cat([d3, s3], dim=1))

        d2 = self.up2(d3)
        s2 = self.gate2(e2, d2)
        d2 = self.dec2(torch.cat([d2, s2], dim=1))

        d1 = self.up1(d2)
        s1 = self.gate1(e1, d1)
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        return self.out(d1)


# -------------------------
# Load EMA weights
# -------------------------
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


# -------------------------
# Metrics
# -------------------------
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset4ChEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = GDNet2D(in_ch=4, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(dice_all), float(dice_pos), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {
            "dice_all": a,
            "dice_pos_only": p,
            "n_pos_slices": npos,
        }

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    name = report["split"]
    m = report["metrics"]
    s = report["sanity"]

    print(f"\n[{name}]")
    print(f"  slices_total            : {s['slices_total']}")
    print(f"  slices_any_tumor_gt     : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred   : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt        : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred      : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")

    for c in [1, 2, 3, 4]:
        k = f"c{c}"
        print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})")


def main():
    print("=== STEP 8.1 AUDIT EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 8.1 AUDIT EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 8.1 AUDIT EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step8_1_gdnet_regionloss_ptumor_0.50/best.pt
[VAL] EMA loaded: epoch=10 best=0.917731


C:\Users\admin\AppData\Local\Temp\ipykernel_13884\803950427.py:302: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL (

[TEST] EMA loaded: epoch=10 best=0.917731


EVAL TEST (slice-avg): 100%|█████████████████████████████████████| 20930/20930 [14:45<00:00, 23.64it/s, mean_fg=0.9305]


=== SUMMARY (dice_all / dice_pos_only) ===



[VAL]
  slices_total            : 45136
  slices_any_tumor_gt     : 16476
  slices_any_tumor_pred   : 16080
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.932706 / 0.829046 (npos=16476)
  WT      : 0.904558 / 0.786425 (npos=16476)
  TC      : 0.928157 / 0.736487 (npos=9839)
  ET      : 0.924776 / 0.655861 (npos=6847)
  c1     : 0.964635 / 0.482433 (npos=2404)
  c2     : 0.896012 / 0.759540 (npos=16134)
  c3     : 0.945400 / 0.698917 (npos=5930)
  c4     : 0.924776 / 0.655861 (npos=6847)

[TEST]
  slices_total            : 41860
  slices_any_tumor_gt     : 14364
  slices_any_tumor_pred   : 14145
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.930460 / 0.815010 (npos=14364)
  WT      : 0.901195 / 0.775203 (npos=14364)
  TC      : 0.924659 / 0.713511 (npos=9490)
  ET      : 0.920297 / 0.624326 (npos=6661)
  c1     : 0.968467 / 0.504882 (npos=2054)
  c2     : 0.890132 / 0.738919 (n

In [3]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 AUDIT EVAL — 2.5D UNet
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50_audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _clamp_z(self, z: int, Z: int) -> int:
        return max(0, min(Z - 1, z))

    def _prep_slice(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)
        Z = mask_vol.shape[2]

        z0 = self._clamp_z(z - 1, Z)
        z1 = self._clamp_z(z, Z)
        z2 = self._clamp_z(z + 1, Z)

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            chans.append(self._prep_slice(vol, z0))
            chans.append(self._prep_slice(vol, z1))
            chans.append(self._prep_slice(vol, z2))

        x = np.stack(chans, axis=0).astype(np.float32)

        m = mask_vol[:, :, z1]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z1)


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(8, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(8, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(dice_all), float(dice_pos), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {
            "dice_all": a,
            "dice_pos_only": p,
            "n_pos_slices": npos,
        }

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    name = report["split"]
    m = report["metrics"]
    s = report["sanity"]

    print(f"\n[{name}]")
    print(f"  slices_total            : {s['slices_total']}")
    print(f"  slices_any_tumor_gt     : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred   : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt        : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred      : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")

    for c in [1, 2, 3, 4]:
        k = f"c{c}"
        print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})")


def main():
    print("=== STEP 9 AUDIT EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9 AUDIT EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()



=== STEP 9 AUDIT EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt


RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [1]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 9 — 2.5D CONTEXT UNET TRAINING
# Input:
#   3 adjacent slices per modality => 12 channels total
# Output:
#   segmentation for the center slice
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5

NUM_WORKERS = 0
PIN_MEMORY = True

EPOCHS = 12
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2
BATCH = 2

P_TUMOR = 0.50
MAX_SLICES_PER_SUBJECT = 32
TUMOR_MIN_FRAC = 0.001

LRU_CACHE_MAX_SUBJECTS = 6
CACHE_IN_RAM = True

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999
RESUME_IF_EXISTS = True

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def valid_gn_groups(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


def lr_at_epoch(epoch: int, total_epochs: int, base_lr: float, warmup_epochs: int):
    if warmup_epochs > 0 and epoch < warmup_epochs:
        return base_lr * float(epoch + 1) / float(warmup_epochs)
    t = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * t))


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999, device: str = "cpu"):
        self.decay = float(decay)
        self.device = device
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)
        self.backup = None

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self.decay
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(d).add_(p.detach().to(self.device), alpha=1.0 - d)

    @torch.no_grad()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model: nn.Module):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


def soft_dice_loss_all_classes(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    tgt = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs_fg = probs[:, 1:]
    tgt_fg = tgt[:, 1:]

    dims = (0, 2, 3)
    inter = (probs_fg * tgt_fg).sum(dims)
    p_sum = probs_fg.sum(dims)
    t_sum = tgt_fg.sum(dims)

    dice = (2.0 * inter + eps) / (p_sum + t_sum + eps)
    empty = (t_sum < eps) & (p_sum < eps)
    dice = torch.where(empty, torch.ones_like(dice), dice)
    return 1.0 - dice.mean()


@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)
        dices.append(d)
    return torch.stack(dices).mean().item()


class BraTS2024SliceDataset2p5D(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        is_train: bool = True,
        p_tumor: float = 0.5,
        tumor_min_frac: float = 0.001,
        max_slices_per_subject: Optional[int] = 32,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.is_train = bool(is_train)
        self.p_tumor = float(p_tumor)
        self.tumor_min_frac = float(tumor_min_frac)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.all_samples: List[Tuple[str, int]] = []
        self.tumor_samples: List[Tuple[str, int]] = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]

            all_z = list(range(Z))
            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            tum_z = [z for z in all_z if (m[:, :, z] > 0).mean() >= self.tumor_min_frac]

            for z in all_z:
                self.all_samples.append((sid, int(z)))
            for z in tum_z:
                self.tumor_samples.append((sid, int(z)))

        if len(self.all_samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.all_samples)

    @staticmethod
    def _zscore(x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _clamp_z(self, z: int, Z: int) -> int:
        return max(0, min(Z - 1, z))

    def _prep_slice(self, vol, z):
        s = self._zscore(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        if self.is_train and (len(self.tumor_samples) > 0) and (random.random() < self.p_tumor):
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.all_samples[idx]

        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)
        Z = mask_vol.shape[2]

        z0 = self._clamp_z(z - 1, Z)
        z1 = self._clamp_z(z, Z)
        z2 = self._clamp_z(z + 1, Z)

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            chans.append(self._prep_slice(vol, z0))
            chans.append(self._prep_slice(vol, z1))
            chans.append(self._prep_slice(vol, z2))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(mask_vol[:, :, z1], (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=valid_gn_groups(out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=valid_gn_groups(out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def save_ckpt(path: str, model: nn.Module, ema: EMA, opt, scaler, epoch: int, best: float):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().clone().cpu() for k, v in ema.shadow.items()},
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best": float(best),
    }, path)


def load_ckpt(path: str, model: nn.Module, ema: EMA, opt=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)

    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)

    if opt is not None and "opt" in ckpt:
        opt.load_state_dict(ckpt["opt"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0))
    best = float(ckpt.get("best", -1.0))
    return start_epoch, best


def main():
    print("=== STEP 9 START (2.5D UNet) ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print(f"BATCH={BATCH} IMG_SIZE={IMG_SIZE} IN_CH={IN_CH} WORKERS={NUM_WORKERS}", flush=True)
    print(f"P_TUMOR={P_TUMOR} EPOCHS={EPOCHS} LR={BASE_LR}", flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)

    subjects = json.load(open(INDEX, "r", encoding="utf-8"))
    train_ids = json.load(open(TRAIN_SPLIT, "r", encoding="utf-8"))
    val_ids = json.load(open(VAL_SPLIT, "r", encoding="utf-8"))

    train_ds = BraTS2024SliceDataset2p5D(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )
    val_ds = BraTS2024SliceDataset2p5D(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(f"Train slices={len(train_ds)} | Val slices={len(val_ds)} | Tumor pool(train)={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0
    start_epoch = 0

    best_path = os.path.join(OUT_DIR, "best.pt")
    last_path = os.path.join(OUT_DIR, "last.pt")

    if RESUME_IF_EXISTS and os.path.exists(best_path):
        start_epoch, best = load_ckpt(best_path, model, ema, opt=opt, scaler=scaler)
        print(f"[RESUME] Loaded best.pt: start_epoch={start_epoch} best={best:.6f}", flush=True)

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_at_epoch(epoch, EPOCHS, BASE_LR, WARMUP_EPOCHS)
        for pg in opt.param_groups:
            pg["lr"] = lr

        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} TRAIN", leave=True)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()

            ema.update(model)

            tr_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        tr_loss /= max(1, len(train_loader.dataset))

        model.eval()
        ema.apply_to(model)

        va_loss, va_dice_sum, n = 0.0, 0.0, 0
        with torch.no_grad():
            pbarv = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} VAL(EMA)", leave=True)
            for x, y in pbarv:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = W_CE * ce(logits, y) + W_DICE * soft_dice_loss_all_classes(logits, y, NUM_CLASSES)

                va_loss += loss.item() * x.size(0)
                pred = torch.argmax(logits, dim=1)
                va_dice_sum += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbarv.set_postfix(vloss=f"{loss.item():.4f}")

        va_loss /= max(1, len(val_loader.dataset))
        va_dice = va_dice_sum / max(1, n)

        ema.restore(model)

        print(
            f"Epoch {epoch+1:02d} | train_loss={tr_loss:.4f} | "
            f"val_loss={va_loss:.4f} | val_meanDiceFG(EMA)={va_dice:.4f} | lr={lr:.2e}",
            flush=True
        )

        save_ckpt(last_path, model, ema, opt, scaler, epoch + 1, best)

        if va_dice > best:
            best = va_dice
            save_ckpt(best_path, model, ema, opt, scaler, epoch + 1, best)
            print("Saved best:", best_path, flush=True)

    print("=== STEP 9 DONE === Best val_meanDiceFG(EMA):", best, flush=True)
    print("Out:", OUT_DIR, flush=True)


if __name__ == "__main__":
    main()



=== STEP 9 START (2.5D UNet) ===
Device: cuda
Torch: 2.5.1+cu121
BATCH=2 IMG_SIZE=256 IN_CH=12 WORKERS=0
P_TUMOR=0.5 EPOCHS=12 LR=0.0002
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50
Train slices=36576 | Val slices=7936 | Tumor pool(train)=12357
[RESUME] Loaded best.pt: start_epoch=11 best=0.896820


C:\Users\admin\AppData\Local\Temp\ipykernel_3520\1953470615.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location="cpu")
Ep 12/12 VAL(EM

Epoch 12 | train_loss=0.1550 | val_loss=0.2196 | val_meanDiceFG(EMA)=0.8978 | lr=4.89e-06


Saved best: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50\best.pt
=== STEP 9 DONE === Best val_meanDiceFG(EMA): 0.8978123280948268
Out: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50


In [2]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 AUDIT EVAL — 2.5D UNet
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50_audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _clamp_z(self, z: int, Z: int) -> int:
        return max(0, min(Z - 1, z))

    def _prep_slice(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)
        Z = mask_vol.shape[2]

        z0 = self._clamp_z(z - 1, Z)
        z1 = self._clamp_z(z, Z)
        z2 = self._clamp_z(z + 1, Z)

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            chans.append(self._prep_slice(vol, z0))
            chans.append(self._prep_slice(vol, z1))
            chans.append(self._prep_slice(vol, z2))

        x = np.stack(chans, axis=0).astype(np.float32)

        m = mask_vol[:, :, z1]
        y = cv2.resize(m, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z1)


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(num_groups=min(8, out_c), num_channels=out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(num_groups=min(8, out_c), num_channels=out_c)
        self.drop = nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(dice_all), float(dice_pos), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {
            "dice_all": a,
            "dice_pos_only": p,
            "n_pos_slices": npos,
        }

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    name = report["split"]
    m = report["metrics"]
    s = report["sanity"]

    print(f"\n[{name}]")
    print(f"  slices_total            : {s['slices_total']}")
    print(f"  slices_any_tumor_gt     : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred   : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt        : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred      : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")

    for c in [1, 2, 3, 4]:
        k = f"c{c}"
        print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})")


def main():
    print("=== STEP 9 AUDIT EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9 AUDIT EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 9 AUDIT EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt
[VAL] EMA loaded: epoch=12 best=0.897812


C:\Users\admin\AppData\Local\Temp\ipykernel_3520\1283945527.py:204: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL (

[TEST] EMA loaded: epoch=12 best=0.897812


EVAL TEST (slice-avg): 100%|█████████████████████████████████████| 20930/20930 [12:54<00:00, 27.03it/s, mean_fg=0.9163]


=== SUMMARY (dice_all / dice_pos_only) ===



[VAL]
  slices_total            : 45136
  slices_any_tumor_gt     : 16476
  slices_any_tumor_pred   : 18616
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.918933 / 0.823376 (npos=16476)
  WT      : 0.872497 / 0.811000 (npos=16476)
  TC      : 0.905495 / 0.768514 (npos=9839)
  ET      : 0.894429 / 0.705560 (npos=6847)
  c1     : 0.965194 / 0.521204 (npos=2404)
  c2     : 0.874134 / 0.783556 (npos=16134)
  c3     : 0.941974 / 0.728492 (npos=5930)
  c4     : 0.894429 / 0.705560 (npos=6847)

[TEST]
  slices_total            : 41860
  slices_any_tumor_gt     : 14364
  slices_any_tumor_pred   : 16467
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.916321 / 0.810913 (npos=14364)
  WT      : 0.866664 / 0.797797 (npos=14364)
  TC      : 0.906577 / 0.745872 (npos=9490)
  ET      : 0.894223 / 0.669443 (npos=6661)
  c1     : 0.968385 / 0.528535 (npos=2054)
  c2     : 0.864616 / 0.760851 (n

In [1]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 9.1 — 5-SLICE 2.5D CONTEXT UNET
# 4 modalities × 5 adjacent slices = 20 input channels
# Expected improvement mainly in TC / ET
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

OUT_DIR = r"C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 20           # 4 modalities × 5 slices

EPOCHS = 12
BATCH = 2
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2

NUM_WORKERS = 0
PIN_MEMORY = True

P_TUMOR = 0.50
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 32

CACHE_IN_RAM = True
LRU_CACHE_MAX_SUBJECTS = 6

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999
RESUME_IF_EXISTS = True

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


# ============================================================
# Utils
# ============================================================

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def lr_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS

    t = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    return BASE_LR * 0.5 * (1 + math.cos(math.pi * t))


# ============================================================
# EMA
# ============================================================

class EMA:
    def __init__(self, model, decay=0.999, device="cpu"):
        self.decay = decay
        self.device = device
        self.shadow = {}
        self.backup = None

        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(
                    p.detach().to(self.device),
                    alpha=(1.0 - self.decay)
                )

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


# ============================================================
# Loss
# ============================================================

def soft_dice_loss(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs = probs[:, 1:]
    target_1h = target_1h[:, 1:]

    dims = (0, 2, 3)
    inter = (probs * target_1h).sum(dims)
    denom = probs.sum(dims) + target_1h.sum(dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    vals = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)
        vals.append(d)
    return torch.stack(vals).mean().item()


# ============================================================
# Dataset
# ============================================================

class BraTS2024SliceDataset5Slice(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        is_train=True,
        p_tumor=0.5,
        tumor_min_frac=0.001,
        max_slices_per_subject=32,
        cache_in_ram=True,
        lru_cache_max_subjects=6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.is_train = is_train
        self.p_tumor = p_tumor
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects

        self.samples = []
        self.tumor_samples = []

        for sid in self.sids:
            mask = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(mask.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > self.max_slices_per_subject:
                idx = np.linspace(0, len(z_all)-1, self.max_slices_per_subject).astype(int)
                z_all = [z_all[i] for i in idx]

            tumor_z = [
                z for z in z_all
                if (mask[:, :, z] > 0).mean() >= self.tumor_min_frac
            ]

            self.samples.extend([(sid, z) for z in z_all])
            self.tumor_samples.extend([(sid, z) for z in tumor_z])

    def __len__(self):
        return len(self.samples)

    def _zscore(self, x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        self.cache[sid] = data
        self.cache.move_to_end(sid)

        if len(self.cache) > self.cache_max:
            self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = self._zscore(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        if self.is_train and len(self.tumor_samples) > 0 and random.random() < self.p_tumor:
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.samples[idx]

        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 2, Z),
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
            self._safe_z(z + 2, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)

        y = cv2.resize(
            mask[:, :, zs[2]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)


# ============================================================
# Model
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet5Slice(nn.Module):
    def __init__(self, in_ch=20, num_classes=5, base=32):
        super().__init__()

        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


# ============================================================
# Checkpoint
# ============================================================

def save_ckpt(path, model, ema, optimizer, scaler, epoch, best):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().cpu() for k, v in ema.shadow.items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best": best,
    }, path)


# ============================================================
# Main
# ============================================================

def main():
    print("=== STEP 9.1 START ===")
    print("Device:", DEVICE)
    print("Torch:", torch.__version__)
    print("IN_CH:", IN_CH)

    subjects = json.load(open(INDEX, "r"))
    train_ids = json.load(open(TRAIN_SPLIT, "r"))
    val_ids = json.load(open(VAL_SPLIT, "r"))

    train_ds = BraTS2024SliceDataset5Slice(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )

    val_ds = BraTS2024SliceDataset5Slice(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(f"Train={len(train_ds)}  Val={len(val_ds)}  TumorTrain={len(train_ds.tumor_samples)}")

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet5Slice(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)

    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY
    )

    ce_loss = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0

    best_path = os.path.join(OUT_DIR, "best.pt")
    last_path = os.path.join(OUT_DIR, "last.pt")

    for epoch in range(EPOCHS):
        lr = lr_schedule(epoch)
        for g in optimizer.param_groups:
            g["lr"] = lr

        # ---------------- TRAIN ----------------
        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} TRAIN")
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True).float()
            y = y.to(DEVICE, non_blocking=True).long()

            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = (
                    W_CE * ce_loss(logits, y)
                    + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)

            train_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        train_loss /= len(train_loader.dataset)

        # ---------------- VAL ----------------
        model.eval()
        ema.apply_to(model)

        val_loss = 0.0
        val_dice = 0.0
        n = 0

        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} VAL(EMA)")
            for x, y in pbar:
                x = x.to(DEVICE, non_blocking=True).float()
                y = y.to(DEVICE, non_blocking=True).long()

                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = (
                        W_CE * ce_loss(logits, y)
                        + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                    )

                pred = torch.argmax(logits, dim=1)

                val_loss += loss.item() * x.size(0)
                val_dice += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbar.set_postfix(vloss=f"{loss.item():.4f}")

        ema.restore(model)

        val_loss /= len(val_loader.dataset)
        val_dice /= n

        print(
            f"Epoch {epoch+1:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_meanDiceFG(EMA)={val_dice:.6f} | "
            f"lr={lr:.2e}"
        )

        save_ckpt(last_path, model, ema, optimizer, scaler, epoch + 1, best)

        if val_dice > best:
            best = val_dice
            save_ckpt(best_path, model, ema, optimizer, scaler, epoch + 1, best)
            print("Saved best:", best_path)

    print("\n=== STEP 9.1 DONE ===")
    print("Best val_meanDiceFG(EMA):", best)
    print("Out:", OUT_DIR)


if __name__ == "__main__":
    main()

=== STEP 9.1 START ===
Device: cuda
Torch: 2.5.1+cu121
IN_CH: 20
Train=36576  Val=7936  TumorTrain=12357


Ep 1/12 VAL(EMA): 100%|██████████████████████████████████████████████| 3968/3968 [04:31<00:00, 14.63it/s, vloss=0.5000]


Epoch 01 | train_loss=0.3759 | val_loss=0.4415 | val_meanDiceFG(EMA)=0.846715 | lr=1.00e-04
Saved best: C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50\best.pt


Ep 2/12 VAL(EMA): 100%|██████████████████████████████████████████████| 3968/3968 [04:33<00:00, 14.49it/s, vloss=0.4853]


Epoch 02 | train_loss=0.3388 | val_loss=0.4356 | val_meanDiceFG(EMA)=0.848540 | lr=2.00e-04
Saved best: C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50\best.pt


...

Epoch 10 | train_loss=0.0967 | val_loss=0.0575 | val_meanDiceFG(EMA)=0.912394 | lr=4.12e-05
Saved best: C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50\best.pt


Ep 11/12 VAL(EMA): 100%|█████████████████████████████████████████████| 3968/3968 [05:25<00:00, 12.20it/s, vloss=0.0000]


Epoch 11 | train_loss=0.0922 | val_loss=0.0563 | val_meanDiceFG(EMA)=0.914557 | lr=1.91e-05
Saved best: C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50\best.pt


Ep 12/12 VAL(EMA): 100%|█████████████████████████████████████████████| 3968/3968 [05:26<00:00, 12.16it/s, vloss=0.0000]


Epoch 12 | train_loss=0.0892 | val_loss=0.0568 | val_meanDiceFG(EMA)=0.913611 | lr=4.89e-06

=== STEP 9.1 DONE ===
Best val_meanDiceFG(EMA): 0.9145573804477951
Out: C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50


In [2]:
import os, json, random
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9.1 AUDIT EVAL — 5-slice 2.5D UNet
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50_audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 20

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


class BraTS2024SliceDataset5SliceEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self.cache = OrderedDict()
        self.cache_max = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(m.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(z_all) - 1, int(self.max_slices_per_subject)).astype(int)
                z_all = [z_all[i] for i in idx]

            for z in z_all:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        self.cache[sid] = data
        self.cache.move_to_end(sid)
        if len(self.cache) > self.cache_max:
            self.cache.popitem(last=False)
        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 2, Z),
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
            self._safe_z(z + 2, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(mask[:, :, zs[2]], (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(zs[2])


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet5Slice(nn.Module):
    def __init__(self, in_ch=20, num_classes=5, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset5SliceEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet5Slice(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name} (slice-avg)", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i]
            yi = y[i]

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            dices_fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                dices_fg.append(dc)

            mean_fg = float(np.mean(dices_fg))
            update_stat(stats, "mean_fg", mean_fg, gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    def finalize(k: str):
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        return float(dice_all), float(dice_pos), int(stats[k]["n_pos"])

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": {}
    }

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        a, p, npos = finalize(k)
        report["metrics"][k] = {
            "dice_all": a,
            "dice_pos_only": p,
            "n_pos_slices": npos,
        }

    return report


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    name = report["split"]
    m = report["metrics"]
    s = report["sanity"]

    print(f"\n[{name}]")
    print(f"  slices_total            : {s['slices_total']}")
    print(f"  slices_any_tumor_gt     : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred   : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt        : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred      : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")

    for c in [1, 2, 3, 4]:
        k = f"c{c}"
        print(f"  {k:7s}: {m[k]['dice_all']:.6f} / {m[k]['dice_pos_only']:.6f} (npos={m[k]['n_pos_slices']})")


def main():
    print("=== STEP 9.1 AUDIT EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY (dice_all / dice_pos_only) ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9.1 AUDIT EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 9.1 AUDIT EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step9_1_5slice_2p5d_unet_ptumor_0.50/best.pt
[VAL] EMA loaded: epoch=11 best=0.914557


C:\Users\admin\AppData\Local\Temp\ipykernel_21664\2966801242.py:214: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL 

[TEST] EMA loaded: epoch=11 best=0.914557


EVAL TEST (slice-avg): 100%|█████████████████████████████████████| 20930/20930 [29:56<00:00, 11.65it/s, mean_fg=0.9283]


=== SUMMARY (dice_all / dice_pos_only) ===



[VAL]
  slices_total            : 45136
  slices_any_tumor_gt     : 16476
  slices_any_tumor_pred   : 17251
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.929316 / 0.831897 (npos=16476)
  WT      : 0.897338 / 0.809011 (npos=16476)
  TC      : 0.922433 / 0.762675 (npos=9839)
  ET      : 0.919197 / 0.681303 (npos=6847)
  c1     : 0.965365 / 0.525264 (npos=2404)
  c2     : 0.890568 / 0.780938 (npos=16134)
  c3     : 0.942133 / 0.728515 (npos=5930)
  c4     : 0.919197 / 0.681303 (npos=6847)

[TEST]
  slices_total            : 41860
  slices_any_tumor_gt     : 14364
  slices_any_tumor_pred   : 15281
  unique_labels_gt        : [0, 1, 2, 3, 4]
  unique_labels_pred      : [0, 1, 2, 3, 4]
  mean_fg : 0.928334 / 0.822634 (npos=14364)
  WT      : 0.891726 / 0.796063 (npos=14364)
  TC      : 0.925764 / 0.743044 (npos=9490)
  ET      : 0.922046 / 0.655886 (npos=6661)
  c1     : 0.969708 / 0.526284 (npos=2054)
  c2     : 0.880432 / 0.757306 (n

In [3]:
import os
import json
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9.2 — Step 9 + light postprocessing audit
# Uses the trained Step 9 3-slice 2.5D model and evaluates:
#   1) raw prediction
#   2) postprocessed prediction
#
# Postprocessing:
#   - remove tiny connected components from whole tumor mask
#   - optional stricter cleanup for ET
#
# Goal:
#   improve WT / ET positive-only on TEST without retraining
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50_postproc_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Runtime
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True

# -------------------------
# Postprocessing knobs
# Tune on VAL first, then inspect TEST
# -------------------------
# Remove tiny WT connected components
MIN_WT_COMPONENT_PIXELS = 40

# Remove tiny ET connected components
MIN_ET_COMPONENT_PIXELS = 12

# If ET component exists outside WT after cleanup, force it back into WT-supporting area
FORCE_ET_INSIDE_WT = True


# ============================================================
# Repro
# ============================================================
def seed_all(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def valid_groups(channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# ============================================================
# Dataset
# ============================================================
class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _clamp_z(self, z: int, Z: int) -> int:
        return max(0, min(Z - 1, z))

    def _prep_slice(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)
        Z = mask_vol.shape[2]

        z0 = self._clamp_z(z - 1, Z)
        z1 = self._clamp_z(z, Z)
        z2 = self._clamp_z(z + 1, Z)

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            chans.append(self._prep_slice(vol, z0))
            chans.append(self._prep_slice(vol, z1))
            chans.append(self._prep_slice(vol, z2))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(mask_vol[:, :, z1], (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z1)


# ============================================================
# Model (must match Step 9)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


# ============================================================
# EMA weight loader
# ============================================================
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


# ============================================================
# Postprocessing
# ============================================================
def remove_small_components(binary_mask: np.ndarray, min_pixels: int) -> np.ndarray:
    """
    binary_mask: uint8 or bool 2D
    returns cleaned uint8 mask {0,1}
    """
    binary_mask = (binary_mask > 0).astype(np.uint8)
    if binary_mask.sum() == 0:
        return binary_mask

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    out = np.zeros_like(binary_mask, dtype=np.uint8)

    for lab in range(1, num_labels):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_pixels:
            out[labels == lab] = 1
    return out


def postprocess_prediction(pred: np.ndarray) -> np.ndarray:
    """
    pred: (H,W) int64 label map in {0,1,2,3,4}
    Strategy:
      1) clean WT small components
      2) clean ET small components
      3) optionally force ET inside WT
    """
    pred = pred.astype(np.int64).copy()

    # Whole tumor mask
    wt = ((pred == 1) | (pred == 2) | (pred == 3) | (pred == 4)).astype(np.uint8)
    wt_clean = remove_small_components(wt, MIN_WT_COMPONENT_PIXELS)

    # Drop all labels outside cleaned WT
    pred[wt_clean == 0] = 0

    # ET cleanup
    et = (pred == 4).astype(np.uint8)
    et_clean = remove_small_components(et, MIN_ET_COMPONENT_PIXELS)

    # remove original ET
    pred[pred == 4] = 0

    if FORCE_ET_INSIDE_WT:
        et_clean = et_clean * wt_clean

    pred[et_clean > 0] = 4

    return pred


# ============================================================
# Metrics
# ============================================================
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats: Dict) -> Dict:
    out = {}
    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        out[k] = {
            "dice_all": float(dice_all),
            "dice_pos_only": float(dice_pos),
            "n_pos_slices": int(stats[k]["n_pos"]),
        }
    return out


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    raw_stats = init_stats()
    pp_stats = init_stats()

    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred_raw": 0,
        "slices_any_tumor_pred_pp": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred_raw": set(),
        "unique_labels_pred_pp": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

        pred_raw = torch.argmax(logits, dim=1)

        b = pred_raw.shape[0]
        for i in range(b):
            pr = pred_raw[i].detach().cpu().numpy().astype(np.int64)
            gt = y[i].detach().cpu().numpy().astype(np.int64)
            pp = postprocess_prediction(pr)

            pr_t = torch.from_numpy(pr)
            gt_t = torch.from_numpy(gt)
            pp_t = torch.from_numpy(pp)

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(np.unique(gt).tolist())
            sanity["unique_labels_pred_raw"].update(np.unique(pr).tolist())
            sanity["unique_labels_pred_pp"].update(np.unique(pp).tolist())

            gt_any = (gt_t > 0).any().item()
            pr_any = (pr_t > 0).any().item()
            pp_any = (pp_t > 0).any().item()

            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred_raw"] += int(pr_any)
            sanity["slices_any_tumor_pred_pp"] += int(pp_any)

            # raw per-class
            raw_fg = []
            pp_fg = []
            for c in [1, 2, 3, 4]:
                dr = dice_bool(pr_t == c, gt_t == c)
                dp = dice_bool(pp_t == c, gt_t == c)

                update_stat(raw_stats, f"c{c}", dr, gt_pos=(gt_t == c).any().item())
                update_stat(pp_stats, f"c{c}", dp, gt_pos=(gt_t == c).any().item())

                raw_fg.append(dr)
                pp_fg.append(dp)

            update_stat(raw_stats, "mean_fg", float(np.mean(raw_fg)), gt_pos=gt_any)
            update_stat(pp_stats, "mean_fg", float(np.mean(pp_fg)), gt_pos=gt_any)

            # composites
            pr_m = masks_composite(pr_t)
            pp_m = masks_composite(pp_t)
            gt_m = masks_composite(gt_t)

            for k in ["WT", "TC", "ET"]:
                dr = dice_bool(pr_m[k], gt_m[k])
                dp = dice_bool(pp_m[k], gt_m[k])

                update_stat(raw_stats, k, dr, gt_pos=gt_m[k].any().item())
                update_stat(pp_stats, k, dp, gt_pos=gt_m[k].any().item())

        running_raw = raw_stats["mean_fg"]["sum_all"] / max(1, raw_stats["mean_fg"]["n_all"])
        running_pp = pp_stats["mean_fg"]["sum_all"] / max(1, pp_stats["mean_fg"]["n_all"])
        pbar.set_postfix(raw=f"{running_raw:.4f}", pp=f"{running_pp:.4f}")

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "postprocess": {
            "MIN_WT_COMPONENT_PIXELS": MIN_WT_COMPONENT_PIXELS,
            "MIN_ET_COMPONENT_PIXELS": MIN_ET_COMPONENT_PIXELS,
            "FORCE_ET_INSIDE_WT": FORCE_ET_INSIDE_WT,
        },
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred_raw": sanity["slices_any_tumor_pred_raw"],
            "slices_any_tumor_pred_pp": sanity["slices_any_tumor_pred_pp"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred_raw": sorted(list(sanity["unique_labels_pred_raw"])),
            "unique_labels_pred_pp": sorted(list(sanity["unique_labels_pred_pp"])),
        },
        "raw_metrics": finalize_stats(raw_stats),
        "postprocessed_metrics": finalize_stats(pp_stats),
    }
    return report


def save_json(path: str, obj: Dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def print_summary(report: Dict):
    print(f"\n[{report['split']}]")
    s = report["sanity"]
    print(f"  slices_total              : {s['slices_total']}")
    print(f"  slices_any_tumor_gt       : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred_raw : {s['slices_any_tumor_pred_raw']}")
    print(f"  slices_any_tumor_pred_pp  : {s['slices_any_tumor_pred_pp']}")
    print(f"  unique_labels_gt          : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred_raw    : {s['unique_labels_pred_raw']}")
    print(f"  unique_labels_pred_pp     : {s['unique_labels_pred_pp']}")

    for title, key in [("RAW", "raw_metrics"), ("POSTPROCESSED", "postprocessed_metrics")]:
        m = report[key]
        print(f"\n  {title}")
        print(f"    mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
        print(f"    WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
        print(f"    TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
        print(f"    ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")


def main():
    print("=== STEP 9.2 POSTPROCESS EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)
    print(
        f"Postprocess: WT>={MIN_WT_COMPONENT_PIXELS}, ET>={MIN_ET_COMPONENT_PIXELS}, "
        f"FORCE_ET_INSIDE_WT={FORCE_ET_INSIDE_WT}",
        flush=True
    )

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9.2 POSTPROCESS EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 9.2 POSTPROCESS EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50_postproc_eval
Postprocess: WT>=40, ET>=12, FORCE_ET_INSIDE_WT=True
[VAL] EMA loaded: epoch=12 best=0.897812


C:\Users\admin\AppData\Local\Temp\ipykernel_21664\1009348763.py:253: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL:

[TEST] EMA loaded: epoch=12 best=0.897812


EVAL TEST: 100%|██████████████████████████████████████████| 20930/20930 [19:22<00:00, 18.00it/s, pp=0.9272, raw=0.9163]


=== SUMMARY ===

[VAL]
  slices_total              : 45136
  slices_any_tumor_gt       : 16476
  slices_any_tumor_pred_raw : 18616
  slices_any_tumor_pred_pp  : 16860
  unique_labels_gt          : [0, 1, 2, 3, 4]
  unique_labels_pred_raw    : [0, 1, 2, 3, 4]
  unique_labels_pred_pp     : [0, 1, 2, 3, 4]

  RAW
    mean_fg : 0.918933 / 0.823376 (npos=16476)
    WT      : 0.872496 / 0.810996 (npos=16476)
    TC      : 0.905495 / 0.768516 (npos=9839)
    ET      : 0.894430 / 0.705564 (npos=6847)

  POSTPROCESSED
    mean_fg : 0.929887 / 0.830973 (npos=16476)
    WT      : 0.897831 / 0.800772 (npos=16476)
    TC      : 0.921750 / 0.765436 (npos=9839)
    ET      : 0.916801 / 0.698660 (npos=6847)

[TEST]
  slices_total              : 41860
  slices_any_tumor_gt       : 14364
  slices_any_tumor_pred_raw : 16468
  slices_any_tumor_pred_pp  : 14828
  unique_labels_gt          : [0, 1, 2, 3, 4]
  unique_labels_pred_raw    : [0, 1, 2, 3, 4]
  unique_labels_pred_pp     : [0, 1, 2, 3, 4]

  RAW
 

In [4]:
import os
import json
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9.2 — Step 9 + LIGHTER postprocessing audit
# Uses the trained Step 9 3-slice 2.5D model and evaluates:
#   1) raw prediction
#   2) postprocessed prediction
#
# Updated lighter settings:
#   - much smaller WT/ET component thresholds
#   - do NOT force ET inside WT
# Goal:
#   remove only tiny speckles without hurting true lesions
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt"

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50_postproc_eval_light"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Runtime
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True

# -------------------------
# Lighter postprocessing knobs
# -------------------------
MIN_WT_COMPONENT_PIXELS = 8
MIN_ET_COMPONENT_PIXELS = 4
FORCE_ET_INSIDE_WT = False


# ============================================================
# Repro
# ============================================================
def seed_all(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def valid_groups(channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# ============================================================
# Dataset
# ============================================================
class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _clamp_z(self, z: int, Z: int) -> int:
        return max(0, min(Z - 1, z))

    def _prep_slice(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)
        Z = mask_vol.shape[2]

        z0 = self._clamp_z(z - 1, Z)
        z1 = self._clamp_z(z, Z)
        z2 = self._clamp_z(z + 1, Z)

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            chans.append(self._prep_slice(vol, z0))
            chans.append(self._prep_slice(vol, z1))
            chans.append(self._prep_slice(vol, z2))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(mask_vol[:, :, z1], (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z1)


# ============================================================
# Model (must match Step 9)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


# ============================================================
# EMA weight loader
# ============================================================
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError("Checkpoint missing EMA dict under key='ema'.")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


# ============================================================
# Postprocessing
# ============================================================
def remove_small_components(binary_mask: np.ndarray, min_pixels: int) -> np.ndarray:
    binary_mask = (binary_mask > 0).astype(np.uint8)
    if binary_mask.sum() == 0:
        return binary_mask

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    out = np.zeros_like(binary_mask, dtype=np.uint8)

    for lab in range(1, num_labels):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_pixels:
            out[labels == lab] = 1
    return out


def postprocess_prediction(pred: np.ndarray) -> np.ndarray:
    """
    Lighter cleanup:
      - remove only tiny WT speckles
      - remove only tiny ET speckles
      - do NOT force ET inside WT
    """
    pred = pred.astype(np.int64).copy()

    wt = ((pred == 1) | (pred == 2) | (pred == 3) | (pred == 4)).astype(np.uint8)
    wt_clean = remove_small_components(wt, MIN_WT_COMPONENT_PIXELS)

    # drop labels outside cleaned WT
    pred[wt_clean == 0] = 0

    # ET cleanup
    et = (pred == 4).astype(np.uint8)
    et_clean = remove_small_components(et, MIN_ET_COMPONENT_PIXELS)

    pred[pred == 4] = 0

    if FORCE_ET_INSIDE_WT:
        et_clean = et_clean * wt_clean

    pred[et_clean > 0] = 4
    return pred


# ============================================================
# Metrics
# ============================================================
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats: Dict) -> Dict:
    out = {}
    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        out[k] = {
            "dice_all": float(dice_all),
            "dice_pos_only": float(dice_pos),
            "n_pos_slices": int(stats[k]["n_pos"]),
        }
    return out


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(f"[{split_name}] EMA loaded: epoch={meta['epoch']} best={meta['best']:.6f}", flush=True)

    raw_stats = init_stats()
    pp_stats = init_stats()

    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred_raw": 0,
        "slices_any_tumor_pred_pp": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred_raw": set(),
        "unique_labels_pred_pp": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

        pred_raw = torch.argmax(logits, dim=1)

        b = pred_raw.shape[0]
        for i in range(b):
            pr = pred_raw[i].detach().cpu().numpy().astype(np.int64)
            gt = y[i].detach().cpu().numpy().astype(np.int64)
            pp = postprocess_prediction(pr)

            pr_t = torch.from_numpy(pr)
            gt_t = torch.from_numpy(gt)
            pp_t = torch.from_numpy(pp)

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(np.unique(gt).tolist())
            sanity["unique_labels_pred_raw"].update(np.unique(pr).tolist())
            sanity["unique_labels_pred_pp"].update(np.unique(pp).tolist())

            gt_any = (gt_t > 0).any().item()
            pr_any = (pr_t > 0).any().item()
            pp_any = (pp_t > 0).any().item()

            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred_raw"] += int(pr_any)
            sanity["slices_any_tumor_pred_pp"] += int(pp_any)

            raw_fg = []
            pp_fg = []
            for c in [1, 2, 3, 4]:
                dr = dice_bool(pr_t == c, gt_t == c)
                dp = dice_bool(pp_t == c, gt_t == c)

                update_stat(raw_stats, f"c{c}", dr, gt_pos=(gt_t == c).any().item())
                update_stat(pp_stats, f"c{c}", dp, gt_pos=(gt_t == c).any().item())

                raw_fg.append(dr)
                pp_fg.append(dp)

            update_stat(raw_stats, "mean_fg", float(np.mean(raw_fg)), gt_pos=gt_any)
            update_stat(pp_stats, "mean_fg", float(np.mean(pp_fg)), gt_pos=gt_any)

            pr_m = masks_composite(pr_t)
            pp_m = masks_composite(pp_t)
            gt_m = masks_composite(gt_t)

            for k in ["WT", "TC", "ET"]:
                dr = dice_bool(pr_m[k], gt_m[k])
                dp = dice_bool(pp_m[k], gt_m[k])

                update_stat(raw_stats, k, dr, gt_pos=gt_m[k].any().item())
                update_stat(pp_stats, k, dp, gt_pos=gt_m[k].any().item())

        running_raw = raw_stats["mean_fg"]["sum_all"] / max(1, raw_stats["mean_fg"]["n_all"])
        running_pp = pp_stats["mean_fg"]["sum_all"] / max(1, pp_stats["mean_fg"]["n_all"])
        pbar.set_postfix(raw=f"{running_raw:.4f}", pp=f"{running_pp:.4f}")

    report = {
        "split": split_name,
        "ckpt": CKPT_BEST,
        "meta": meta,
        "postprocess": {
            "MIN_WT_COMPONENT_PIXELS": MIN_WT_COMPONENT_PIXELS,
            "MIN_ET_COMPONENT_PIXELS": MIN_ET_COMPONENT_PIXELS,
            "FORCE_ET_INSIDE_WT": FORCE_ET_INSIDE_WT,
        },
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred_raw": sanity["slices_any_tumor_pred_raw"],
            "slices_any_tumor_pred_pp": sanity["slices_any_tumor_pred_pp"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred_raw": sorted(list(sanity["unique_labels_pred_raw"])),
            "unique_labels_pred_pp": sorted(list(sanity["unique_labels_pred_pp"])),
        },
        "raw_metrics": finalize_stats(raw_stats),
        "postprocessed_metrics": finalize_stats(pp_stats),
    }
    return report


def save_json(path: str, obj: Dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def print_summary(report: Dict):
    print(f"\n[{report['split']}]")
    s = report["sanity"]
    print(f"  slices_total              : {s['slices_total']}")
    print(f"  slices_any_tumor_gt       : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred_raw : {s['slices_any_tumor_pred_raw']}")
    print(f"  slices_any_tumor_pred_pp  : {s['slices_any_tumor_pred_pp']}")
    print(f"  unique_labels_gt          : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred_raw    : {s['unique_labels_pred_raw']}")
    print(f"  unique_labels_pred_pp     : {s['unique_labels_pred_pp']}")

    for title, key in [("RAW", "raw_metrics"), ("POSTPROCESSED", "postprocessed_metrics")]:
        m = report[key]
        print(f"\n  {title}")
        print(f"    mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
        print(f"    WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
        print(f"    TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
        print(f"    ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")


def main():
    print("=== STEP 9.2 LIGHT POSTPROCESS EVAL START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT:", CKPT_BEST, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)
    print(
        f"Postprocess: WT>={MIN_WT_COMPONENT_PIXELS}, ET>={MIN_ET_COMPONENT_PIXELS}, "
        f"FORCE_ET_INSIDE_WT={FORCE_ET_INSIDE_WT}",
        flush=True
    )

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n=== SUMMARY ===", flush=True)
    print_summary(val_report)
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9.2 LIGHT POSTPROCESS EVAL DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 9.2 LIGHT POSTPROCESS EVAL START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50/best.pt
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_ptumor_0.50_postproc_eval_light
Postprocess: WT>=8, ET>=4, FORCE_ET_INSIDE_WT=False
[VAL] EMA loaded: epoch=12 best=0.897812


C:\Users\admin\AppData\Local\Temp\ipykernel_21664\733373996.py:246: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL: 

[TEST] EMA loaded: epoch=12 best=0.897812


EVAL TEST: 100%|██████████████████████████████████████████| 20930/20930 [19:25<00:00, 17.96it/s, pp=0.9201, raw=0.9163]


=== SUMMARY ===

[VAL]
  slices_total              : 45136
  slices_any_tumor_gt       : 16476
  slices_any_tumor_pred_raw : 18616
  slices_any_tumor_pred_pp  : 18141
  unique_labels_gt          : [0, 1, 2, 3, 4]
  unique_labels_pred_raw    : [0, 1, 2, 3, 4]
  unique_labels_pred_pp     : [0, 1, 2, 3, 4]

  RAW
    mean_fg : 0.918933 / 0.823376 (npos=16476)
    WT      : 0.872496 / 0.810996 (npos=16476)
    TC      : 0.905495 / 0.768516 (npos=9839)
    ET      : 0.894430 / 0.705564 (npos=6847)

  POSTPROCESSED
    mean_fg : 0.922900 / 0.826946 (npos=16476)
    WT      : 0.881596 / 0.810557 (npos=16476)
    TC      : 0.911240 / 0.768142 (npos=9839)
    ET      : 0.902580 / 0.704664 (npos=6847)

[TEST]
  slices_total              : 41860
  slices_any_tumor_gt       : 14364
  slices_any_tumor_pred_raw : 16468
  slices_any_tumor_pred_pp  : 16023
  unique_labels_gt          : [0, 1, 2, 3, 4]
  unique_labels_pred_raw    : [0, 1, 2, 3, 4]
  unique_labels_pred_pp     : [0, 1, 2, 3, 4]

  RAW
 

In [1]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 9 MULTI-SEED RUNNER
# Trains the winning 3-slice 2.5D model for multiple seeds
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

BASE_OUT = r"C:/GDNET/logs/step9_2p5d_unet_multiseed"

# Choose:
SEEDS = [43, 44]          # two extra runs
# SEEDS = [42, 43, 44]    # all three runs

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5

EPOCHS = 12
BATCH = 2
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2

NUM_WORKERS = 0
PIN_MEMORY = True

P_TUMOR = 0.50
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 32

CACHE_IN_RAM = True
LRU_CACHE_MAX_SUBJECTS = 6

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999
RESUME_IF_EXISTS = True

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


# ============================================================
# Utils
# ============================================================

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def lr_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    return BASE_LR * 0.5 * (1 + math.cos(math.pi * t))


# ============================================================
# EMA
# ============================================================

class EMA:
    def __init__(self, model, decay=0.999, device="cpu"):
        self.decay = decay
        self.device = device
        self.shadow = {}
        self.backup = None

        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(
                    p.detach().to(self.device),
                    alpha=(1.0 - self.decay)
                )

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


# ============================================================
# Loss
# ============================================================

def soft_dice_loss(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs = probs[:, 1:]
    target_1h = target_1h[:, 1:]

    dims = (0, 2, 3)
    inter = (probs * target_1h).sum(dims)
    denom = probs.sum(dims) + target_1h.sum(dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    vals = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)
        vals.append(d)
    return torch.stack(vals).mean().item()


# ============================================================
# Dataset
# ============================================================

class BraTS2024SliceDataset2p5D(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        is_train=True,
        p_tumor=0.5,
        tumor_min_frac=0.001,
        max_slices_per_subject=32,
        cache_in_ram=True,
        lru_cache_max_subjects=6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.is_train = is_train
        self.p_tumor = p_tumor
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects

        self.samples = []
        self.tumor_samples = []

        for sid in self.sids:
            mask = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(mask.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > self.max_slices_per_subject:
                idx = np.linspace(0, len(z_all)-1, self.max_slices_per_subject).astype(int)
                z_all = [z_all[i] for i in idx]

            tumor_z = [
                z for z in z_all
                if (mask[:, :, z] > 0).mean() >= self.tumor_min_frac
            ]

            self.samples.extend([(sid, z) for z in z_all])
            self.tumor_samples.extend([(sid, z) for z in tumor_z])

    def __len__(self):
        return len(self.samples)

    def _zscore(self, x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        self.cache[sid] = data
        self.cache.move_to_end(sid)

        if len(self.cache) > self.cache_max:
            self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = self._zscore(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        if self.is_train and len(self.tumor_samples) > 0 and random.random() < self.p_tumor:
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.samples[idx]

        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)

        y = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)


# ============================================================
# Model
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_ch=12, num_classes=5, base=32):
        super().__init__()

        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


# ============================================================
# Checkpoint
# ============================================================

def save_ckpt(path, model, ema, optimizer, scaler, epoch, best, seed):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().cpu() for k, v in ema.shadow.items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best": best,
        "seed": seed,
    }, path)


def load_ckpt(path, model, ema, optimizer=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=True)

    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)

    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0))
    best = float(ckpt.get("best", -1.0))
    return start_epoch, best


# ============================================================
# One seed run
# ============================================================

def run_one_seed(seed: int):
    print(f"\n========== SEED {seed} ==========", flush=True)
    seed_all(seed)

    out_dir = os.path.join(BASE_OUT, f"seed_{seed}")
    os.makedirs(out_dir, exist_ok=True)

    subjects = json.load(open(INDEX, "r"))
    train_ids = json.load(open(TRAIN_SPLIT, "r"))
    val_ids = json.load(open(VAL_SPLIT, "r"))

    train_ds = BraTS2024SliceDataset2p5D(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )

    val_ds = BraTS2024SliceDataset2p5D(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(f"Seed {seed} | Train={len(train_ds)} Val={len(val_ds)} TumorTrain={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)

    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY
    )

    ce_loss = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0
    start_epoch = 0

    best_path = os.path.join(out_dir, "best.pt")
    last_path = os.path.join(out_dir, "last.pt")

    if RESUME_IF_EXISTS and os.path.exists(best_path):
        start_epoch, best = load_ckpt(best_path, model, ema, optimizer=optimizer, scaler=scaler)
        print(f"[RESUME seed {seed}] start_epoch={start_epoch} best={best:.6f}", flush=True)

    history = []

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_schedule(epoch)
        for g in optimizer.param_groups:
            g["lr"] = lr

        # ---------------- TRAIN ----------------
        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Seed {seed} Ep {epoch+1}/{EPOCHS} TRAIN")
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True).float()
            y = y.to(DEVICE, non_blocking=True).long()

            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = (
                    W_CE * ce_loss(logits, y)
                    + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)

            train_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        train_loss /= len(train_loader.dataset)

        # ---------------- VAL ----------------
        model.eval()
        ema.apply_to(model)

        val_loss = 0.0
        val_dice = 0.0
        n = 0

        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Seed {seed} Ep {epoch+1}/{EPOCHS} VAL(EMA)")
            for x, y in pbar:
                x = x.to(DEVICE, non_blocking=True).float()
                y = y.to(DEVICE, non_blocking=True).long()

                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = (
                        W_CE * ce_loss(logits, y)
                        + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                    )

                pred = torch.argmax(logits, dim=1)

                val_loss += loss.item() * x.size(0)
                val_dice += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbar.set_postfix(vloss=f"{loss.item():.4f}")

        ema.restore(model)

        val_loss /= len(val_loader.dataset)
        val_dice /= n

        row = {
            "seed": seed,
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_meanDiceFG_ema": float(val_dice),
            "lr": float(lr),
        }
        history.append(row)

        print(
            f"Seed {seed} | Epoch {epoch+1:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_meanDiceFG(EMA)={val_dice:.6f} | "
            f"lr={lr:.2e}",
            flush=True
        )

        save_ckpt(last_path, model, ema, optimizer, scaler, epoch + 1, best, seed)

        if val_dice > best:
            best = val_dice
            save_ckpt(best_path, model, ema, optimizer, scaler, epoch + 1, best, seed)
            print(f"Saved best: {best_path}", flush=True)

    # save history
    hist_path = os.path.join(out_dir, "history.json")
    with open(hist_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    print(f"=== SEED {seed} DONE === best={best:.6f}", flush=True)

    return {
        "seed": seed,
        "best_val_meanDiceFG_ema": float(best),
        "best_path": best_path,
        "history_path": hist_path,
        "out_dir": out_dir,
    }


# ============================================================
# Main
# ============================================================

def main():
    print("=== STEP 9 MULTI-SEED START ===")
    print("Device:", DEVICE)
    print("Torch:", torch.__version__)
    print("Seeds:", SEEDS)
    print("BASE_OUT:", BASE_OUT)

    os.makedirs(BASE_OUT, exist_ok=True)

    results = []
    for seed in SEEDS:
        results.append(run_one_seed(seed))

    summary_path = os.path.join(BASE_OUT, "multiseed_summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    vals = [r["best_val_meanDiceFG_ema"] for r in results]
    mean_val = float(np.mean(vals)) if len(vals) > 0 else float("nan")
    std_val = float(np.std(vals)) if len(vals) > 0 else float("nan")

    print("\n=== MULTI-SEED SUMMARY ===")
    for r in results:
        print(
            f"Seed {r['seed']} | best_val_meanDiceFG(EMA)={r['best_val_meanDiceFG_ema']:.6f} | {r['best_path']}"
        )
    print(f"\nMean ± std = {mean_val:.6f} ± {std_val:.6f}")
    print("Saved summary:", summary_path)


if __name__ == "__main__":
    main()

=== STEP 9 MULTI-SEED START ===
Device: cuda
Torch: 2.5.1+cu121
Seeds: [43, 44]
BASE_OUT: C:/GDNET/logs/step9_2p5d_unet_multiseed

========== SEED 43 ==========
Seed 43 | Train=36576 Val=7936 TumorTrain=12357
[RESUME seed 43] start_epoch=11 best=0.914629


C:\Users\admin\AppData\Local\Temp\ipykernel_21072\674996070.py:359: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location="cpu")
Seed 43 Ep 12/1

Seed 43 | Epoch 12 | train_loss=0.0939 | val_loss=0.0591 | val_meanDiceFG(EMA)=0.914462 | lr=4.89e-06


=== SEED 43 DONE === best=0.914629

========== SEED 44 ==========
Seed 44 | Train=36576 Val=7936 TumorTrain=12357


Seed 44 Ep 1/12 VAL(EMA): 100%|██████████████████████████████████████| 3968/3968 [03:55<00:00, 16.81it/s, vloss=0.5000]

...

Seed 44 Ep 11/12 VAL(EMA): 100%|█████████████████████████████████████| 3968/3968 [05:05<00:00, 12.99it/s, vloss=0.0000]

Seed 44 | Epoch 11 | train_loss=0.0954 | val_loss=0.0607 | val_meanDiceFG(EMA)=0.912227 | lr=1.91e-05



Seed 44 Ep 12/12 VAL(EMA): 100%|█████████████████████████████████████| 3968/3968 [05:01<00:00, 13.16it/s, vloss=0.0000]

Seed 44 | Epoch 12 | train_loss=0.0914 | val_loss=0.0610 | val_meanDiceFG(EMA)=0.910497 | lr=4.89e-06


=== SEED 44 DONE === best=0.912736

=== MULTI-SEED SUMMARY ===
Seed 43 | best_val_meanDiceFG(EMA)=0.914629 | C:/GDNET/logs/step9_2p5d_unet_multiseed\seed_43\best.pt
Seed 44 | best_val_meanDiceFG(EMA)=0.912736 | C:/GDNET/logs/step9_2p5d_unet_multiseed\seed_44\best.pt

Mean ± std = 0.913683 ± 0.000947
Saved summary: C:/GDNET/logs/step9_2p5d_unet_multiseed\multiseed_summary.json


In [2]:
import os, json
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 MULTI-SEED AUDIT EVAL
# Evaluates seed_43 and seed_44 best checkpoints
# Summarizes mean ± std for VAL/TEST
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPTS = {
    43: r"C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43/best.pt",
    44: r"C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44/best.pt",
}

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


def seed_all(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


def valid_groups(channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            Z = m.shape[2]
            all_z = list(range(Z))

            if self.max_slices_per_subject is not None and len(all_z) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(all_z) - 1, int(self.max_slices_per_subject)).astype(int)
                all_z = [all_z[i] for i in idx]

            for z in all_z:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if not self.cache_in_ram:
            return self._load_subject_nocache(sid)

        if sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        data = self._load_subject_nocache(sid)
        self._ram_cache[sid] = data
        self._ram_cache.move_to_end(sid)
        if len(self._ram_cache) > self.cache_max_subjects:
            self._ram_cache.popitem(last=False)
        return data

    def _load_subject_nocache(self, sid):
        p = self.subjects[sid]
        t1 = nib.load(p["t1"]).get_fdata().astype(np.float32)
        t1ce = nib.load(p["t1ce"]).get_fdata().astype(np.float32)
        t2 = nib.load(p["t2"]).get_fdata().astype(np.float32)
        flair = nib.load(p["flair"]).get_fdata().astype(np.float32)
        mask = nib.load(p["mask"]).get_fdata().astype(np.int16)
        return (t1, t1ce, t2, flair, mask)

    def _clamp_z(self, z: int, Z: int) -> int:
        return max(0, min(Z - 1, z))

    def _prep_slice(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask_vol = self._load_subject(sid)
        Z = mask_vol.shape[2]

        z0 = self._clamp_z(z - 1, Z)
        z1 = self._clamp_z(z, Z)
        z2 = self._clamp_z(z + 1, Z)

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            chans.append(self._prep_slice(vol, z0))
            chans.append(self._prep_slice(vol, z1))
            chans.append(self._prep_slice(vol, z2))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(mask_vol[:, :, z1], (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(z1)


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError(f"Checkpoint missing EMA dict: {ckpt_path}")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
    }


def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats: Dict) -> Dict:
    out = {}
    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        out[k] = {
            "dice_all": float(dice_all),
            "dice_pos_only": float(dice_pos),
            "n_pos_slices": int(stats[k]["n_pos"]),
        }
    return out


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str, ckpt_path: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, ckpt_path)

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"{split_name} | {os.path.basename(os.path.dirname(ckpt_path))}", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i].detach().cpu()
            yi = y[i].detach().cpu()

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                fg.append(dc)

            update_stat(stats, "mean_fg", float(np.mean(fg)), gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    return {
        "ckpt": ckpt_path,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": finalize_stats(stats),
    }


def aggregate_metric(reports: Dict[int, Dict], metric_key: str, subkey: str):
    vals = [reports[s]["metrics"][metric_key][subkey] for s in sorted(reports.keys())]
    return float(np.mean(vals)), float(np.std(vals)), vals


def print_aggregate(title: str, reports: Dict[int, Dict]):
    print(f"\n=== {title} ===")
    for metric in ["mean_fg", "WT", "TC", "ET"]:
        mean_all, std_all, _ = aggregate_metric(reports, metric, "dice_all")
        mean_pos, std_pos, _ = aggregate_metric(reports, metric, "dice_pos_only")
        npos = reports[sorted(reports.keys())[0]]["metrics"][metric]["n_pos_slices"]
        print(
            f"{metric:7s}: "
            f"{mean_all:.6f} ± {std_all:.6f}  /  "
            f"{mean_pos:.6f} ± {std_pos:.6f}  "
            f"(npos={npos})"
        )


def main():
    print("=== STEP 9 MULTI-SEED AUDIT START ===")
    print("Device:", DEVICE)
    print("Torch:", torch.__version__)
    print("CKPTS:", CKPTS)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    all_reports = {"VAL": {}, "TEST": {}}

    for seed, ckpt_path in CKPTS.items():
        print(f"\n----- Seed {seed} -----")
        val_report = eval_split(subjects, val_ids, "VAL", ckpt_path)
        test_report = eval_split(subjects, test_ids, "TEST", ckpt_path)

        all_reports["VAL"][seed] = val_report
        all_reports["TEST"][seed] = test_report

    with open(os.path.join(OUT_DIR, "multiseed_audit_reports.json"), "w", encoding="utf-8") as f:
        json.dump(all_reports, f, indent=2)

    print_aggregate("VAL mean ± std", all_reports["VAL"])
    print_aggregate("TEST mean ± std", all_reports["TEST"])

    print("\nSaved:", os.path.join(OUT_DIR, "multiseed_audit_reports.json"))
    print("=== STEP 9 MULTI-SEED AUDIT DONE ===")


if __name__ == "__main__":
    main()

=== STEP 9 MULTI-SEED AUDIT START ===
Device: cuda
Torch: 2.5.1+cu121
CKPTS: {43: 'C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43/best.pt', 44: 'C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44/best.pt'}

----- Seed 43 -----


C:\Users\admin\AppData\Local\Temp\ipykernel_21072\4206598034.py:213: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
VAL | see


----- Seed 44 -----


VAL | seed_44: 100%|█████████████████████████████████████████████| 22568/22568 [17:55<00:00, 20.98it/s, mean_fg=0.2613]
TEST | seed_44: 100%|████████████████████████████████████████████| 20930/20930 [16:38<00:00, 20.97it/s, mean_fg=0.4174]



=== VAL mean ± std ===
mean_fg: 0.273307 ± 0.012011  /  0.172044 ± 0.016328  (npos=16476)
WT     : 0.003735 ± 0.002448  /  0.010232 ± 0.006706  (npos=16476)
TC     : 0.001945 ± 0.000980  /  0.008315 ± 0.005104  (npos=9839)
ET     : 0.273054 ± 0.059340  /  0.000446 ± 0.000313  (npos=6847)

=== TEST mean ± std ===
mean_fg: 0.263465 ± 0.153946  /  0.197551 ± 0.125813  (npos=14364)
WT     : 0.014235 ± 0.006226  /  0.041483 ± 0.018144  (npos=14364)
TC     : 0.005188 ± 0.002534  /  0.005972 ± 0.005734  (npos=9490)
ET     : 0.253168 ± 0.244850  /  0.001592 ± 0.001588  (npos=6661)

Saved: C:/GDNET/logs/step9_2p5d_unet_multiseed_audit_eval\multiseed_audit_reports.json
=== STEP 9 MULTI-SEED AUDIT DONE ===


In [3]:
import os
import json
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 SINGLE-SEED AUDIT EVAL
# Use this same script twice:
#   1) for seed_43/best.pt
#   2) for seed_44/best.pt
#
# Only change:
#   CKPT_BEST
#   OUT_DIR
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

# ---------- CHANGE THESE ----------
CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43/best.pt"
OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43_audit_eval"
# ----------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


# ============================================================
# Repro
# ============================================================
def seed_all(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def valid_groups(channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# ============================================================
# Dataset
# ============================================================
class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(m.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(z_all) - 1, int(self.max_slices_per_subject)).astype(int)
                z_all = [z_all[i] for i in idx]

            for z in z_all:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        if self.cache_in_ram:
            self._ram_cache[sid] = data
            self._ram_cache.move_to_end(sid)
            if len(self._ram_cache) > self.cache_max_subjects:
                self._ram_cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(zs[1])


# ============================================================
# Model (must match Step 9)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


# ============================================================
# Load checkpoint
# ============================================================
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError(f"Checkpoint missing EMA dict: {ckpt_path}")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
        "seed": ckpt.get("seed", None),
    }


# ============================================================
# Metrics
# ============================================================
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats: Dict) -> Dict:
    out = {}
    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        out[k] = {
            "dice_all": float(dice_all),
            "dice_pos_only": float(dice_pos),
            "n_pos_slices": int(stats[k]["n_pos"]),
        }
    return out


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(
        f"[{split_name}] loaded seed={meta['seed']} epoch={meta['epoch']} best={meta['best']:.6f}",
        flush=True
    )

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i].detach().cpu()
            yi = y[i].detach().cpu()

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                fg.append(dc)

            update_stat(stats, "mean_fg", float(np.mean(fg)), gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    return {
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": finalize_stats(stats),
    }


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    m = report["metrics"]
    s = report["sanity"]

    print(f"  slices_total          : {s['slices_total']}")
    print(f"  slices_any_tumor_gt   : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt      : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred    : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")


def main():
    print("=== STEP 9 SINGLE-SEED AUDIT START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT_BEST:", CKPT_BEST, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n[VAL]")
    print_summary(val_report)

    print("\n[TEST]")
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9 SINGLE-SEED AUDIT DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 9 SINGLE-SEED AUDIT START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT_BEST: C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43/best.pt
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43_audit_eval
[VAL] loaded seed=43 epoch=11 best=0.914629


C:\Users\admin\AppData\Local\Temp\ipykernel_21072\2117505345.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL:

[TEST] loaded seed=43 epoch=11 best=0.914629


EVAL TEST: 100%|█████████████████████████████████████████████████| 20930/20930 [16:41<00:00, 20.90it/s, mean_fg=0.1095]


[VAL]
  slices_total          : 45136
  slices_any_tumor_gt   : 16476
  slices_any_tumor_pred : 45136
  unique_labels_gt      : [0, 1, 2, 3, 4]
  unique_labels_pred    : [0, 1, 2, 3, 4]
  mean_fg : 0.285317 / 0.155716 (npos=16476)
  WT      : 0.001287 / 0.003526 (npos=16476)
  TC      : 0.000966 / 0.003211 (npos=9839)
  ET      : 0.332394 / 0.000133 (npos=6847)

[TEST]
  slices_total          : 41860
  slices_any_tumor_gt   : 14364
  slices_any_tumor_pred : 41860
  unique_labels_gt      : [0, 1, 2, 3, 4]
  unique_labels_pred    : [0, 1, 2, 3, 4]
  mean_fg : 0.109519 / 0.071739 (npos=14364)
  WT      : 0.008008 / 0.023339 (npos=14364)
  TC      : 0.002654 / 0.011705 (npos=9490)
  ET      : 0.008318 / 0.003181 (npos=6661)

Saved: C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_43_audit_eval


=== STEP 9 SINGLE-SEED AUDIT DONE ===


In [4]:
import os
import json
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 SINGLE-SEED AUDIT EVAL
# Use this same script twice:
#   1) for seed_43/best.pt
#   2) for seed_44/best.pt
#
# Only change:
#   CKPT_BEST
#   OUT_DIR
# ============================================================

# -------------------------
# Paths
# -------------------------
INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

# ---------- CHANGE THESE ----------
CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44/best.pt"
OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44_audit_eval"
# ----------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
NUM_CLASSES = 5
IN_CH = 12

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


# ============================================================
# Repro
# ============================================================
def seed_all(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def valid_groups(channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


# ============================================================
# Dataset
# ============================================================
class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index: dict,
        subject_ids: list,
        image_size: int = 256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram: bool = True,
        lru_cache_max_subjects: int = 3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = int(image_size)
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = bool(cache_in_ram)

        self._ram_cache = OrderedDict()
        self.cache_max_subjects = int(lru_cache_max_subjects)

        self.samples: List[Tuple[str, int]] = []
        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(m.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > int(self.max_slices_per_subject):
                idx = np.linspace(0, len(z_all) - 1, int(self.max_slices_per_subject)).astype(int)
                z_all = [z_all[i] for i in idx]

            for z in z_all:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self._ram_cache:
            self._ram_cache.move_to_end(sid)
            return self._ram_cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        if self.cache_in_ram:
            self._ram_cache[sid] = data
            self._ram_cache.move_to_end(sid)
            if len(self._ram_cache) > self.cache_max_subjects:
                self._ram_cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long(), sid, int(zs[1])


# ============================================================
# Model (must match Step 9)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop=p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop=p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop=p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop=p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop=p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop=p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop=p_drop)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


# ============================================================
# Load checkpoint
# ============================================================
def load_ema_weights_into_model(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "ema" not in ckpt or not isinstance(ckpt["ema"], dict):
        raise RuntimeError(f"Checkpoint missing EMA dict: {ckpt_path}")

    ema_sd = ckpt["ema"]
    sd = model.state_dict()
    for k in sd.keys():
        if k in ema_sd and torch.is_tensor(ema_sd[k]):
            sd[k] = ema_sd[k]
    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", ckpt.get("best_val_dice", ckpt.get("val_dice", -1.0)))),
        "seed": ckpt.get("seed", None),
    }


# ============================================================
# Metrics
# ============================================================
def dice_bool(p: torch.Tensor, t: torch.Tensor, eps: float = 1e-6) -> float:
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2.0 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl: torch.Tensor) -> Dict[str, torch.Tensor]:
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)
    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    stats = {}
    for k in keys:
        stats[k] = {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
    return stats


def update_stat(stats, key: str, dice: float, gt_pos: bool):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats: Dict) -> Dict:
    out = {}
    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        dice_all = stats[k]["sum_all"] / max(1, stats[k]["n_all"])
        dice_pos = stats[k]["sum_pos"] / max(1, stats[k]["n_pos"])
        out[k] = {
            "dice_all": float(dice_all),
            "dice_pos_only": float(dice_pos),
            "n_pos_slices": int(stats[k]["n_pos"]),
        }
    return out


@torch.no_grad()
def eval_split(subjects: dict, split_ids: list, split_name: str) -> Dict:
    ds = BraTS2024SliceDataset2p5DEval(
        subjects, split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32, p_drop=0.10).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights_into_model(model, CKPT_BEST)
    print(
        f"[{split_name}] loaded seed={meta['seed']} epoch={meta['epoch']} best={meta['best']:.6f}",
        flush=True
    )

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)
    for x, y, sid, z in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)
        pred = torch.argmax(logits, dim=1)

        b = pred.shape[0]
        for i in range(b):
            pi = pred[i].detach().cpu()
            yi = y[i].detach().cpu()

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()
            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            fg = []
            for c in [1, 2, 3, 4]:
                dc = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", dc, gt_pos=(yi == c).any().item())
                fg.append(dc)

            update_stat(stats, "mean_fg", float(np.mean(fg)), gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)
            for k in ["WT", "TC", "ET"]:
                dk = dice_bool(pm[k], tm[k])
                update_stat(stats, k, dk, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    return {
        "ckpt": CKPT_BEST,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": finalize_stats(stats),
    }


def save_json(path: str, obj: Dict):
    def default(o):
        if isinstance(o, set):
            return sorted(list(o))
        return str(o)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=default)


def print_summary(report: Dict):
    m = report["metrics"]
    s = report["sanity"]

    print(f"  slices_total          : {s['slices_total']}")
    print(f"  slices_any_tumor_gt   : {s['slices_any_tumor_gt']}")
    print(f"  slices_any_tumor_pred : {s['slices_any_tumor_pred']}")
    print(f"  unique_labels_gt      : {s['unique_labels_gt']}")
    print(f"  unique_labels_pred    : {s['unique_labels_pred']}")

    print(f"  mean_fg : {m['mean_fg']['dice_all']:.6f} / {m['mean_fg']['dice_pos_only']:.6f} (npos={m['mean_fg']['n_pos_slices']})")
    print(f"  WT      : {m['WT']['dice_all']:.6f} / {m['WT']['dice_pos_only']:.6f} (npos={m['WT']['n_pos_slices']})")
    print(f"  TC      : {m['TC']['dice_all']:.6f} / {m['TC']['dice_pos_only']:.6f} (npos={m['TC']['n_pos_slices']})")
    print(f"  ET      : {m['ET']['dice_all']:.6f} / {m['ET']['dice_pos_only']:.6f} (npos={m['ET']['n_pos_slices']})")


def main():
    print("=== STEP 9 SINGLE-SEED AUDIT START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("workers:", NUM_WORKERS, flush=True)
    print("CKPT_BEST:", CKPT_BEST, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    val_report = eval_split(subjects, val_ids, "VAL")
    test_report = eval_split(subjects, test_ids, "TEST")

    save_json(os.path.join(OUT_DIR, "report_val.json"), val_report)
    save_json(os.path.join(OUT_DIR, "report_test.json"), test_report)

    print("\n[VAL]")
    print_summary(val_report)

    print("\n[TEST]")
    print_summary(test_report)

    print("\nSaved:", OUT_DIR, flush=True)
    print("=== STEP 9 SINGLE-SEED AUDIT DONE ===", flush=True)


if __name__ == "__main__":
    main()

=== STEP 9 SINGLE-SEED AUDIT START ===
Device: cuda
Torch: 2.5.1+cu121
workers: 0
CKPT_BEST: C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44/best.pt
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44_audit_eval
[VAL] loaded seed=44 epoch=10 best=0.912736


C:\Users\admin\AppData\Local\Temp\ipykernel_21072\1398624285.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL:

[TEST] loaded seed=44 epoch=10 best=0.912736


EVAL TEST: 100%|█████████████████████████████████████████████████| 20930/20930 [16:34<00:00, 21.05it/s, mean_fg=0.1734]



[VAL]
  slices_total          : 45136
  slices_any_tumor_gt   : 16476
  slices_any_tumor_pred : 45136
  unique_labels_gt      : [0, 1, 2, 3, 4]
  unique_labels_pred    : [0, 1, 2, 3, 4]
  mean_fg : 0.570183 / 0.391206 (npos=16476)
  WT      : 0.019695 / 0.053954 (npos=16476)
  TC      : 0.508890 / 0.000026 (npos=9839)
  ET      : 0.846464 / 0.000003 (npos=6847)

[TEST]
  slices_total          : 41860
  slices_any_tumor_gt   : 14364
  slices_any_tumor_pred : 41860
  unique_labels_gt      : [0, 1, 2, 3, 4]
  unique_labels_pred    : [0, 1, 2, 3, 4]
  mean_fg : 0.173393 / 0.107351 (npos=14364)
  WT      : 0.011874 / 0.034604 (npos=14364)
  TC      : 0.003885 / 0.017135 (npos=9490)
  ET      : 0.001739 / 0.000865 (npos=6661)

Saved: C:/GDNET/logs/step9_2p5d_unet_multiseed/seed_44_audit_eval
=== STEP 9 SINGLE-SEED AUDIT DONE ===


In [ ]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 9 MULTI-SEED RUNNER (CLEAN)
# Fresh runs only. No resume.
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

# NEW clean output folder
BASE_OUT = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean"

SEEDS = [43, 44]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5

EPOCHS = 12
BATCH = 2
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2

NUM_WORKERS = 0
PIN_MEMORY = True

P_TUMOR = 0.50
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 32

CACHE_IN_RAM = True
LRU_CACHE_MAX_SUBJECTS = 6

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999

# IMPORTANT: force fresh runs
RESUME_IF_EXISTS = False

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def lr_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    return BASE_LR * 0.5 * (1 + math.cos(math.pi * t))


class EMA:
    def __init__(self, model, decay=0.999, device="cpu"):
        self.decay = decay
        self.device = device
        self.shadow = {}
        self.backup = None
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(
                    p.detach().to(self.device),
                    alpha=(1.0 - self.decay)
                )

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model):
        if self.backup is None:
            return
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None


def soft_dice_loss(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs = probs[:, 1:]
    target_1h = target_1h[:, 1:]

    dims = (0, 2, 3)
    inter = (probs * target_1h).sum(dims)
    denom = probs.sum(dims) + target_1h.sum(dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    vals = []
    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)
        vals.append(d)
    return torch.stack(vals).mean().item()


class BraTS2024SliceDataset2p5D(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        is_train=True,
        p_tumor=0.5,
        tumor_min_frac=0.001,
        max_slices_per_subject=32,
        cache_in_ram=True,
        lru_cache_max_subjects=6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.is_train = is_train
        self.p_tumor = p_tumor
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects

        self.samples = []
        self.tumor_samples = []

        for sid in self.sids:
            mask = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(mask.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > self.max_slices_per_subject:
                idx = np.linspace(0, len(z_all)-1, self.max_slices_per_subject).astype(int)
                z_all = [z_all[i] for i in idx]

            tumor_z = [
                z for z in z_all
                if (mask[:, :, z] > 0).mean() >= self.tumor_min_frac
            ]

            self.samples.extend([(sid, z) for z in z_all])
            self.tumor_samples.extend([(sid, z) for z in tumor_z])

    def __len__(self):
        return len(self.samples)

    def _zscore(self, x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        self.cache[sid] = data
        self.cache.move_to_end(sid)
        if len(self.cache) > self.cache_max:
            self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = self._zscore(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        if self.is_train and len(self.tumor_samples) > 0 and random.random() < self.p_tumor:
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.samples[idx]

        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_ch=12, num_classes=5, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def save_ckpt(path, model, ema, optimizer, scaler, epoch, best, seed):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().cpu() for k, v in ema.shadow.items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best": best,
        "seed": seed,
    }, path)


def run_one_seed(seed: int):
    print(f"\n========== SEED {seed} ==========", flush=True)
    seed_all(seed)

    out_dir = os.path.join(BASE_OUT, f"seed_{seed}")
    os.makedirs(out_dir, exist_ok=True)

    subjects = json.load(open(INDEX, "r"))
    train_ids = json.load(open(TRAIN_SPLIT, "r"))
    val_ids = json.load(open(VAL_SPLIT, "r"))

    train_ds = BraTS2024SliceDataset2p5D(
        subjects, train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )

    val_ds = BraTS2024SliceDataset2p5D(
        subjects, val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(f"Seed {seed} | Train={len(train_ds)} Val={len(val_ds)} TumorTrain={len(train_ds.tumor_samples)}", flush=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    ce_loss = nn.CrossEntropyLoss()
    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    ema = EMA(model, decay=EMA_DECAY, device=("cuda" if DEVICE == "cuda" else "cpu"))

    best = -1.0

    best_path = os.path.join(out_dir, "best.pt")
    last_path = os.path.join(out_dir, "last.pt")

    history = []

    for epoch in range(EPOCHS):
        lr = lr_schedule(epoch)
        for g in optimizer.param_groups:
            g["lr"] = lr

        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Seed {seed} Ep {epoch+1}/{EPOCHS} TRAIN")
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True).float()
            y = y.to(DEVICE, non_blocking=True).long()

            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = (
                    W_CE * ce_loss(logits, y)
                    + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)

            train_loss += loss.item() * x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        train_loss /= len(train_loader.dataset)

        model.eval()
        ema.apply_to(model)

        val_loss = 0.0
        val_dice = 0.0
        n = 0

        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Seed {seed} Ep {epoch+1}/{EPOCHS} VAL(EMA)")
            for x, y in pbar:
                x = x.to(DEVICE, non_blocking=True).float()
                y = y.to(DEVICE, non_blocking=True).long()

                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = (
                        W_CE * ce_loss(logits, y)
                        + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                    )

                pred = torch.argmax(logits, dim=1)

                val_loss += loss.item() * x.size(0)
                val_dice += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbar.set_postfix(vloss=f"{loss.item():.4f}")

        ema.restore(model)

        val_loss /= len(val_loader.dataset)
        val_dice /= n

        row = {
            "seed": seed,
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_meanDiceFG_ema": float(val_dice),
            "lr": float(lr),
        }
        history.append(row)

        print(
            f"Seed {seed} | Epoch {epoch+1:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_meanDiceFG(EMA)={val_dice:.6f} | "
            f"lr={lr:.2e}",
            flush=True
        )

        save_ckpt(last_path, model, ema, optimizer, scaler, epoch + 1, best, seed)

        if val_dice > best:
            best = val_dice
            save_ckpt(best_path, model, ema, optimizer, scaler, epoch + 1, best, seed)
            print(f"Saved best: {best_path}", flush=True)

    with open(os.path.join(out_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    print(f"=== SEED {seed} DONE === best={best:.6f}", flush=True)

    return {
        "seed": seed,
        "best_val_meanDiceFG_ema": float(best),
        "best_path": best_path,
        "out_dir": out_dir,
    }


def main():
    print("=== STEP 9 MULTI-SEED CLEAN START ===")
    print("Device:", DEVICE)
    print("Torch:", torch.__version__)
    print("Seeds:", SEEDS)
    print("BASE_OUT:", BASE_OUT)

    os.makedirs(BASE_OUT, exist_ok=True)

    results = []
    for seed in SEEDS:
        results.append(run_one_seed(seed))

    with open(os.path.join(BASE_OUT, "multiseed_summary.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    vals = [r["best_val_meanDiceFG_ema"] for r in results]
    print("\n=== MULTI-SEED CLEAN SUMMARY ===")
    for r in results:
        print(f"Seed {r['seed']} | best_val_meanDiceFG(EMA)={r['best_val_meanDiceFG_ema']:.6f} | {r['best_path']}")
    print(f"\nMean ± std = {np.mean(vals):.6f} ± {np.std(vals):.6f}")
    print("Saved summary:", os.path.join(BASE_OUT, "multiseed_summary.json"))


if __name__ == "__main__":
    main()

=== STEP 9 MULTI-SEED CLEAN START ===
Device: cuda
Torch: 2.5.1+cu121
Seeds: [43, 44]
BASE_OUT: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean

========== SEED 43 ==========
Seed 43 | Train=36576 Val=7936 TumorTrain=12357


Seed 43 Ep 1/12 VAL(EMA): 100%|█████████████████████████████████████████████| 3968/3968 [04:00<00:00, 16.51it/s, vloss=0.5000]

Seed 43 | Epoch 01 | train_loss=0.3835 | val_loss=0.4424 | val_meanDiceFG(EMA)=0.844571 | lr=1.00e-04


Saved best: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean\seed_43\best.pt


Seed 43 Ep 2/12 VAL(EMA): 100%|█████████████████████████████████████████████| 3968/3968 [04:01<00:00, 16.46it/s, vloss=0.4999]

...

Seed 43 | Epoch 03 | train_loss=0.2863 | val_loss=0.1089 | val_meanDiceFG(EMA)=0.888787 | lr=2.00e-04


Saved best: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean\seed_43\best.pt


Seed 43 Ep 4/12 VAL(EMA): 100%|██████████████████████████████████████| 3968/3968 [04:00<00:00, 16.50it/s, vloss=0.0001]


Seed 43 | Epoch 04 | train_loss=0.1530 | val_loss=0.0755 | val_meanDiceFG(EMA)=0.894181 | lr=1.95e-04
Saved best: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean\seed_43\best.pt


Seed 43 Ep 5/12 TRAIN:   0%|                                                                 | 0/18288 [00:00<?, ?it/s]

In [1]:
import os, json, math, random
from collections import OrderedDict
from typing import Optional, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm


# ============================================================
# STEP 9 MULTI-SEED RUNNER WITH SAVE + RESUME
# Trains 3-slice 2.5D UNet for seeds 43 and 44
# Saves:
#   - last.pt every epoch
#   - best.pt when validation Dice improves
#   - history.json after every epoch
# Resumes:
#   - from last.pt if available
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TRAIN_SPLIT = r"C:/GDNET/splits/brats2024_train_labeled.json"
VAL_SPLIT   = r"C:/GDNET/splits/brats2024_val_labeled.json"

BASE_OUT = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume"

SEEDS = [43, 44]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5

EPOCHS = 12
BATCH = 2
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2

NUM_WORKERS = 0
PIN_MEMORY = True

P_TUMOR = 0.50
TUMOR_MIN_FRAC = 0.001
MAX_SLICES_PER_SUBJECT = 32

CACHE_IN_RAM = True
LRU_CACHE_MAX_SUBJECTS = 6

W_CE = 0.5
W_DICE = 0.5
MAX_GRAD_NORM = 1.0

EMA_DECAY = 0.999

# Resume from last.pt if it exists
RESUME_IF_EXISTS = True

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def lr_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS

    t = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    return BASE_LR * 0.5 * (1 + math.cos(math.pi * t))


class EMA:
    def __init__(self, model, decay=0.999, device="cpu"):
        self.decay = decay
        self.device = device
        self.shadow = {}
        self.backup = None

        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.detach().clone().to(device)

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(
                    p.detach().to(self.device),
                    alpha=(1.0 - self.decay)
                )

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n].to(p.device))

    @torch.no_grad()
    def restore(self, model):
        if self.backup is None:
            return

        for n, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])

        self.backup = None


def soft_dice_loss(logits, target, num_classes=5, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    target_1h = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    probs = probs[:, 1:]
    target_1h = target_1h[:, 1:]

    dims = (0, 2, 3)
    inter = (probs * target_1h).sum(dims)
    denom = probs.sum(dims) + target_1h.sum(dims)

    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


@torch.no_grad()
def mean_dice_fg(pred, target, num_classes=5, eps=1e-6):
    vals = []

    for c in range(1, num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        d = (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)
        vals.append(d)

    return torch.stack(vals).mean().item()


class BraTS2024SliceDataset2p5D(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        is_train=True,
        p_tumor=0.5,
        tumor_min_frac=0.001,
        max_slices_per_subject=32,
        cache_in_ram=True,
        lru_cache_max_subjects=6,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.is_train = is_train
        self.p_tumor = p_tumor
        self.tumor_min_frac = tumor_min_frac
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram

        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects

        self.samples = []
        self.tumor_samples = []

        for sid in self.sids:
            mask = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(mask.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > self.max_slices_per_subject:
                idx = np.linspace(0, len(z_all) - 1, self.max_slices_per_subject).astype(int)
                z_all = [z_all[i] for i in idx]

            tumor_z = [
                z for z in z_all
                if (mask[:, :, z] > 0).mean() >= self.tumor_min_frac
            ]

            self.samples.extend([(sid, z) for z in z_all])
            self.tumor_samples.extend([(sid, z) for z in tumor_z])

    def __len__(self):
        return len(self.samples)

    def _zscore(self, x):
        x = x.astype(np.float32)
        return (x - x.mean()) / (x.std() + 1e-8)

    def _load_subject(self, sid):
        if sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        self.cache[sid] = data
        self.cache.move_to_end(sid)

        if len(self.cache) > self.cache_max:
            self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = self._zscore(vol[:, :, z])
        return cv2.resize(
            s,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

    def __getitem__(self, idx):
        if self.is_train and len(self.tumor_samples) > 0 and random.random() < self.p_tumor:
            sid, z = random.choice(self.tumor_samples)
        else:
            sid, z = self.samples[idx]

        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []

        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)

        y = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

        return torch.from_numpy(x), torch.from_numpy(y)


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)

        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_ch=12, num_classes=5, base=32):
        super().__init__()

        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def save_ckpt(path, model, ema, optimizer, scaler, epoch, best, seed):
    torch.save({
        "model": model.state_dict(),
        "ema": {k: v.detach().cpu() for k, v in ema.shadow.items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "best": float(best),
        "seed": int(seed),
    }, path)


def load_ckpt(path, model, ema, optimizer=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu")

    model.load_state_dict(ckpt["model"], strict=True)

    if "ema" in ckpt and isinstance(ckpt["ema"], dict):
        for k, v in ckpt["ema"].items():
            if k in ema.shadow:
                ema.shadow[k] = v.detach().clone().to(ema.device)

    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])

    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0))
    best = float(ckpt.get("best", -1.0))
    seed = ckpt.get("seed", None)

    return start_epoch, best, seed


def save_history(history_path, history):
    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)


def load_history(history_path):
    if os.path.exists(history_path):
        with open(history_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def run_one_seed(seed: int):
    print(f"\n========== SEED {seed} ==========", flush=True)
    seed_all(seed)

    out_dir = os.path.join(BASE_OUT, f"seed_{seed}")
    os.makedirs(out_dir, exist_ok=True)

    best_path = os.path.join(out_dir, "best.pt")
    last_path = os.path.join(out_dir, "last.pt")
    history_path = os.path.join(out_dir, "history.json")

    subjects = json.load(open(INDEX, "r"))
    train_ids = json.load(open(TRAIN_SPLIT, "r"))
    val_ids = json.load(open(VAL_SPLIT, "r"))

    train_ds = BraTS2024SliceDataset2p5D(
        subjects,
        train_ids,
        image_size=IMG_SIZE,
        is_train=True,
        p_tumor=P_TUMOR,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=CACHE_IN_RAM,
        lru_cache_max_subjects=LRU_CACHE_MAX_SUBJECTS,
    )

    val_ds = BraTS2024SliceDataset2p5D(
        subjects,
        val_ids,
        image_size=IMG_SIZE,
        is_train=False,
        p_tumor=0.0,
        tumor_min_frac=TUMOR_MIN_FRAC,
        max_slices_per_subject=MAX_SLICES_PER_SUBJECT,
        cache_in_ram=True,
        lru_cache_max_subjects=2,
    )

    print(
        f"Seed {seed} | Train={len(train_ds)} "
        f"Val={len(val_ds)} TumorTrain={len(train_ds.tumor_samples)}",
        flush=True
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(
        in_ch=IN_CH,
        num_classes=NUM_CLASSES,
        base=32
    ).to(DEVICE)

    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,
        weight_decay=WEIGHT_DECAY
    )

    ce_loss = nn.CrossEntropyLoss()

    scaler = GradScaler(
        "cuda",
        enabled=(USE_AMP and DEVICE == "cuda")
    )

    ema = EMA(
        model,
        decay=EMA_DECAY,
        device=("cuda" if DEVICE == "cuda" else "cpu")
    )

    best = -1.0
    start_epoch = 0
    history = []

    if RESUME_IF_EXISTS and os.path.exists(last_path):
        start_epoch, best, ckpt_seed = load_ckpt(
            last_path,
            model,
            ema,
            optimizer=optimizer,
            scaler=scaler
        )

        history = load_history(history_path)

        print(
            f"[RESUME seed {seed}] Loaded last.pt | "
            f"ckpt_seed={ckpt_seed} | start_epoch={start_epoch} | best={best:.6f}",
            flush=True
        )

    elif RESUME_IF_EXISTS and os.path.exists(best_path):
        start_epoch, best, ckpt_seed = load_ckpt(
            best_path,
            model,
            ema,
            optimizer=None,
            scaler=None
        )

        history = load_history(history_path)

        print(
            f"[RESUME seed {seed}] Loaded best.pt only | "
            f"ckpt_seed={ckpt_seed} | start_epoch={start_epoch} | best={best:.6f}",
            flush=True
        )

    if start_epoch >= EPOCHS:
        print(
            f"[SKIP seed {seed}] Already completed: "
            f"start_epoch={start_epoch}, EPOCHS={EPOCHS}",
            flush=True
        )

        return {
            "seed": seed,
            "best_val_meanDiceFG_ema": float(best),
            "best_path": best_path,
            "last_path": last_path,
            "out_dir": out_dir,
        }

    for epoch in range(start_epoch, EPOCHS):
        lr = lr_schedule(epoch)

        for g in optimizer.param_groups:
            g["lr"] = lr

        # -------------------------
        # TRAIN
        # -------------------------
        model.train()
        train_loss = 0.0

        pbar = tqdm(
            train_loader,
            desc=f"Seed {seed} Ep {epoch + 1}/{EPOCHS} TRAIN"
        )

        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True).float()
            y = y.to(DEVICE, non_blocking=True).long()

            if DEVICE == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(x)
                loss = (
                    W_CE * ce_loss(logits, y)
                    + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()

            ema.update(model)

            train_loss += loss.item() * x.size(0)

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{lr:.2e}"
            )

        train_loss /= max(1, len(train_loader.dataset))

        # -------------------------
        # VALIDATION WITH EMA
        # -------------------------
        model.eval()
        ema.apply_to(model)

        val_loss = 0.0
        val_dice = 0.0
        n = 0

        with torch.no_grad():
            pbar = tqdm(
                val_loader,
                desc=f"Seed {seed} Ep {epoch + 1}/{EPOCHS} VAL(EMA)"
            )

            for x, y in pbar:
                x = x.to(DEVICE, non_blocking=True).float()
                y = y.to(DEVICE, non_blocking=True).long()

                if DEVICE == "cuda":
                    x = x.contiguous(memory_format=torch.channels_last)

                with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                    logits = model(x)
                    loss = (
                        W_CE * ce_loss(logits, y)
                        + W_DICE * soft_dice_loss(logits, y, NUM_CLASSES)
                    )

                pred = torch.argmax(logits, dim=1)

                val_loss += loss.item() * x.size(0)
                val_dice += mean_dice_fg(pred, y, NUM_CLASSES) * x.size(0)
                n += x.size(0)

                pbar.set_postfix(vloss=f"{loss.item():.4f}")

        ema.restore(model)

        val_loss /= max(1, len(val_loader.dataset))
        val_dice /= max(1, n)

        row = {
            "seed": seed,
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_meanDiceFG_ema": float(val_dice),
            "lr": float(lr),
        }

        history.append(row)
        save_history(history_path, history)

        print(
            f"Seed {seed} | Epoch {epoch + 1:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_meanDiceFG(EMA)={val_dice:.6f} | "
            f"best={best:.6f} | "
            f"lr={lr:.2e}",
            flush=True
        )

        # Save last checkpoint every epoch
        save_ckpt(
            last_path,
            model,
            ema,
            optimizer,
            scaler,
            epoch + 1,
            best,
            seed
        )

        # Save best checkpoint if improved
        if val_dice > best:
            best = val_dice

            save_ckpt(
                best_path,
                model,
                ema,
                optimizer,
                scaler,
                epoch + 1,
                best,
                seed
            )

            # update last.pt with new best value too
            save_ckpt(
                last_path,
                model,
                ema,
                optimizer,
                scaler,
                epoch + 1,
                best,
                seed
            )

            print(f"Saved best: {best_path}", flush=True)

    print(f"=== SEED {seed} DONE === best={best:.6f}", flush=True)

    return {
        "seed": seed,
        "best_val_meanDiceFG_ema": float(best),
        "best_path": best_path,
        "last_path": last_path,
        "out_dir": out_dir,
    }


def main():
    print("=== STEP 9 MULTI-SEED RESUME START ===", flush=True)
    print("Device:", DEVICE, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("Seeds:", SEEDS, flush=True)
    print("BASE_OUT:", BASE_OUT, flush=True)
    print("RESUME_IF_EXISTS:", RESUME_IF_EXISTS, flush=True)

    os.makedirs(BASE_OUT, exist_ok=True)

    results = []

    for seed in SEEDS:
        results.append(run_one_seed(seed))

    summary_path = os.path.join(BASE_OUT, "multiseed_summary.json")

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    vals = [r["best_val_meanDiceFG_ema"] for r in results]

    print("\n=== MULTI-SEED RESUME SUMMARY ===", flush=True)

    for r in results:
        print(
            f"Seed {r['seed']} | "
            f"best_val_meanDiceFG(EMA)={r['best_val_meanDiceFG_ema']:.6f} | "
            f"{r['best_path']}",
            flush=True
        )

    print(
        f"\nMean ± std = {np.mean(vals):.6f} ± {np.std(vals):.6f}",
        flush=True
    )

    print("Saved summary:", summary_path, flush=True)


if __name__ == "__main__":
    main()

=== STEP 9 MULTI-SEED RESUME START ===
Device: cuda
Torch: 2.5.1+cu121
Seeds: [43, 44]
BASE_OUT: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume
RESUME_IF_EXISTS: True

========== SEED 43 ==========
Seed 43 | Train=36576 Val=7936 TumorTrain=12357
[RESUME seed 43] Loaded last.pt | ckpt_seed=43 | start_epoch=12 | best=0.915187
[SKIP seed 43] Already completed: start_epoch=12, EPOCHS=12

========== SEED 44 ==========


C:\Users\admin\AppData\Local\Temp\ipykernel_19264\1476238476.py:354: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location="cpu")


Seed 44 | Train=36576 Val=7936 TumorTrain=12357
[RESUME seed 44] Loaded last.pt | ckpt_seed=44 | start_epoch=11 | best=0.912257


Seed 44 Ep 12/12 VAL(EMA): 100%|█████████████████████████████████████| 3968/3968 [04:00<00:00, 16.52it/s, vloss=0.0000]

Seed 44 | Epoch 12 | train_loss=0.0901 | val_loss=0.0577 | val_meanDiceFG(EMA)=0.913223 | best=0.912257 | lr=4.89e-06


Saved best: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume\seed_44\best.pt
=== SEED 44 DONE === best=0.913223

=== MULTI-SEED RESUME SUMMARY ===
Seed 43 | best_val_meanDiceFG(EMA)=0.915187 | C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume\seed_43\best.pt
Seed 44 | best_val_meanDiceFG(EMA)=0.913223 | C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume\seed_44\best.pt

Mean ± std = 0.914205 ± 0.000982
Saved summary: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume\multiseed_summary.json


In [2]:
import os, json
from collections import OrderedDict
from typing import Optional, Dict, List, Tuple

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 CLEAN MULTI-SEED AUDIT
# Audits seed_43 and seed_44 checkpoints from clean-resume run
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPTS = {
    43: r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_43/best.pt",
    44: r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_44/best.pt",
}

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/audit_eval"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5
BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True
INCLUDE_LABEL3_IN_COMPOSITES = True


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(self, subjects_index, subject_ids, image_size=256,
                 max_slices_per_subject: Optional[int] = None,
                 cache_in_ram=True, lru_cache_max_subjects=3):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram
        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects
        self.samples = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(m.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > self.max_slices_per_subject:
                idx = np.linspace(0, len(z_all) - 1, self.max_slices_per_subject).astype(int)
                z_all = [z_all[i] for i in idx]

            for z in z_all:
                self.samples.append((sid, int(z)))

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        if self.cache_in_ram:
            self.cache[sid] = data
            self.cache.move_to_end(sid)
            if len(self.cache) > self.cache_max:
                self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [self._safe_z(z - 1, Z), self._safe_z(z, Z), self._safe_z(z + 1, Z)]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)
        y = cv2.resize(mask[:, :, zs[1]], (self.image_size, self.image_size),
                       interpolation=cv2.INTER_NEAREST).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_c=12, num_classes=5, base=32, p_drop=0.10):
        super().__init__()
        self.enc1 = ConvBlock(in_c, base, p_drop)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base, base * 2, p_drop)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base * 2, base * 4, p_drop)
        self.pool3 = nn.MaxPool2d(2)
        self.bot = ConvBlock(base * 4, base * 8, p_drop)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBlock(base * 8, base * 4, p_drop)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBlock(base * 4, base * 2, p_drop)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBlock(base * 2, base, p_drop)
        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bot(self.pool3(e3))

        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)


def load_ema_weights(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "ema" not in ckpt:
        raise RuntimeError(f"No EMA found in checkpoint: {ckpt_path}")

    sd = model.state_dict()
    for k in sd.keys():
        if k in ckpt["ema"]:
            sd[k] = ckpt["ema"][k]

    model.load_state_dict(sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", -1.0)),
        "seed": ckpt.get("seed", None),
    }


def dice_bool(p, t, eps=1e-6):
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl):
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)

    et = (lbl == 4)
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    return {k: {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0} for k in keys}


def update_stat(stats, key, dice, gt_pos):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1
    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats):
    out = {}
    for k, v in stats.items():
        out[k] = {
            "dice_all": float(v["sum_all"] / max(1, v["n_all"])),
            "dice_pos_only": float(v["sum_pos"] / max(1, v["n_pos"])),
            "n_pos_slices": int(v["n_pos"]),
        }
    return out


@torch.no_grad()
def eval_split(subjects, split_ids, split_name, ckpt_path):
    ds = BraTS2024SliceDataset2p5DEval(subjects, split_ids, image_size=IMG_SIZE)
    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(in_c=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)
    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.eval()

    meta = load_ema_weights(model, ckpt_path)
    print(f"\n[{split_name}] seed={meta['seed']} epoch={meta['epoch']} best={meta['best']:.6f}")

    stats = init_stats()
    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

        pred = torch.argmax(logits, dim=1)

        for i in range(pred.shape[0]):
            pi = pred[i].detach().cpu()
            yi = y[i].detach().cpu()

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()

            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            fg_scores = []
            for c in [1, 2, 3, 4]:
                d = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", d, gt_pos=(yi == c).any().item())
                fg_scores.append(d)

            update_stat(stats, "mean_fg", float(np.mean(fg_scores)), gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)

            for k in ["WT", "TC", "ET"]:
                d = dice_bool(pm[k], tm[k])
                update_stat(stats, k, d, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    return {
        "ckpt": ckpt_path,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": finalize_stats(stats),
    }


def print_report(seed, split, report):
    s = report["sanity"]
    m = report["metrics"]

    print(f"\n===== Seed {seed} | {split} =====")
    print(f"slices_total          : {s['slices_total']}")
    print(f"slices_any_tumor_gt   : {s['slices_any_tumor_gt']}")
    print(f"slices_any_tumor_pred : {s['slices_any_tumor_pred']}")
    print(f"unique_labels_gt      : {s['unique_labels_gt']}")
    print(f"unique_labels_pred    : {s['unique_labels_pred']}")

    for k in ["mean_fg", "WT", "TC", "ET"]:
        print(
            f"{k:7s}: {m[k]['dice_all']:.6f} / "
            f"{m[k]['dice_pos_only']:.6f} "
            f"(npos={m[k]['n_pos_slices']})"
        )


def aggregate(all_reports, split):
    print(f"\n\n=== {split} mean ± std across seeds ===")

    for metric in ["mean_fg", "WT", "TC", "ET"]:
        all_vals = []
        pos_vals = []

        for seed in CKPTS.keys():
            r = all_reports[seed][split]["metrics"][metric]
            all_vals.append(r["dice_all"])
            pos_vals.append(r["dice_pos_only"])

        all_vals = np.array(all_vals, dtype=float)
        pos_vals = np.array(pos_vals, dtype=float)

        print(
            f"{metric:7s}: "
            f"{all_vals.mean():.6f} ± {all_vals.std():.6f} / "
            f"{pos_vals.mean():.6f} ± {pos_vals.std():.6f}"
        )


def main():
    print("=== STEP 9 CLEAN MULTI-SEED AUDIT START ===")
    print("Device:", DEVICE)
    print("Torch:", torch.__version__)
    print("OUT_DIR:", OUT_DIR)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)
    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)
    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    all_reports = {}

    for seed, ckpt in CKPTS.items():
        print(f"\n\n############################")
        print(f"Auditing seed {seed}")
        print(f"Checkpoint: {ckpt}")
        print(f"############################")

        val_report = eval_split(subjects, val_ids, "VAL", ckpt)
        test_report = eval_split(subjects, test_ids, "TEST", ckpt)

        all_reports[seed] = {
            "VAL": val_report,
            "TEST": test_report,
        }

        print_report(seed, "VAL", val_report)
        print_report(seed, "TEST", test_report)

    json_path = os.path.join(OUT_DIR, "audit_all_seeds.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_reports, f, indent=2)

    aggregate(all_reports, "VAL")
    aggregate(all_reports, "TEST")

    print("\nSaved:", json_path)
    print("=== STEP 9 CLEAN MULTI-SEED AUDIT DONE ===")


if __name__ == "__main__":
    main()

=== STEP 9 CLEAN MULTI-SEED AUDIT START ===
Device: cuda
Torch: 2.5.1+cu121
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/audit_eval


############################
Auditing seed 43
Checkpoint: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_43/best.pt
############################


C:\Users\admin\AppData\Local\Temp\ipykernel_19264\1891849226.py:184: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")



[VAL] seed=43 epoch=12 best=0.915187


EVAL VAL: 100%|██████████████████████████████████████████████████| 22568/22568 [16:36<00:00, 22.65it/s, mean_fg=0.6636]



[TEST] seed=43 epoch=12 best=0.915187


...


===== Seed 43 | VAL =====
slices_total          : 45136
slices_any_tumor_gt   : 16476
slices_any_tumor_pred : 44526
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 2, 3, 4]
mean_fg: 0.663592 / 0.512101 (npos=16476)
WT     : 0.013499 / 0.000078 (npos=16476)
TC     : 0.762806 / 0.000000 (npos=9839)
ET     : 0.830712 / 0.000000 (npos=6847)

===== Seed 43 | TEST =====
slices_total          : 41860
slices_any_tumor_gt   : 14364
slices_any_tumor_pred : 41860
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.267685 / 0.189486 (npos=14364)
WT     : 0.004774 / 0.013914 (npos=14364)
TC     : 0.002048 / 0.009034 (npos=9490)
ET     : 0.217877 / 0.000199 (npos=6661)


############################
Auditing seed 44
Checkpoint: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_44/best.pt
############################

[VAL] seed=44 epoch=12 best=0.913223


EVAL VAL: 100%|██████████████████████████████████████████████████| 22568/22568 [15:04<00:00, 24.96it/s, mean_fg=0.0455]



[TEST] seed=44 epoch=12 best=0.913223


EVAL TEST: 100%|█████████████████████████████████████████████████| 20930/20930 [14:03<00:00, 24.81it/s, mean_fg=0.0654]



===== Seed 44 | VAL =====
slices_total          : 45136
slices_any_tumor_gt   : 16476
slices_any_tumor_pred : 45136
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.045487 / 0.047119 (npos=16476)
WT     : 0.035069 / 0.096073 (npos=16476)
TC     : 0.008575 / 0.039336 (npos=9839)
ET     : 0.000519 / 0.003274 (npos=6847)

===== Seed 44 | TEST =====
slices_total          : 41860
slices_any_tumor_gt   : 14364
slices_any_tumor_pred : 41860
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.065405 / 0.019900 (npos=14364)
WT     : 0.000930 / 0.002711 (npos=14364)
TC     : 0.000377 / 0.001664 (npos=9490)
ET     : 0.044001 / 0.000280 (npos=6661)


=== VAL mean ± std across seeds ===
mean_fg: 0.354539 ± 0.309053 / 0.279610 ± 0.232491
WT     : 0.024284 ± 0.010785 / 0.048075 ± 0.047997
TC     : 0.385690 ± 0.377116 / 0.019668 ± 0.019668
ET     : 0.415615 ± 0.415096 / 0.001637 ± 0.001637


=== TEST mean ± std across se

In [3]:
import os
import json
from collections import OrderedDict
from typing import Optional, Dict

import numpy as np
import nibabel as nib
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# STEP 9 CLEAN MULTI-SEED AUDIT — FIXED MODEL NAMES
# Important:
#   This model uses the SAME names as training:
#   e1, p1, e2, p2, e3, p3, b, u3, d3, u2, d2, u1, d1, out
#
# It also checks that EMA tensors are actually loaded.
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
VAL_SPLIT = r"C:/GDNET/splits/brats2024_val_labeled.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPTS = {
    43: r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_43/best.pt",
    44: r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_44/best.pt",
}

OUT_DIR = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/audit_eval_fixed"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5

BATCH = 2
NUM_WORKERS = 0
PIN_MEMORY = True

INCLUDE_LABEL3_IN_COMPOSITES = True


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


class BraTS2024SliceDataset2p5DEval(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        max_slices_per_subject: Optional[int] = None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.max_slices_per_subject = max_slices_per_subject
        self.cache_in_ram = cache_in_ram
        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects
        self.samples = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(m.shape[2]))

            if self.max_slices_per_subject is not None and len(z_all) > self.max_slices_per_subject:
                idx = np.linspace(0, len(z_all) - 1, self.max_slices_per_subject).astype(int)
                z_all = [z_all[i] for i in idx]

            for z in z_all:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No evaluation samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        if self.cache_in_ram:
            self.cache[sid] = data
            self.cache.move_to_end(sid)
            if len(self.cache) > self.cache_max:
                self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(
            s,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA,
        )

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)

        y = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST,
        ).astype(np.int64)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


# ============================================================
# Model — EXACT TRAINING NAMES
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)

        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_ch=12, num_classes=5, base=32):
        super().__init__()

        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def load_ema_weights(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "ema" not in ckpt:
        raise RuntimeError(f"No EMA found in checkpoint: {ckpt_path}")

    ema_sd = ckpt["ema"]
    model_sd = model.state_dict()

    copied = 0
    missing = []
    shape_mismatch = []

    for k in model_sd.keys():
        if k in ema_sd:
            if tuple(ema_sd[k].shape) == tuple(model_sd[k].shape):
                model_sd[k] = ema_sd[k]
                copied += 1
            else:
                shape_mismatch.append((k, tuple(ema_sd[k].shape), tuple(model_sd[k].shape)))
        else:
            missing.append(k)

    print(f"Loaded EMA tensors: {copied}/{len(model_sd)}")

    if missing:
        print("Missing EMA keys, first 10:", missing[:10])

    if shape_mismatch:
        print("Shape mismatches, first 10:", shape_mismatch[:10])

    if copied < len(model_sd) * 0.90:
        raise RuntimeError(
            f"Too few EMA tensors loaded: {copied}/{len(model_sd)}. "
            f"Model architecture/name mismatch."
        )

    model.load_state_dict(model_sd, strict=True)

    return {
        "epoch": int(ckpt.get("epoch", -1)),
        "best": float(ckpt.get("best", -1.0)),
        "seed": ckpt.get("seed", None),
    }


def dice_bool(p, t, eps=1e-6):
    inter = (p & t).sum().item()
    ps = p.sum().item()
    ts = t.sum().item()
    return float((2 * inter + eps) / (ps + ts + eps))


def masks_composite(lbl):
    if INCLUDE_LABEL3_IN_COMPOSITES:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 3) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 3) | (lbl == 4)
    else:
        wt = (lbl == 1) | (lbl == 2) | (lbl == 4)
        tc = (lbl == 1) | (lbl == 4)

    et = lbl == 4
    return {"WT": wt, "TC": tc, "ET": et}


def init_stats():
    keys = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]
    return {
        k: {"sum_all": 0.0, "n_all": 0, "sum_pos": 0.0, "n_pos": 0}
        for k in keys
    }


def update_stat(stats, key, dice, gt_pos):
    stats[key]["sum_all"] += float(dice)
    stats[key]["n_all"] += 1

    if gt_pos:
        stats[key]["sum_pos"] += float(dice)
        stats[key]["n_pos"] += 1


def finalize_stats(stats):
    out = {}

    for k, v in stats.items():
        out[k] = {
            "dice_all": float(v["sum_all"] / max(1, v["n_all"])),
            "dice_pos_only": float(v["sum_pos"] / max(1, v["n_pos"])),
            "n_pos_slices": int(v["n_pos"]),
        }

    return out


@torch.no_grad()
def eval_split(subjects, split_ids, split_name, ckpt_path):
    ds = BraTS2024SliceDataset2p5DEval(
        subjects,
        split_ids,
        image_size=IMG_SIZE,
        max_slices_per_subject=None,
        cache_in_ram=True,
        lru_cache_max_subjects=3,
    )

    loader = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(PIN_MEMORY and DEVICE == "cuda"),
    )

    model = UNet2p5D(
        in_ch=IN_CH,
        num_classes=NUM_CLASSES,
        base=32,
    ).to(DEVICE)

    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    model.eval()

    meta = load_ema_weights(model, ckpt_path)

    print(
        f"\n[{split_name}] seed={meta['seed']} "
        f"epoch={meta['epoch']} best={meta['best']:.6f}",
        flush=True,
    )

    stats = init_stats()

    sanity = {
        "slices_total": 0,
        "slices_any_tumor_gt": 0,
        "slices_any_tumor_pred": 0,
        "unique_labels_gt": set(),
        "unique_labels_pred": set(),
    }

    pbar = tqdm(loader, desc=f"EVAL {split_name}", leave=True)

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

        pred = torch.argmax(logits, dim=1)

        for i in range(pred.shape[0]):
            pi = pred[i].detach().cpu()
            yi = y[i].detach().cpu()

            sanity["slices_total"] += 1
            sanity["unique_labels_gt"].update(yi.unique().tolist())
            sanity["unique_labels_pred"].update(pi.unique().tolist())

            gt_any = (yi > 0).any().item()
            pr_any = (pi > 0).any().item()

            sanity["slices_any_tumor_gt"] += int(gt_any)
            sanity["slices_any_tumor_pred"] += int(pr_any)

            fg_scores = []

            for c in [1, 2, 3, 4]:
                d = dice_bool(pi == c, yi == c)
                update_stat(stats, f"c{c}", d, gt_pos=(yi == c).any().item())
                fg_scores.append(d)

            update_stat(stats, "mean_fg", float(np.mean(fg_scores)), gt_pos=gt_any)

            pm = masks_composite(pi)
            tm = masks_composite(yi)

            for k in ["WT", "TC", "ET"]:
                d = dice_bool(pm[k], tm[k])
                update_stat(stats, k, d, gt_pos=tm[k].any().item())

        running = stats["mean_fg"]["sum_all"] / max(1, stats["mean_fg"]["n_all"])
        pbar.set_postfix(mean_fg=f"{running:.4f}")

    return {
        "ckpt": ckpt_path,
        "meta": meta,
        "sanity": {
            "slices_total": sanity["slices_total"],
            "slices_any_tumor_gt": sanity["slices_any_tumor_gt"],
            "slices_any_tumor_pred": sanity["slices_any_tumor_pred"],
            "unique_labels_gt": sorted(list(sanity["unique_labels_gt"])),
            "unique_labels_pred": sorted(list(sanity["unique_labels_pred"])),
        },
        "metrics": finalize_stats(stats),
    }


def print_report(seed, split, report):
    s = report["sanity"]
    m = report["metrics"]

    print(f"\n===== Seed {seed} | {split} =====")
    print(f"slices_total          : {s['slices_total']}")
    print(f"slices_any_tumor_gt   : {s['slices_any_tumor_gt']}")
    print(f"slices_any_tumor_pred : {s['slices_any_tumor_pred']}")
    print(f"unique_labels_gt      : {s['unique_labels_gt']}")
    print(f"unique_labels_pred    : {s['unique_labels_pred']}")

    for k in ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]:
        print(
            f"{k:7s}: {m[k]['dice_all']:.6f} / "
            f"{m[k]['dice_pos_only']:.6f} "
            f"(npos={m[k]['n_pos_slices']})"
        )


def aggregate(all_reports, split):
    print(f"\n\n=== {split} mean ± std across seeds ===")

    for metric in ["mean_fg", "WT", "TC", "ET"]:
        all_vals = []
        pos_vals = []

        for seed in CKPTS.keys():
            r = all_reports[seed][split]["metrics"][metric]
            all_vals.append(r["dice_all"])
            pos_vals.append(r["dice_pos_only"])

        all_vals = np.array(all_vals, dtype=float)
        pos_vals = np.array(pos_vals, dtype=float)

        print(
            f"{metric:7s}: "
            f"{all_vals.mean():.6f} ± {all_vals.std():.6f} / "
            f"{pos_vals.mean():.6f} ± {pos_vals.std():.6f}"
        )


def main():
    print("=== STEP 9 CLEAN MULTI-SEED AUDIT FIXED START ===")
    print("Device:", DEVICE)
    print("Torch:", torch.__version__)
    print("OUT_DIR:", OUT_DIR)

    with open(INDEX, "r", encoding="utf-8") as f:
        subjects = json.load(f)

    with open(VAL_SPLIT, "r", encoding="utf-8") as f:
        val_ids = json.load(f)

    with open(TEST_SPLIT, "r", encoding="utf-8") as f:
        test_ids = json.load(f)

    all_reports = {}

    for seed, ckpt in CKPTS.items():
        print("\n\n############################")
        print(f"Auditing seed {seed}")
        print(f"Checkpoint: {ckpt}")
        print("############################")

        val_report = eval_split(subjects, val_ids, "VAL", ckpt)
        test_report = eval_split(subjects, test_ids, "TEST", ckpt)

        all_reports[seed] = {
            "VAL": val_report,
            "TEST": test_report,
        }

        print_report(seed, "VAL", val_report)
        print_report(seed, "TEST", test_report)

    json_path = os.path.join(OUT_DIR, "audit_all_seeds_fixed.json")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_reports, f, indent=2)

    aggregate(all_reports, "VAL")
    aggregate(all_reports, "TEST")

    print("\nSaved:", json_path)
    print("=== STEP 9 CLEAN MULTI-SEED AUDIT FIXED DONE ===")


if __name__ == "__main__":
    main()

=== STEP 9 CLEAN MULTI-SEED AUDIT FIXED START ===
Device: cuda
Torch: 2.5.1+cu121
OUT_DIR: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/audit_eval_fixed


############################
Auditing seed 43
Checkpoint: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_43/best.pt
############################
Loaded EMA tensors: 50/50

[VAL] seed=43 epoch=12 best=0.915187


C:\Users\admin\AppData\Local\Temp\ipykernel_19264\1777573980.py:227: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
EVAL VAL:

Loaded EMA tensors: 50/50

[TEST] seed=43 epoch=12 best=0.915187


EVAL TEST: 100%|█████████████████████████████████████████████████| 20930/20930 [14:06<00:00, 24.72it/s, mean_fg=0.9275]



===== Seed 43 | VAL =====
slices_total          : 45136
slices_any_tumor_gt   : 16476
slices_any_tumor_pred : 16996
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.930455 / 0.833062 (npos=16476)
WT     : 0.898163 / 0.803562 (npos=16476)
TC     : 0.922888 / 0.759882 (npos=9839)
ET     : 0.920327 / 0.689774 (npos=6847)
c1     : 0.966671 / 0.520251 (npos=2404)
c2     : 0.890790 / 0.774495 (npos=16134)
c3     : 0.944033 / 0.719371 (npos=5930)
c4     : 0.920327 / 0.689774 (npos=6847)

===== Seed 43 | TEST =====
slices_total          : 41860
slices_any_tumor_gt   : 14364
slices_any_tumor_pred : 15062
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.927470 / 0.819925 (npos=14364)
WT     : 0.891876 / 0.791001 (npos=14364)
TC     : 0.922613 / 0.738417 (npos=9490)
ET     : 0.918104 / 0.656183 (npos=6661)
c1     : 0.969142 / 0.505013 (npos=2054)
c2     : 0.880513 / 0.752434 (npos=14084)
c3     : 0.942121 / 0.679

EVAL VAL: 100%|██████████████████████████████████████████████████| 22568/22568 [14:54<00:00, 25.24it/s, mean_fg=0.9289]


Loaded EMA tensors: 50/50

[TEST] seed=44 epoch=12 best=0.913223


EVAL TEST: 100%|█████████████████████████████████████████████████| 20930/20930 [13:52<00:00, 25.13it/s, mean_fg=0.9269]


===== Seed 44 | VAL =====
slices_total          : 45136
slices_any_tumor_gt   : 16476
slices_any_tumor_pred : 17359
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.928924 / 0.832220 (npos=16476)
WT     : 0.894193 / 0.806645 (npos=16476)
TC     : 0.923668 / 0.754211 (npos=9839)
ET     : 0.921326 / 0.676496 (npos=6847)
c1     : 0.965113 / 0.509702 (npos=2404)
c2     : 0.887011 / 0.779790 (npos=16134)
c3     : 0.942245 / 0.721953 (npos=5930)
c4     : 0.921326 / 0.676496 (npos=6847)

===== Seed 44 | TEST =====
slices_total          : 41860
slices_any_tumor_gt   : 14364
slices_any_tumor_pred : 15343
unique_labels_gt      : [0, 1, 2, 3, 4]
unique_labels_pred    : [0, 1, 2, 3, 4]
mean_fg: 0.926925 / 0.819710 (npos=14364)
WT     : 0.888250 / 0.791085 (npos=14364)
TC     : 0.924119 / 0.733469 (npos=9490)
ET     : 0.919652 / 0.651048 (npos=6661)
c1     : 0.968862 / 0.498318 (npos=2054)
c2     : 0.877854 / 0.754541 (npos=14084)
c3     : 0.941331 / 0.681

In [4]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PAPER-READY ANALYSIS FOR STEP 9 FINAL MODEL
# Creates:
#   1. Final metric tables
#   2. Mean ± std tables
#   3. CSV + Excel outputs
#   4. Bar plots for VAL/TEST pos-only Dice
# ============================================================

AUDIT_JSON = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/audit_eval_fixed/audit_all_seeds_fixed.json"

OUT_DIR = r"C:/GDNET/paper_outputs/step9_final_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_METRICS = ["mean_fg", "WT", "TC", "ET", "c1", "c2", "c3", "c4"]


def load_reports(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def flatten_reports(data):
    rows = []

    for seed_str, seed_data in data.items():
        seed = int(seed_str)

        for split in ["VAL", "TEST"]:
            report = seed_data[split]
            metrics = report["metrics"]

            for metric in TARGET_METRICS:
                rows.append({
                    "seed": seed,
                    "split": split,
                    "metric": metric,
                    "dice_all": metrics[metric]["dice_all"],
                    "dice_pos_only": metrics[metric]["dice_pos_only"],
                    "n_pos_slices": metrics[metric]["n_pos_slices"],
                    "slices_total": report["sanity"]["slices_total"],
                    "slices_any_tumor_gt": report["sanity"]["slices_any_tumor_gt"],
                    "slices_any_tumor_pred": report["sanity"]["slices_any_tumor_pred"],
                    "checkpoint": report["ckpt"],
                    "best_val_meanDiceFG": report["meta"]["best"],
                    "epoch": report["meta"]["epoch"],
                })

    return pd.DataFrame(rows)


def make_summary(df):
    summary = (
        df
        .groupby(["split", "metric"], as_index=False)
        .agg(
            dice_all_mean=("dice_all", "mean"),
            dice_all_std=("dice_all", "std"),
            dice_pos_mean=("dice_pos_only", "mean"),
            dice_pos_std=("dice_pos_only", "std"),
            n_pos_slices=("n_pos_slices", "first"),
        )
    )

    summary["dice_all_mean_std"] = summary.apply(
        lambda r: f"{r['dice_all_mean']:.6f} ± {r['dice_all_std']:.6f}", axis=1
    )
    summary["dice_pos_mean_std"] = summary.apply(
        lambda r: f"{r['dice_pos_mean']:.6f} ± {r['dice_pos_std']:.6f}", axis=1
    )

    return summary


def save_tables(df, summary):
    raw_csv = os.path.join(OUT_DIR, "all_seed_metrics_raw.csv")
    summary_csv = os.path.join(OUT_DIR, "final_summary_mean_std.csv")
    xlsx_path = os.path.join(OUT_DIR, "final_step9_results_tables.xlsx")

    df.to_csv(raw_csv, index=False)
    summary.to_csv(summary_csv, index=False)

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Raw per seed")
        summary.to_excel(writer, index=False, sheet_name="Mean std summary")

        for split in ["VAL", "TEST"]:
            s = summary[summary["split"] == split].copy()
            s.to_excel(writer, index=False, sheet_name=f"{split} summary")

    print("Saved:")
    print(raw_csv)
    print(summary_csv)
    print(xlsx_path)


def print_paper_tables(summary):
    for split in ["VAL", "TEST"]:
        print(f"\n===== {split} FINAL PAPER TABLE =====")
        s = summary[summary["split"] == split].copy()

        for metric in ["mean_fg", "WT", "TC", "ET"]:
            row = s[s["metric"] == metric].iloc[0]
            print(
                f"{metric:7s} | "
                f"dice_all: {row['dice_all_mean_std']} | "
                f"pos-only: {row['dice_pos_mean_std']}"
            )


def plot_pos_only(summary):
    for split in ["VAL", "TEST"]:
        s = summary[
            (summary["split"] == split) &
            (summary["metric"].isin(["WT", "TC", "ET"]))
        ].copy()

        labels = s["metric"].tolist()
        means = s["dice_pos_mean"].values
        stds = s["dice_pos_std"].values

        plt.figure(figsize=(7, 5))
        plt.bar(labels, means, yerr=stds, capsize=5)
        plt.ylim(0, 1)
        plt.ylabel("Positive-only Dice")
        plt.title(f"{split} WT / TC / ET Positive-only Dice")
        plt.grid(axis="y", alpha=0.3)

        out_png = os.path.join(OUT_DIR, f"{split.lower()}_positive_only_dice_barplot.png")
        out_pdf = os.path.join(OUT_DIR, f"{split.lower()}_positive_only_dice_barplot.pdf")

        plt.tight_layout()
        plt.savefig(out_png, dpi=300)
        plt.savefig(out_pdf)
        plt.close()

        print("Saved plot:", out_png)
        print("Saved plot:", out_pdf)


def main():
    data = load_reports(AUDIT_JSON)

    df = flatten_reports(data)
    summary = make_summary(df)

    save_tables(df, summary)
    print_paper_tables(summary)
    plot_pos_only(summary)

    print("\n=== DONE ===")
    print("Output folder:", OUT_DIR)


if __name__ == "__main__":
    main()

Saved:
C:/GDNET/paper_outputs/step9_final_analysis\all_seed_metrics_raw.csv
C:/GDNET/paper_outputs/step9_final_analysis\final_summary_mean_std.csv
C:/GDNET/paper_outputs/step9_final_analysis\final_step9_results_tables.xlsx

===== VAL FINAL PAPER TABLE =====
mean_fg | dice_all: 0.929690 ± 0.001083 | pos-only: 0.832641 ± 0.000596
WT      | dice_all: 0.896178 ± 0.002808 | pos-only: 0.805103 ± 0.002180
TC      | dice_all: 0.923278 ± 0.000552 | pos-only: 0.757046 ± 0.004010
ET      | dice_all: 0.920827 ± 0.000706 | pos-only: 0.683135 ± 0.009389

===== TEST FINAL PAPER TABLE =====
mean_fg | dice_all: 0.927197 ± 0.000386 | pos-only: 0.819817 ± 0.000152
WT      | dice_all: 0.890063 ± 0.002564 | pos-only: 0.791043 ± 0.000060
TC      | dice_all: 0.923366 ± 0.001065 | pos-only: 0.735943 ± 0.003498
ET      | dice_all: 0.918878 ± 0.001095 | pos-only: 0.653616 ± 0.003631
Saved plot: C:/GDNET/paper_outputs/step9_final_analysis\val_positive_only_dice_barplot.png
Saved plot: C:/GDNET/paper_outputs/step

In [1]:
import os, json, random, gc
from collections import OrderedDict
from typing import List, Tuple

import numpy as np
import nibabel as nib
import cv2
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm import tqdm


# ============================================================
# QUALITATIVE FIGURES FOR PAPER — MEMORY-SAFE VERSION
# Fixes CUDA OOM from DataLoader pin_memory.
# ============================================================

INDEX = r"C:/GDNET/experiments/brats2024_index.json"
TEST_SPLIT = r"C:/GDNET/splits/brats2024_test_labeled.json"

CKPT_BEST = r"C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_43/best.pt"

OUT_DIR = r"C:/GDNET/paper_outputs/step9_qualitative_figures_seed43"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True

IMG_SIZE = 256
IN_CH = 12
NUM_CLASSES = 5

BATCH = 1
NUM_WORKERS = 0
PIN_MEMORY = False

N_RANDOM = 12
N_BEST = 8
N_WORST = 8

RANDOM_SEED = 42


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_all(RANDOM_SEED)


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def valid_groups(channels, max_groups=8):
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


def zscore2d(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)


def normalise_img(x):
    x = x.astype(np.float32)
    p1, p99 = np.percentile(x, [1, 99])
    x = np.clip(x, p1, p99)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


def dice_binary(p, t, eps=1e-6):
    p = p.astype(bool)
    t = t.astype(bool)
    return float((2 * np.logical_and(p, t).sum() + eps) / (p.sum() + t.sum() + eps))


def mean_fg_dice_np(pred, gt):
    return float(np.mean([dice_binary(pred == c, gt == c) for c in [1, 2, 3, 4]]))


def wt_tc_et_dice(pred, gt):
    return {
        "WT": dice_binary(np.isin(pred, [1, 2, 3, 4]), np.isin(gt, [1, 2, 3, 4])),
        "TC": dice_binary(np.isin(pred, [1, 3, 4]), np.isin(gt, [1, 3, 4])),
        "ET": dice_binary(pred == 4, gt == 4),
    }


def mask_to_rgb(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.float32)
    rgb[mask == 1] = [1.0, 0.0, 0.0]
    rgb[mask == 2] = [0.0, 1.0, 0.0]
    rgb[mask == 3] = [0.0, 0.3, 1.0]
    rgb[mask == 4] = [1.0, 1.0, 0.0]
    return rgb


def overlay_mask(img, mask, alpha=0.45):
    img_rgb = np.stack([img, img, img], axis=-1)
    mask_rgb = mask_to_rgb(mask)
    mask_bin = mask > 0
    out = img_rgb.copy()
    out[mask_bin] = (1 - alpha) * img_rgb[mask_bin] + alpha * mask_rgb[mask_bin]
    return np.clip(out, 0, 1)


class BraTS2024QualDataset(Dataset):
    def __init__(
        self,
        subjects_index,
        subject_ids,
        image_size=256,
        cache_in_ram=False,
        lru_cache_max_subjects=1,
        tumor_only=True,
    ):
        self.subjects = subjects_index
        self.sids = list(subject_ids)
        self.image_size = image_size
        self.cache_in_ram = cache_in_ram
        self.cache = OrderedDict()
        self.cache_max = lru_cache_max_subjects
        self.samples: List[Tuple[str, int]] = []

        for sid in self.sids:
            m = nib.load(self.subjects[sid]["mask"]).get_fdata().astype(np.int16)
            z_all = list(range(m.shape[2]))

            if tumor_only:
                z_all = [z for z in z_all if (m[:, :, z] > 0).any()]

            for z in z_all:
                self.samples.append((sid, int(z)))

        if len(self.samples) == 0:
            raise RuntimeError("No samples found.")

    def __len__(self):
        return len(self.samples)

    def _load_subject(self, sid):
        if self.cache_in_ram and sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]

        p = self.subjects[sid]
        data = (
            nib.load(p["t1"]).get_fdata().astype(np.float32),
            nib.load(p["t1ce"]).get_fdata().astype(np.float32),
            nib.load(p["t2"]).get_fdata().astype(np.float32),
            nib.load(p["flair"]).get_fdata().astype(np.float32),
            nib.load(p["mask"]).get_fdata().astype(np.int16),
        )

        if self.cache_in_ram:
            self.cache[sid] = data
            self.cache.move_to_end(sid)
            if len(self.cache) > self.cache_max:
                self.cache.popitem(last=False)

        return data

    def _safe_z(self, z, Z):
        return max(0, min(Z - 1, z))

    def _prep(self, vol, z):
        s = zscore2d(vol[:, :, z])
        return cv2.resize(s, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)

    def _display_img(self, flair, z):
        img = flair[:, :, z]
        img = cv2.resize(img, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        return normalise_img(img)

    def __getitem__(self, idx):
        sid, z = self.samples[idx]
        t1, t1ce, t2, flair, mask = self._load_subject(sid)
        Z = mask.shape[2]

        zs = [
            self._safe_z(z - 1, Z),
            self._safe_z(z, Z),
            self._safe_z(z + 1, Z),
        ]

        chans = []
        for vol in [t1, t1ce, t2, flair]:
            for zz in zs:
                chans.append(self._prep(vol, zz))

        x = np.stack(chans, axis=0).astype(np.float32)

        gt = cv2.resize(
            mask[:, :, zs[1]],
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST,
        ).astype(np.int64)

        display = self._display_img(flair, zs[1])

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(gt).long(),
            torch.from_numpy(display).float(),
            sid,
            int(zs[1]),
        )


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, p_drop=0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(valid_groups(out_c), out_c)
        self.drop = nn.Dropout2d(p_drop)

    def forward(self, x):
        x = F.relu(self.gn1(self.conv1(x)), inplace=True)
        x = self.drop(x)
        x = F.relu(self.gn2(self.conv2(x)), inplace=True)
        return x


class UNet2p5D(nn.Module):
    def __init__(self, in_ch=12, num_classes=5, base=32):
        super().__init__()

        self.e1 = ConvBlock(in_ch, base)
        self.p1 = nn.MaxPool2d(2)

        self.e2 = ConvBlock(base, base * 2)
        self.p2 = nn.MaxPool2d(2)

        self.e3 = ConvBlock(base * 2, base * 4)
        self.p3 = nn.MaxPool2d(2)

        self.b = ConvBlock(base * 4, base * 8)

        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)

        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)

        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))

        d2 = self.u2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))

        d1 = self.u1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


def load_ema_weights(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "ema" not in ckpt:
        raise RuntimeError("Checkpoint does not contain EMA weights.")

    model_sd = model.state_dict()
    ema_sd = ckpt["ema"]

    copied = 0
    for k in model_sd.keys():
        if k in ema_sd and tuple(ema_sd[k].shape) == tuple(model_sd[k].shape):
            model_sd[k] = ema_sd[k]
            copied += 1

    print(f"Loaded EMA tensors: {copied}/{len(model_sd)}")

    if copied < len(model_sd) * 0.9:
        raise RuntimeError("EMA loading failed: model/checkpoint mismatch.")

    model.load_state_dict(model_sd, strict=True)
    return ckpt


def save_case_figure(case, out_path, title_prefix=""):
    sid = case["sid"]
    z = case["z"]
    img = case["img"]
    gt = case["gt"]
    pred = case["pred"]
    mean_dice = case["mean_fg"]
    reg = case["regions"]

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))

    axes[0].imshow(img, cmap="gray")
    axes[0].set_title("FLAIR")

    axes[1].imshow(gt, cmap="nipy_spectral", vmin=0, vmax=4)
    axes[1].set_title("Ground Truth")

    axes[2].imshow(pred, cmap="nipy_spectral", vmin=0, vmax=4)
    axes[2].set_title("Prediction")

    axes[3].imshow(overlay_mask(img, gt))
    axes[3].set_title("GT Overlay")

    axes[4].imshow(overlay_mask(img, pred))
    axes[4].set_title("Prediction Overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"{title_prefix} {sid} | slice {z} | "
        f"MeanFG={mean_dice:.3f} | WT={reg['WT']:.3f}, TC={reg['TC']:.3f}, ET={reg['ET']:.3f}",
        fontsize=12,
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_montage(cases, out_path, title="Qualitative montage"):
    n = len(cases)
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.2 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, case in enumerate(cases):
        img = case["img"]
        gt = case["gt"]
        pred = case["pred"]

        axes[i, 0].imshow(img, cmap="gray")
        axes[i, 0].set_title("FLAIR")

        axes[i, 1].imshow(gt, cmap="nipy_spectral", vmin=0, vmax=4)
        axes[i, 1].set_title("GT")

        axes[i, 2].imshow(pred, cmap="nipy_spectral", vmin=0, vmax=4)
        axes[i, 2].set_title("Prediction")

        axes[i, 3].imshow(overlay_mask(img, pred))
        axes[i, 3].set_title(f"{case['sid']} z={case['z']} Dice={case['mean_fg']:.3f}")

        for j in range(4):
            axes[i, j].axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


@torch.no_grad()
def main():
    print("=== QUALITATIVE FIGURE GENERATION START ===")
    print("Device:", DEVICE)
    print("Checkpoint:", CKPT_BEST)
    print("Output:", OUT_DIR)

    subjects = json.load(open(INDEX, "r", encoding="utf-8"))
    test_ids = json.load(open(TEST_SPLIT, "r", encoding="utf-8"))

    ds = BraTS2024QualDataset(
        subjects,
        test_ids,
        image_size=IMG_SIZE,
        cache_in_ram=False,
        lru_cache_max_subjects=1,
        tumor_only=True,
    )

    loader = DataLoader(
        ds,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )

    model = UNet2p5D(in_ch=IN_CH, num_classes=NUM_CLASSES, base=32).to(DEVICE)

    if DEVICE == "cuda":
        model = model.to(memory_format=torch.channels_last)

    model.eval()
    ckpt = load_ema_weights(model, CKPT_BEST)

    print(
        f"Using checkpoint seed={ckpt.get('seed')} epoch={ckpt.get('epoch')} best={ckpt.get('best')}",
        flush=True,
    )

    cases = []

    for x, gt, img, sid, z in tqdm(loader, desc="Inference"):
        x = x.to(DEVICE, non_blocking=False)

        if DEVICE == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)

        with autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(x)

        pred = torch.argmax(logits, dim=1).cpu().numpy()[0]
        gt_np = gt.numpy()[0]
        img_np = img.numpy()[0]

        mfg = mean_fg_dice_np(pred, gt_np)
        reg = wt_tc_et_dice(pred, gt_np)

        cases.append(
            {
                "sid": sid[0],
                "z": int(z.item()),
                "img": img_np,
                "gt": gt_np,
                "pred": pred,
                "mean_fg": mfg,
                "regions": reg,
            }
        )

        del x, gt, img, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    cases_sorted = sorted(cases, key=lambda c: c["mean_fg"])

    worst_cases = cases_sorted[:N_WORST]
    best_cases = cases_sorted[-N_BEST:][::-1]
    random_cases = random.sample(cases, min(N_RANDOM, len(cases)))

    folders = {
        "best": os.path.join(OUT_DIR, "best_cases"),
        "worst": os.path.join(OUT_DIR, "worst_cases"),
        "random": os.path.join(OUT_DIR, "random_cases"),
    }

    for f in folders.values():
        os.makedirs(f, exist_ok=True)

    for group_name, group_cases in [
        ("best", best_cases),
        ("worst", worst_cases),
        ("random", random_cases),
    ]:
        for i, case in enumerate(group_cases):
            out_png = os.path.join(
                folders[group_name],
                f"{group_name}_{i+1:02d}_{case['sid']}_z{case['z']}_dice{case['mean_fg']:.3f}.png",
            )
            save_case_figure(case, out_png, title_prefix=group_name.upper())

    save_montage(
        best_cases[:6],
        os.path.join(OUT_DIR, "montage_best_cases.png"),
        title="Best segmentation examples",
    )

    save_montage(
        worst_cases[:6],
        os.path.join(OUT_DIR, "montage_worst_cases.png"),
        title="Worst segmentation examples",
    )

    save_montage(
        random_cases[:6],
        os.path.join(OUT_DIR, "montage_random_cases.png"),
        title="Random segmentation examples",
    )

    rows = []
    for c in cases:
        rows.append(
            {
                "subject": c["sid"],
                "slice": c["z"],
                "mean_fg_dice": c["mean_fg"],
                "WT_dice": c["regions"]["WT"],
                "TC_dice": c["regions"]["TC"],
                "ET_dice": c["regions"]["ET"],
            }
        )

    df = pd.DataFrame(rows)
    csv_path = os.path.join(OUT_DIR, "qualitative_case_scores.csv")
    df.to_csv(csv_path, index=False)

    print("\nSaved qualitative figures to:")
    print(OUT_DIR)
    print("Best cases:", folders["best"])
    print("Worst cases:", folders["worst"])
    print("Random cases:", folders["random"])
    print("Saved CSV:", csv_path)
    print("=== QUALITATIVE FIGURE GENERATION DONE ===")


if __name__ == "__main__":
    main()

=== QUALITATIVE FIGURE GENERATION START ===
Device: cuda
Checkpoint: C:/GDNET/logs/step9_2p5d_unet_multiseed_clean_resume/seed_43/best.pt
Output: C:/GDNET/paper_outputs/step9_qualitative_figures_seed43
Loaded EMA tensors: 50/50
Using checkpoint seed=43 epoch=12 best=0.9151873347732509


C:\Users\admin\AppData\Local\Temp\ipykernel_17988\3015252081.py:283: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")
Inference


Saved qualitative figures to:
C:/GDNET/paper_outputs/step9_qualitative_figures_seed43
Best cases: C:/GDNET/paper_outputs/step9_qualitative_figures_seed43\best_cases
Worst cases: C:/GDNET/paper_outputs/step9_qualitative_figures_seed43\worst_cases
Random cases: C:/GDNET/paper_outputs/step9_qualitative_figures_seed43\random_cases
Saved CSV: C:/GDNET/paper_outputs/step9_qualitative_figures_seed43\qualitative_case_scores.csv
=== QUALITATIVE FIGURE GENERATION DONE ===


In [2]:
import os
import pandas as pd


# ============================================================
# FINAL ABLATION TABLE — GDNet / Step 9 Study
# Creates:
#   - ablation_results.csv
#   - ablation_results.xlsx
#   - paper_ready_ablation_table.csv
# ============================================================

OUT_DIR = r"C:/GDNET/paper_outputs/final_ablation_table"
os.makedirs(OUT_DIR, exist_ok=True)

rows = [
    {
        "Experiment": "Step 7.1d",
        "Model / Strategy": "2D UNet + EMA + mixed sampling",
        "Notes": "Best p_tumor=0.50 baseline before 2.5D context",
        "VAL mean_fg pos-only": 0.817695,
        "VAL WT pos-only": 0.805846,
        "VAL TC pos-only": 0.757526,
        "VAL ET pos-only": 0.698255,
        "TEST mean_fg pos-only": 0.805831,
        "TEST WT pos-only": 0.790243,
        "TEST TC pos-only": 0.739571,
        "TEST ET pos-only": 0.663476,
    },
    {
        "Experiment": "Step 7.3b",
        "Model / Strategy": "2D UNet + crop phase + LR reset",
        "Notes": "Crop fine-tuning phase",
        "VAL mean_fg pos-only": 0.815381,
        "VAL WT pos-only": 0.795425,
        "VAL TC pos-only": 0.753141,
        "VAL ET pos-only": 0.682125,
        "TEST mean_fg pos-only": 0.806602,
        "TEST WT pos-only": 0.783569,
        "TEST TC pos-only": 0.733833,
        "TEST ET pos-only": 0.653758,
    },
    {
        "Experiment": "Step 8",
        "Model / Strategy": "GDNet-style integration",
        "Notes": "GDNet architecture branch",
        "VAL mean_fg pos-only": 0.829642,
        "VAL WT pos-only": 0.794313,
        "VAL TC pos-only": 0.744131,
        "VAL ET pos-only": 0.665268,
        "TEST mean_fg pos-only": 0.816351,
        "TEST WT pos-only": 0.782649,
        "TEST TC pos-only": 0.723506,
        "TEST ET pos-only": 0.635527,
    },
    {
        "Experiment": "Step 8.1",
        "Model / Strategy": "GDNet + region-aware loss",
        "Notes": "Region-aware loss added to GDNet branch",
        "VAL mean_fg pos-only": 0.829046,
        "VAL WT pos-only": 0.786425,
        "VAL TC pos-only": 0.736487,
        "VAL ET pos-only": 0.655861,
        "TEST mean_fg pos-only": 0.815010,
        "TEST WT pos-only": 0.775203,
        "TEST TC pos-only": 0.713511,
        "TEST ET pos-only": 0.624326,
    },
    {
        "Experiment": "Step 9",
        "Model / Strategy": "3-slice 2.5D UNet + EMA + mixed sampling",
        "Notes": "Best single-seed model; 4 modalities × 3 slices = 12 channels",
        "VAL mean_fg pos-only": 0.823376,
        "VAL WT pos-only": 0.811000,
        "VAL TC pos-only": 0.768514,
        "VAL ET pos-only": 0.705560,
        "TEST mean_fg pos-only": 0.810913,
        "TEST WT pos-only": 0.797797,
        "TEST TC pos-only": 0.745872,
        "TEST ET pos-only": 0.669443,
    },
    {
        "Experiment": "Step 9.1",
        "Model / Strategy": "5-slice 2.5D UNet",
        "Notes": "Wider context; 4 modalities × 5 slices = 20 channels",
        "VAL mean_fg pos-only": 0.831897,
        "VAL WT pos-only": 0.809011,
        "VAL TC pos-only": 0.762675,
        "VAL ET pos-only": 0.681303,
        "TEST mean_fg pos-only": 0.822634,
        "TEST WT pos-only": 0.796063,
        "TEST TC pos-only": 0.743044,
        "TEST ET pos-only": 0.655886,
    },
    {
        "Experiment": "Step 9.2",
        "Model / Strategy": "3-slice 2.5D + light postprocessing",
        "Notes": "Connected-component cleanup; did not improve pos-only metrics",
        "VAL mean_fg pos-only": 0.826946,
        "VAL WT pos-only": 0.810557,
        "VAL TC pos-only": 0.768142,
        "VAL ET pos-only": 0.704664,
        "TEST mean_fg pos-only": 0.813972,
        "TEST WT pos-only": 0.797262,
        "TEST TC pos-only": 0.745345,
        "TEST ET pos-only": 0.668367,
    },
    {
        "Experiment": "Final multi-seed",
        "Model / Strategy": "3-slice 2.5D UNet + EMA + mixed sampling",
        "Notes": "Clean 2-seed robustness result, seed 43 and seed 44",
        "VAL mean_fg pos-only": "0.832641 ± 0.000596",
        "VAL WT pos-only": "0.805103 ± 0.002180",
        "VAL TC pos-only": "0.757046 ± 0.004010",
        "VAL ET pos-only": "0.683135 ± 0.009389",
        "TEST mean_fg pos-only": "0.819817 ± 0.000152",
        "TEST WT pos-only": "0.791043 ± 0.000060",
        "TEST TC pos-only": "0.735943 ± 0.003498",
        "TEST ET pos-only": "0.653616 ± 0.003631",
    },
]

df = pd.DataFrame(rows)

# Ranking based on single-run TEST mean_fg where numeric
rank_df = df.copy()
rank_df["TEST mean_fg numeric"] = pd.to_numeric(rank_df["TEST mean_fg pos-only"], errors="coerce")
rank_df["Rank by TEST mean_fg"] = rank_df["TEST mean_fg numeric"].rank(
    method="min",
    ascending=False
)

df["Rank by TEST mean_fg"] = rank_df["Rank by TEST mean_fg"]


# Paper-ready shorter version
paper_cols = [
    "Experiment",
    "Model / Strategy",
    "TEST mean_fg pos-only",
    "TEST WT pos-only",
    "TEST TC pos-only",
    "TEST ET pos-only",
    "VAL mean_fg pos-only",
    "Notes",
]

paper_df = df[paper_cols].copy()


# Save outputs
csv_path = os.path.join(OUT_DIR, "ablation_results_full.csv")
paper_csv_path = os.path.join(OUT_DIR, "paper_ready_ablation_table.csv")
xlsx_path = os.path.join(OUT_DIR, "ablation_results.xlsx")

df.to_csv(csv_path, index=False)
paper_df.to_csv(paper_csv_path, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Full ablation")
    paper_df.to_excel(writer, index=False, sheet_name="Paper table")


print("Saved:")
print(csv_path)
print(paper_csv_path)
print(xlsx_path)

print("\n===== PAPER-READY ABLATION TABLE =====")
print(paper_df.to_string(index=False))


Saved:
C:/GDNET/paper_outputs/final_ablation_table\ablation_results_full.csv
C:/GDNET/paper_outputs/final_ablation_table\paper_ready_ablation_table.csv
C:/GDNET/paper_outputs/final_ablation_table\ablation_results.xlsx

===== PAPER-READY ABLATION TABLE =====
      Experiment                         Model / Strategy TEST mean_fg pos-only    TEST WT pos-only    TEST TC pos-only    TEST ET pos-only VAL mean_fg pos-only                                                         Notes
       Step 7.1d           2D UNet + EMA + mixed sampling              0.805831            0.790243            0.739571            0.663476             0.817695                Best p_tumor=0.50 baseline before 2.5D context
       Step 7.3b          2D UNet + crop phase + LR reset              0.806602            0.783569            0.733833            0.653758             0.815381                                        Crop fine-tuning phase
          Step 8                  GDNet-style integration              0.